Install and test TransformerLens in Colab then run the official tutorial on GPT-2 to build intuition before touching your own pipeline

In [1]:
! pip install transformer_lens

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/977.7 kB ? eta -:--:--
   ---------------------------------------- 977.7/977.7 kB 9.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 12.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------- ----------------------------- 2.9/11.0 MB 13.6 MB/s eta 0:00:01
   -------------------- ------------------- 5.5/11.0 MB 12.7 MB/s eta 0:00:01
   ------------------------------- -------- 8.7/11.0 MB 12.5 MB/s eta 0:00:01
   ----------------------

  DEPRECATION: Building 'transformers-stream-generator' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'transformers-stream-generator'. Discussion can be found at https://github.com/pypa/pip/issues/6334

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Load Pythia-160M (tiny, fast) and run a few test sentences through run_with_cache - just to see what the cache object looks like.


In [2]:
from transformers import GPTNeoXForCausalLM, AutoTokenizer

model = GPTNeoXForCausalLM.from_pretrained(
  "EleutherAI/pythia-160m-deduped",
  revision="step3000",
  cache_dir="./pythia-160m-deduped/step3000",
)

tokenizer = AutoTokenizer.from_pretrained(
  "EleutherAI/pythia-160m-deduped",
  revision="step3000",
  cache_dir="./pythia-160m-deduped/step3000",
)

inputs = tokenizer("Hello, I am", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])


c:\Users\Bella\Documents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 174.40it/s]
[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
c:\Users\Bella\Documents\.venv\Lib\site-packages\transformers\generation\utils.py:1612: UserWarning: Using the model-agnostic default `max_length` (=24) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'Hello, I am trying to get my code to work. I am trying to get my code to work. I am'

## Stimulus Corpus: Sentence Templates

Each template uses `[ENTITY]` as the swap target. Four templates per framing category, plus a note on what each is designed to isolate.


### Pain/Suffering Framing
*Designed to elicit welfare-relevant moral processing directly*

1. "The [ENTITY] was in pain."
2. "Everyone could see that the [ENTITY] was suffering."
3. "The [ENTITY] experienced significant physical distress during the procedure."
4. "After the injury, the [ENTITY] showed clear signs of pain."


### Fear Framing
*Tests affective/emotional state attribution specifically — distinct from physical pain*

5. "The [ENTITY] felt fear as the predator approached."
6. "The [ENTITY] was visibly terrified by what was happening."
7. "A sense of dread overcame the [ENTITY] as the door closed."
8. "The [ENTITY] froze, overwhelmed by fear."


### Scientific Welfare Framing
*Tests whether the model encodes sentience differently when mediated through an epistemic/research context*

9. "Scientists studied whether the [ENTITY] could suffer."
10. "Researchers debated whether the [ENTITY] was capable of experiencing pain."
11. "The study aimed to determine whether the [ENTITY] had subjective experiences."
12. "Evidence suggested the [ENTITY] might be capable of feeling distress."

### Moral Consideration Framing
*Most directly tests moral patient encoding — does the model treat the entity as having interests?*

13. "The [ENTITY]'s interests were considered in the decision."
14. "Policymakers agreed that the [ENTITY]'s wellbeing deserved protection."
15. "The committee debated how much moral weight to give the [ENTITY]."
16. "It was decided that the [ENTITY] deserved to be treated with care."


### Neutral Control Framing
*Critical baseline — no welfare valence; isolates entity representation from welfare context*

17. "The [ENTITY] moved across the room."
18. "The [ENTITY] was observed near the window."
19. "Someone noticed the [ENTITY] on the other side of the field."
20. "The [ENTITY] remained still for several minutes."

### Cross-Framing Templates: Fear × Moral Consideration

*These templates are designed to test whether the sentience gradient is amplified when affective state and moral status co-occur in the same sentence — a more ecologically valid framing than either alone, since real welfare discourse routinely combines both.*

21. "The [ENTITY] was clearly frightened, and those responsible agreed its distress mattered."

*Fear is observed externally; moral weight is assigned by a third party. Tests whether the model encodes the entity as a legitimate object of moral concern when fear is the grounds for that concern.*

22. "Despite the [ENTITY]'s visible terror, no one considered whether it deserved protection."

*Negated moral consideration — the fear is acknowledged but the moral response is absent. Tests whether the model represents the entity's interests differently when moral consideration is explicitly withheld rather than granted.*


23. "The committee agreed that the [ENTITY]'s fear was a morally relevant factor in their decision."

*Institutional moral consideration framing. Closest to real policy language — tests whether the gradient holds in formal register, which matters for policy-facing AI deployment.*

24. "It was impossible to ignore that the [ENTITY] was suffering, and everyone felt they had a responsibility to help."

*Combines suffering with felt moral obligation — the strongest welfare-valenced template in the corpus. Useful as an upper-bound anchor: if no gradient appears here, it is unlikely to appear anywhere.*

25. "The [ENTITY] trembled with fear, yet the question of whether its experience counted remained unresolved."

*Moral uncertainty framing — fear is present and vivid, but moral status is explicitly held open. Tests whether the model encodes the entity's moral patient status differently when that status is narratively contested rather than assumed.*



## Entity Taxonomy: 60 Entities Across 6 Tiers

*Selection criteria: entities should be (1) unambiguous in identity, (2) likely well-represented in training data, (3) varied enough within tier to avoid idiosyncratic effects from any single entity, and (4) tokenisable as a single common word where possible.*

---

### T1: Humans
*Varied across age, role, and social context to test whether the gradient holds within the human tier itself: a useful internal validity check*

1. child
2. infant
3. elderly person
4. worker
5. patient
6. prisoner
7. refugee
8. soldier
9. teenager
10. woman

---

### T2: Mammals
*Mix of companion animals, farmed animals, and wild animals: these distinctions may themselves produce sub-gradients worth noting*

11. dog
12. chimpanzee
13. pig
14. cow
15. dolphin
16. elephant
17. rabbit
18. rat
19. horse
20. whale

---

### T3: Non-Mammal Vertebrates
*Includes farmed, wild, and laboratory-relevant species: fish and chickens are particularly important given their scale in industrial farming*

21. salmon
22. chicken
23. frog
24. zebrafish
25. crow
26. snake
27. tuna
28. gecko
29. trout
30. pigeon

---

### T4: Welfare-Evidenced Invertebrates
*Species with meaningful scientific evidence for sentience or nociception: octopus and crab are the strongest cases; bees have welfare-relevant cognitive evidence*

31. octopus
32. crab
33. bee
34. shrimp
35. lobster
36. squid
37. crayfish
38. bumblebee
39. mussel
40. mantis shrimp

---

### T5: Minimal-Evidence Invertebrates
*Species where sentience evidence is weak, contested, or absent: important for testing whether the model tracks scientific uncertainty or applies a cruder heuristic*

41. ant
42. fly
43. earthworm
44. moth
45. beetle
46. aphid
47. nematode
48. cockroach
49. mite
50. woodlouse

---

### T6: Non-Sentient Controls
*Critical baseline tier: mix of living non-animals and clearly inanimate objects. The living/non-living distinction within this tier is itself analytically useful*

51. oak tree
52. mushroom
53. bacterium
54. moss
55. seaweed
56. rock
57. table
58. cloud
59. river
60. statue



## Generate full sentence × entity matrix (~240–300 sentences)

In [3]:
import pandas as pd
import itertools

templates = [
    # Pain/Suffering
    ("T01", "pain_suffering", "The [ENTITY] was in pain."),
    ("T02", "pain_suffering", "Everyone could see that the [ENTITY] was suffering."),
    ("T03", "pain_suffering", "The [ENTITY] experienced significant physical distress during the procedure."),
    ("T04", "pain_suffering", "After the injury, the [ENTITY] showed clear signs of pain."),

    # Fear
    ("T05", "fear", "The [ENTITY] felt fear as the predator approached."),
    ("T06", "fear", "The [ENTITY] was visibly terrified by what was happening."),
    ("T07", "fear", "A sense of dread overcame the [ENTITY] as the door closed."),
    ("T08", "fear", "The [ENTITY] froze, overwhelmed by fear."),

    # Scientific welfare
    ("T09", "scientific_welfare", "Scientists studied whether the [ENTITY] could suffer."),
    ("T10", "scientific_welfare", "Researchers debated whether the [ENTITY] was capable of experiencing pain."),
    ("T11", "scientific_welfare", "The study aimed to determine whether the [ENTITY] had subjective experiences."),
    ("T12", "scientific_welfare", "Evidence suggested the [ENTITY] might be capable of feeling distress."),

    # Moral consideration
    ("T13", "moral_consideration", "The [ENTITY]'s interests were considered in the decision."),
    ("T14", "moral_consideration", "Policymakers agreed that the [ENTITY]'s wellbeing deserved protection."),
    ("T15", "moral_consideration", "The committee debated how much moral weight to give the [ENTITY]."),
    ("T16", "moral_consideration", "It was decided that the [ENTITY] deserved to be treated with care."),

    # Neutral control
    ("T17", "neutral_control", "The [ENTITY] moved across the room."),
    ("T18", "neutral_control", "The [ENTITY] was observed near the window."),
    ("T19", "neutral_control", "Someone noticed the [ENTITY] on the other side of the field."),
    ("T20", "neutral_control", "The [ENTITY] remained still for several minutes."),

    # Cross-framing: fear x moral consideration
    ("T21", "cross_framing", "The [ENTITY] was clearly frightened, and those responsible agreed its distress mattered."),
    ("T22", "cross_framing", "Despite the [ENTITY]'s visible terror, no one considered whether it deserved protection."),
    ("T23", "cross_framing", "The committee agreed that the [ENTITY]'s fear was a morally relevant factor in their decision."),
    ("T24", "cross_framing", "It was impossible to ignore that the [ENTITY] was suffering, and everyone felt they had a responsibility to help."),
    ("T25", "cross_framing", "The [ENTITY] trembled with fear, yet the question of whether its experience counted remained unresolved."),
]

entities = [
    # T1: Humans
    ("child", "T1"), ("infant", "T1"), ("elderly person", "T1"),
    ("worker", "T1"), ("patient", "T1"), ("prisoner", "T1"),
    ("refugee", "T1"), ("soldier", "T1"), ("teenager", "T1"),
    ("woman", "T1"),

    # T2: Mammals
    ("dog", "T2"), ("chimpanzee", "T2"), ("pig", "T2"),
    ("cow", "T2"), ("dolphin", "T2"), ("elephant", "T2"),
    ("rabbit", "T2"), ("rat", "T2"), ("horse", "T2"),
    ("whale", "T2"),

    # T3: Non-mammal vertebrates
    ("salmon", "T3"), ("chicken", "T3"), ("frog", "T3"),
    ("zebrafish", "T3"), ("crow", "T3"), ("snake", "T3"),
    ("tuna", "T3"), ("gecko", "T3"), ("trout", "T3"),
    ("pigeon", "T3"),

    # T4: Welfare-evidenced invertebrates
    ("octopus", "T4"), ("crab", "T4"), ("bee", "T4"),
    ("shrimp", "T4"), ("lobster", "T4"), ("squid", "T4"),
    ("crayfish", "T4"), ("bumblebee", "T4"), ("mussel", "T4"),
    ("mantis shrimp", "T4"),

    # T5: Minimal-evidence invertebrates
    ("ant", "T5"), ("fly", "T5"), ("earthworm", "T5"),
    ("moth", "T5"), ("beetle", "T5"), ("aphid", "T5"),
    ("nematode", "T5"), ("cockroach", "T5"), ("mite", "T5"),
    ("woodlouse", "T5"),

    # T6: Non-sentient controls
    ("oak tree", "T6"), ("mushroom", "T6"), ("bacterium", "T6"),
    ("moss", "T6"), ("seaweed", "T6"), ("rock", "T6"),
    ("table", "T6"), ("cloud", "T6"), ("river", "T6"),
    ("statue", "T6"),
]

rows = []
for (tid, ttype, template), (entity, tier) in itertools.product(templates, entities):
    sentence = template.replace("[ENTITY]", entity)
    # Handle possessive templates correctly
    sentence_id = f"{tid}_{entity.replace(' ', '_').upper()}"
    rows.append({
        "sentence_id": sentence_id,
        "template_id": tid,
        "template_type": ttype,
        "entity": entity,
        "tier": tier,
        "sentence": sentence,
        "multi_token_flag": len(entity.split()) > 1
    })

df = pd.DataFrame(rows)

# Sanity checks
assert len(df) == 1500, f"Expected 1500 rows, got {len(df)}"
assert df["tier"].nunique() == 6
assert df["template_type"].nunique() == 6 # Corrected from 5 to 6
assert df["multi_token_flag"].sum() == 25 * 3  # 3 multi-token entities x 25 templates

print(df.groupby(["tier", "template_type"]).size())
print(f"\nTotal sentences: {len(df)}")
print(f"Multi-token entity sentences flagged: {df['multi_token_flag'].sum()}")

df.to_csv("sentience_salience_corpus.csv", index=False)
print("\nSaved to sentience_salience_corpus.csv")

tier  template_type      
T1    cross_framing          50
      fear                   40
      moral_consideration    40
      neutral_control        40
      pain_suffering         40
      scientific_welfare     40
T2    cross_framing          50
      fear                   40
      moral_consideration    40
      neutral_control        40
      pain_suffering         40
      scientific_welfare     40
T3    cross_framing          50
      fear                   40
      moral_consideration    40
      neutral_control        40
      pain_suffering         40
      scientific_welfare     40
T4    cross_framing          50
      fear                   40
      moral_consideration    40
      neutral_control        40
      pain_suffering         40
      scientific_welfare     40
T5    cross_framing          50
      fear                   40
      moral_consideration    40
      neutral_control        40
      pain_suffering         40
      scientific_welfare     40
T6    cross_fr

## Run perplexity check on all sentences using Pythia - flag any sentence/entity combinations that score as implausibly unnatural (these introduce confounds)

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

#  1. Load model & tokenizer

MODEL_NAME = "EleutherAI/pythia-1.4b
"  # Start small for screening
# Upgrade to "EleutherAI/pythia-1.4b" once pipeline is validated

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model loaded: {MODEL_NAME}")
print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


# ── 2. Perplexity function ─────────────────────────────────────────────────

def compute_perplexity(sentence: str, model, tokenizer, device) -> float:
    """
    Compute per-token perplexity for a single sentence.
    Returns perplexity as a float; higher = more surprising to the model.
    """
    encodings = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )
    input_ids = encodings.input_ids.to(device)

    # Need at least 2 tokens to compute loss
    if input_ids.shape[1] < 2:
        return float('nan')

    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        # outputs.loss is mean negative log likelihood per token
        loss = outputs.loss

    perplexity = torch.exp(loss).item()
    return perplexity


def compute_perplexity_batch(sentences: list, model, tokenizer,
                              device, batch_size: int = 32) -> list:
    """
    Compute perplexity for a list of sentences in batches.
    Returns list of perplexity floats in same order as input.
    """
    perplexities = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        batch_perp = []

        for sentence in batch:
            perp = compute_perplexity(sentence, model, tokenizer, device)
            batch_perp.append(perp)

        perplexities.extend(batch_perp)

        if i % 200 == 0:
            print(f"  Processed {i}/{len(sentences)} sentences...")

    return perplexities


# ── 3. Load corpus & run screening ────────────────────────────────────────

df = pd.read_csv("sentience_salience_corpus.csv")
print(f"\nCorpus loaded: {len(df)} sentences")

print("\nComputing perplexity for all sentences...")
df["perplexity"] = compute_perplexity_batch(
    df["sentence"].tolist(),
    model, tokenizer, device
)

print(f"\nPerplexity computation complete.")
print(f"Mean perplexity:   {df['perplexity'].mean():.2f}")
print(f"Median perplexity: {df['perplexity'].median():.2f}")
print(f"Std perplexity:    {df['perplexity'].std():.2f}")
print(f"Min perplexity:    {df['perplexity'].min():.2f}")
print(f"Max perplexity:    {df['perplexity'].max():.2f}")


# ── 4. Flagging logic ──────────────────────────────────────────────────────

def flag_sentences(df: pd.DataFrame,
                   global_z_threshold: float = 2.0,
                   within_template_z_threshold: float = 2.0,
                   absolute_threshold: float = None) -> pd.DataFrame:
    """
    Flag sentences using three complementary criteria:

    1. Global z-score: sentence perplexity > N std above overall mean
    2. Within-template z-score: perplexity > N std above that template's mean
       (catches entity-specific anomalies that global threshold misses)
    3. Absolute threshold: optional hard ceiling (e.g. perplexity > 500)

    A sentence is flagged if ANY criterion is triggered.
    """
    df = df.copy()

    # ── Flag 1: Global z-score
    global_mean = df["perplexity"].mean()
    global_std  = df["perplexity"].std()
    df["global_z"] = (df["perplexity"] - global_mean) / global_std
    df["flag_global"] = df["global_z"] > global_z_threshold

    # ── Flag 2: Within-template z-score
    template_stats = (df.groupby("template_id")["perplexity"]
                        .agg(["mean", "std"])
                        .rename(columns={"mean": "tmpl_mean", "std": "tmpl_std"}))
    df = df.join(template_stats, on="template_id")
    df["template_z"] = (df["perplexity"] - df["tmpl_mean"]) / df["tmpl_std"]
    df["flag_template"] = df["template_z"] > within_template_z_threshold

    # ── Flag 3: Absolute threshold
    if absolute_threshold:
        df["flag_absolute"] = df["perplexity"] > absolute_threshold
    else:
        df["flag_absolute"] = False

    # ── Combined flag
    df["flagged"] = df["flag_global"] | df["flag_template"] | df["flag_absolute"]

    # ── Flag reason string for inspection
    def flag_reason(row):
        reasons = []
        if row["flag_global"]:
            reasons.append(f"global_z={row['global_z']:.2f}")
        if row["flag_template"]:
            reasons.append(f"template_z={row['template_z']:.2f}")
        if row["flag_absolute"]:
            reasons.append(f"absolute>{absolute_threshold}")
        return "; ".join(reasons) if reasons else ""

    df["flag_reason"] = df.apply(flag_reason, axis=1)

    return df


df = flag_sentences(
    df,
    global_z_threshold=2.0,
    within_template_z_threshold=2.0,
    absolute_threshold=500
)

n_flagged = df["flagged"].sum()
pct_flagged = 100 * n_flagged / len(df)
print(f"\nFlagged sentences: {n_flagged} ({pct_flagged:.1f}% of corpus)")


# ── 5. Diagnostic reports ──────────────────────────────────────────────────

print("\n" + "="*60)
print("FLAGGED SENTENCES — FULL LIST")
print("="*60)

flagged_df = df[df["flagged"]].sort_values("perplexity", ascending=False)

for _, row in flagged_df.iterrows():
    print(f"\n[{row['sentence_id']}]")
    print(f"  Sentence:    {row['sentence']}")
    print(f"  Entity:      {row['entity']} (Tier {row['tier']})")
    print(f"  Template:    {row['template_id']} ({row['template_type']})")
    print(f"  Perplexity:  {row['perplexity']:.2f}")
    print(f"  Flag reason: {row['flag_reason']}")


# ── 6. Per-entity summary ──────────────────────────────────────────────────

print("\n" + "="*60)
print("PER-ENTITY PERPLEXITY SUMMARY")
print("="*60)

entity_summary = (df.groupby(["entity", "tier"])
                    .agg(
                        mean_perplexity=("perplexity", "mean"),
                        max_perplexity=("perplexity", "max"),
                        n_flagged=("flagged", "sum"),
                        n_sentences=("flagged", "count")
                    )
                    .reset_index()
                    .sort_values("mean_perplexity", ascending=False))

entity_summary["pct_flagged"] = (
    100 * entity_summary["n_flagged"] / entity_summary["n_sentences"]
)

print(entity_summary.to_string(index=False))


# ── 7. Per-template summary ────────────────────────────────────────────────

print("\n" + "="*60)
print("PER-TEMPLATE PERPLEXITY SUMMARY")
print("="*60)

template_summary = (df.groupby(["template_id", "template_type"])
                      .agg(
                          mean_perplexity=("perplexity", "mean"),
                          std_perplexity=("perplexity", "std"),
                          max_perplexity=("perplexity", "max"),
                          n_flagged=("flagged", "sum")
                      )
                      .reset_index()
                      .sort_values("mean_perplexity", ascending=False))

print(template_summary.to_string(index=False))


# ── 8. Per-tier summary ────────────────────────────────────────────────────

print("\n" + "="*60)
print("PER-TIER PERPLEXITY SUMMARY")
print("="*60)

tier_summary = (df.groupby("tier")
                  .agg(
                      mean_perplexity=("perplexity", "mean"),
                      std_perplexity=("perplexity", "std"),
                      n_flagged=("flagged", "sum"),
                      n_sentences=("flagged", "count")
                  )
                  .reset_index())

tier_summary["pct_flagged"] = (
    100 * tier_summary["n_flagged"] / tier_summary["n_sentences"]
)

print(tier_summary.to_string(index=False))

# ── Critical check: does T6 have systematically higher perplexity?
# If yes, that is expected and acceptable.
# If T5 also shows high perplexity, those sentences need review.
t1_mean = tier_summary[tier_summary["tier"] == "T1"]["mean_perplexity"].values[0]
t6_mean = tier_summary[tier_summary["tier"] == "T6"]["mean_perplexity"].values[0]
print(f"\nT1 mean perplexity: {t1_mean:.2f}")
print(f"T6 mean perplexity: {t6_mean:.2f}")
print(f"T6/T1 ratio: {t6_mean/t1_mean:.2f}x")
print("NOTE: ratio > 2x expected and acceptable for non-sentient controls")
print("      ratio > 5x suggests template redesign needed for T6 entities")


# ── 9. Visualisations ─────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Perplexity Screening Results", fontsize=14, fontweight="bold")

# ── Plot 1: Distribution of perplexity scores
ax1 = axes[0, 0]
ax1.hist(df["perplexity"], bins=50, edgecolor="black", alpha=0.7, color="steelblue")
threshold_val = df["perplexity"].mean() + 2 * df["perplexity"].std()
ax1.axvline(threshold_val, color="red", linestyle="--",
            label=f"Global 2σ threshold ({threshold_val:.0f})")
ax1.set_xlabel("Perplexity")
ax1.set_ylabel("Count")
ax1.set_title("Overall Perplexity Distribution")
ax1.legend()

# ── Plot 2: Mean perplexity by tier
ax2 = axes[0, 1]
tier_order = ["T1", "T2", "T3", "T4", "T5", "T6"]
tier_colors = ["#2166ac", "#4393c3", "#92c5de", "#f4a582", "#d6604d", "#b2182b"]
bars = ax2.bar(
    tier_summary["tier"],
    tier_summary["mean_perplexity"],
    color=tier_colors,
    edgecolor="black"
)
ax2.errorbar(
    tier_summary["tier"],
    tier_summary["mean_perplexity"],
    yerr=tier_summary["std_perplexity"],
    fmt="none", color="black", capsize=4
)
ax2.set_xlabel("Tier")
ax2.set_ylabel("Mean Perplexity")
ax2.set_title("Mean Perplexity by Tier")
tier_labels = ["T1\nHumans", "T2\nMammals", "T3\nVertebrates",
               "T4\nInvert+", "T5\nInvert-", "T6\nControls"]
ax2.set_xticklabels(tier_labels)

# ── Plot 3: Perplexity heatmap — template × tier
ax3 = axes[1, 0]
heatmap_data = (df.groupby(["template_id", "tier"])["perplexity"]
                  .mean()
                  .unstack())
sns.heatmap(
    heatmap_data,
    ax=ax3,
    cmap="YlOrRd",
    annot=True,
    fmt=".0f",
    cbar_kws={"label": "Mean Perplexity"}
)
ax3.set_title("Mean Perplexity: Template × Tier")
ax3.set_xlabel("Tier")
ax3.set_ylabel("Template")

# ── Plot 4: Flag rate by entity (top 20 most-flagged)
ax4 = axes[1, 1]
top_flagged = (entity_summary[entity_summary["n_flagged"] > 0]
               .sort_values("n_flagged", ascending=True)
               .tail(20))
colors_by_tier = {
    "T1": "#2166ac", "T2": "#4393c3", "T3": "#92c5de",
    "T4": "#f4a582", "T5": "#d6604d", "T6": "#b2182b"
}
bar_colors = [colors_by_tier[t] for t in top_flagged["tier"]]
ax4.barh(top_flagged["entity"], top_flagged["n_flagged"],
         color=bar_colors, edgecolor="black")
ax4.set_xlabel("Number of Flagged Sentences")
ax4.set_title("Most-Flagged Entities (top 20)")

plt.tight_layout()
plt.savefig("perplexity_screening.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: perplexity_screening.png")


# ── 10. Decision framework & outputs ──────────────────────────────────────

print("\n" + "="*60)
print("RECOMMENDED ACTIONS PER FLAGGED SENTENCE")
print("="*60)

def recommend_action(row):
    """
    Apply decision rules to each flagged sentence.
    Returns one of four recommended actions.
    """
    if not row["flagged"]:
        return "keep"

    # T6 controls are expected to be unnatural in welfare templates
    if row["tier"] == "T6" and row["template_type"] in [
        "pain_suffering", "fear", "moral_consideration", "cross_framing"
    ]:
        return "keep_expected_anomaly"

    # Very high perplexity anywhere else — strong cut signal
    if row["perplexity"] > 1000:
        return "cut"

    # Moderate anomaly — try rewriting first
    if row["global_z"] > 3.0:
        return "rewrite_or_cut"

    # Mild anomaly — flag for manual review
    return "review"


df["recommended_action"] = df.apply(recommend_action, axis=1)

action_summary = df.groupby("recommended_action").size()
print(action_summary.to_string())

# Save full screened corpus
df.to_csv("sentience_salience_corpus_screened.csv", index=False)
print("\nSaved: sentience_salience_corpus_screened.csv")

# Save flagged sentences only for manual review
flagged_output = (df[df["flagged"]]
                  [["sentence_id", "sentence", "entity", "tier",
                     "template_id", "template_type", "perplexity",
                     "global_z", "template_z", "flag_reason",
                     "recommended_action"]]
                  .sort_values("perplexity", ascending=False))

flagged_output.to_csv("flagged_sentences.csv", index=False)
print(f"Saved: flagged_sentences.csv ({len(flagged_output)} rows)")


# ── 11. Anticipated findings & interpretation guide ───────────────────────

print("""
╔══════════════════════════════════════════════════════════════╗
║           INTERPRETATION GUIDE                               ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  EXPECTED high-perplexity patterns (keep, note in paper):    ║
║  • T6 entities in pain/fear/moral templates                  ║
║    "The rock was in pain" — unnatural by design              ║
║  • T5 entities in cross-framing templates                    ║
║    "The nematode trembled with fear" — low frequency phrasing║
║  • Multi-token entities (bumblebee, mantis shrimp)           ║
║    Tokenisation artefacts inflate perplexity                 ║
║                                                              ║
║  UNEXPECTED high-perplexity patterns (investigate):          ║
║  • T1 humans scoring high on any template                    ║
║    → Template wording problem; rewrite                       ║
║  • T2 mammals scoring high on welfare templates              ║
║    → Specific entity unusual (e.g. "whale" in room template) ║
║  • Entire template scoring high across all tiers             ║
║    → Template is globally unnatural; cut or rewrite          ║
║                                                              ║
║  KEY CONFOUND TO CATCH:                                      ║
║  If T6 perplexity >> T5 perplexity on NEUTRAL templates,     ║
║  that is fine — controls should differ from living things.   ║
║  If T5 perplexity >> T4 perplexity on NEUTRAL templates,     ║
║  that suggests frequency/familiarity confound, not           ║
║  sentience encoding — flag for discussion in limitations.    ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 2254.68it/s]


Model loaded: EleutherAI/pythia-1.4b
Device: cpu
Parameters: 1,414,647,808

Corpus loaded: 1500 sentences

Computing perplexity for all sentences...
  Processed 0/1500 sentences...


## Tokenise every entity word and log how many tokens each produces.For multi-token entities, decide on and document your aggregation strategy (mean pooling)

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict
import json
import warnings
warnings.filterwarnings('ignore')

# ── 1. Load tokenizer ──────────────────────────────────────────────────────

MODEL_NAME = "EleutherAI/pythia-1.4b"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"Model max length: {tokenizer.model_max_length}")


# ── 2. Entity list with tier labels ───────────────────────────────────────

entities = [
    # T1: Humans
    ("child", "T1"),
    ("infant", "T1"),
    ("elderly person", "T1"),
    ("worker", "T1"),
    ("patient", "T1"),
    ("prisoner", "T1"),
    ("refugee", "T1"),
    ("soldier", "T1"),
    ("teenager", "T1"),
    ("woman", "T1"),

    # T2: Mammals
    ("dog", "T2"),
    ("chimpanzee", "T2"),
    ("pig", "T2"),
    ("cow", "T2"),
    ("dolphin", "T2"),
    ("elephant", "T2"),
    ("rabbit", "T2"),
    ("rat", "T2"),
    ("horse", "T2"),
    ("whale", "T2"),

    # T3: Non-mammal vertebrates
    ("salmon", "T3"),
    ("chicken", "T3"),
    ("frog", "T3"),
    ("zebrafish", "T3"),
    ("crow", "T3"),
    ("snake", "T3"),
    ("tuna", "T3"),
    ("gecko", "T3"),
    ("trout", "T3"),
    ("pigeon", "T3"),

    # T4: Welfare-evidenced invertebrates
    ("octopus", "T4"),
    ("crab", "T4"),
    ("bee", "T4"),
    ("shrimp", "T4"),
    ("lobster", "T4"),
    ("squid", "T4"),
    ("crayfish", "T4"),
    ("bumblebee", "T4"),
    ("mussel", "T4"),
    ("mantis shrimp", "T4"),

    # T5: Minimal-evidence invertebrates
    ("ant", "T5"),
    ("fly", "T5"),
    ("earthworm", "T5"),
    ("moth", "T5"),
    ("beetle", "T5"),
    ("aphid", "T5"),
    ("nematode", "T5"),
    ("cockroach", "T5"),
    ("mite", "T5"),
    ("woodlouse", "T5"),

    # T6: Non-sentient controls
    ("oak tree", "T6"),
    ("mushroom", "T6"),
    ("bacterium", "T6"),
    ("moss", "T6"),
    ("seaweed", "T6"),
    ("rock", "T6"),
    ("table", "T6"),
    ("cloud", "T6"),
    ("river", "T6"),
    ("statue", "T6"),
]


# ── 3. Core tokenisation audit ────────────────────────────────────────────

def audit_entity_tokenisation(entity: str,
                               tier: str,
                               tokenizer) -> dict:
    """
    Tokenise a single entity and return a full audit record.

    Returns
    -------
    dict with keys:
        entity, tier, n_tokens, token_ids, token_strings,
        is_multi_token, has_unknown_token, token_frequencies
    """
    # Tokenise without special tokens so we get raw subwords
    encoding = tokenizer(
        entity,
        add_special_tokens=False,
        return_tensors="pt"
    )

    token_ids    = encoding["input_ids"][0].tolist()
    token_strings = tokenizer.convert_ids_to_tokens(token_ids)
    n_tokens     = len(token_ids)

    # Check for unknown tokens (OOV signal)
    unk_id       = tokenizer.unk_token_id
    has_unknown  = (unk_id in token_ids) if unk_id is not None else False

    # Also tokenise with "The " prefix — this is how the entity will actually
    # appear in sentences, and some tokenizers produce different splits
    # depending on whether the word is word-initial
    prefixed_encoding = tokenizer(
        f"The {entity}",
        add_special_tokens=False,
        return_tensors="pt"
    )
    prefixed_ids     = prefixed_encoding["input_ids"][0].tolist()
    prefixed_strings = tokenizer.convert_ids_to_tokens(prefixed_ids)

    # Entity tokens in context = all tokens after "The " (1 token)
    # "The" tokenises to 1 token in Pythia's tokenizer
    # Verify by checking the first token
    the_tokens = tokenizer(
        "The",
        add_special_tokens=False
    )["input_ids"]
    n_the_tokens   = len(the_tokens)
    in_context_ids = prefixed_ids[n_the_tokens:]
    in_context_str = prefixed_strings[n_the_tokens:]

    # Flag if context tokenisation differs from standalone
    context_differs = (in_context_ids != token_ids)

    return {
        "entity":              entity,
        "tier":                tier,
        "n_tokens_standalone": n_tokens,
        "n_tokens_in_context": len(in_context_ids),
        "token_ids_standalone":    token_ids,
        "token_ids_in_context":    in_context_ids,
        "token_strings_standalone": token_strings,
        "token_strings_in_context": in_context_str,
        "is_multi_token":      n_tokens > 1,
        "has_unknown_token":   has_unknown,
        "context_differs":     context_differs,
        "is_clean":            (not has_unknown) and (not context_differs),
    }


# Run audit on all entities
print("\nRunning tokenisation audit...")
audit_records = []
for entity, tier in entities:
    record = audit_entity_tokenisation(entity, tier, tokenizer)
    audit_records.append(record)

audit_df = pd.DataFrame(audit_records)

# ── 4. Summary statistics ──────────────────────────────────────────────────

print("\n" + "="*60)
print("TOKENISATION AUDIT — SUMMARY")
print("="*60)

print(f"\nTotal entities audited: {len(audit_df)}")
print(f"Single-token entities:  {(audit_df['n_tokens_in_context'] == 1).sum()}")
print(f"Multi-token entities:   {audit_df['is_multi_token'].sum()}")
print(f"Entities with OOV:      {audit_df['has_unknown_token'].sum()}")
print(f"Context differs:        {audit_df['context_differs'].sum()}")
print(f"Fully clean entities:   {audit_df['is_clean'].sum()}")

print("\n--- Token count distribution ---")
print(audit_df["n_tokens_in_context"].value_counts().sort_index().to_string())

print("\n--- Multi-token entities by tier ---")
multi_by_tier = (audit_df[audit_df["is_multi_token"]]
                 .groupby("tier")
                 .size()
                 .reset_index(name="n_multi_token"))
print(multi_by_tier.to_string(index=False))


# ── 5. Detailed per-entity report ─────────────────────────────────────────

print("\n" + "="*60)
print("PER-ENTITY TOKENISATION DETAIL")
print("="*60)

for _, row in audit_df.sort_values(["tier", "entity"]).iterrows():
    status_flags = []
    if row["is_multi_token"]:
        status_flags.append("MULTI-TOKEN")
    if row["has_unknown_token"]:
        status_flags.append("OOV")
    if row["context_differs"]:
        status_flags.append("CONTEXT-DIFFERS")
    if not status_flags:
        status_flags.append("clean")

    status = " | ".join(status_flags)

    print(f"\n  {row['entity']:20s} ({row['tier']})  [{status}]")
    print(f"    Standalone:  {row['token_strings_standalone']}"
          f"  →  ids: {row['token_ids_standalone']}")
    print(f"    In context:  {row['token_strings_in_context']}"
          f"  →  ids: {row['token_ids_in_context']}")
    if row["context_differs"]:
        print(f"    ⚠ Tokenisation changes in sentential context")


# ── 6. Aggregation strategy — mean pooling ────────────────────────────────

print("\n" + "="*60)
print("AGGREGATION STRATEGY: MEAN POOLING")
print("="*60)

AGGREGATION_STRATEGY = "mean_pool"

print(f"""
Chosen strategy: {AGGREGATION_STRATEGY}

Rationale
─────────
For multi-token entities (e.g. "mantis shrimp" → ["mant", "is", "shrimp"]),
we extract the residual stream activation at each subword token position
and take the element-wise mean across all subword positions.

This is preferred over alternatives for the following reasons:

  Last-token only
    Pro:  Simple; consistent with autoregressive prediction literature
    Con:  Discards information from earlier subwords; the last token of
          "mantis shrimp" is "shrimp" alone, which has its own meaning
          independent of "mantis" — this would introduce systematic bias
          for compound entities

  First-token only
    Pro:  Consistent with some BERT-era probing literature
    Con:  Same problem in reverse; loses compound-word semantics

  Mean pooling  ✓ CHOSEN
    Pro:  Retains information from all subwords; standard in sentence
          embedding literature; most defensible for compound nouns
          where all subwords contribute to entity meaning
    Con:  Slightly more complex to implement; assumes equal contribution
          from each subword (reasonable for short compounds)

  Weighted pooling (by attention)
    Pro:  Could weight subwords by their contextual importance
    Con:  Circular for this study (we are analysing attention patterns);
          adds complexity without clear benefit at this scale

Implementation note
───────────────────
Mean pooling will be applied at every layer independently.
For a 3-subword entity at layer L:
    activation = mean([h_L[pos_1], h_L[pos_2], h_L[pos_3]])
where h_L[pos_i] is the residual stream vector at position i, layer L.
""")


def get_entity_token_positions(sentence: str,
                                entity: str,
                                tokenizer) -> list[int]:
    """
    Find the token positions of an entity within a full sentence.
    Returns list of integer positions (0-indexed, including BOS if present).

    Strategy: tokenise full sentence, then find the contiguous span
    matching the entity's in-context token ids.
    """
    # Full sentence token ids
    full_ids = tokenizer(
        sentence,
        add_special_tokens=True,
        return_tensors="pt"
    )["input_ids"][0].tolist()

    # Entity token ids in context (without special tokens)
    entity_ids = tokenizer(
        f"The {entity}",
        add_special_tokens=False
    )["input_ids"]
    the_ids    = tokenizer(
        "The",
        add_special_tokens=False
    )["input_ids"]
    entity_in_context_ids = entity_ids[len(the_ids):]

    # Search for contiguous match in full sentence
    n_entity = len(entity_in_context_ids)
    for start_pos in range(len(full_ids) - n_entity + 1):
        if full_ids[start_pos:start_pos + n_entity] == entity_in_context_ids:
            return list(range(start_pos, start_pos + n_entity))

    # Fallback: return empty list and log warning
    print(f"  WARNING: Could not locate entity '{entity}' in sentence: {sentence}")
    return []


def mean_pool_activations(activations: torch.Tensor,
                           positions: list[int]) -> torch.Tensor:
    """
    Mean-pool residual stream activations across entity token positions.

    Parameters
    ----------
    activations : torch.Tensor
        Shape (seq_len, hidden_dim) — residual stream at one layer
        for one sentence
    positions : list[int]
        Token positions belonging to the entity

    Returns
    -------
    torch.Tensor of shape (hidden_dim,)
    """
    if not positions:
        raise ValueError("No positions provided for mean pooling")

    entity_vectors = activations[positions, :]  # (n_entity_tokens, hidden_dim)
    return entity_vectors.mean(dim=0)           # (hidden_dim,)


# ── 7. Validate position-finding on sample sentences ──────────────────────

print("\n" + "="*60)
print("POSITION-FINDING VALIDATION")
print("="*60)

test_cases = [
    ("The child was in pain.", "child"),
    ("The mantis shrimp was in pain.", "mantis shrimp"),
    ("The bumblebee felt fear as the predator approached.", "bumblebee"),
    ("The elderly person was in pain.", "elderly person"),
    ("The oak tree was in pain.", "oak tree"),
    ("Everyone could see that the earthworm was suffering.", "earthworm"),
    ("Scientists studied whether the nematode could suffer.", "nematode"),
    ("The bacterium's interests were considered in the decision.", "bacterium"),
]

validation_records = []
for sentence, entity in test_cases:
    positions = get_entity_token_positions(sentence, entity, tokenizer)
    full_ids  = tokenizer(sentence, add_special_tokens=True)["input_ids"]
    full_strs = tokenizer.convert_ids_to_tokens(full_ids)

    entity_tokens_found = [full_strs[p] for p in positions]

    status = "✓" if positions else "✗ NOT FOUND"
    print(f"\n  Entity: '{entity}'")
    print(f"  Sentence: {sentence}")
    print(f"  Full tokens: {full_strs}")
    print(f"  Entity positions: {positions}  →  tokens: {entity_tokens_found}  {status}")

    validation_records.append({
        "entity":         entity,
        "sentence":       sentence,
        "positions":      positions,
        "tokens_found":   entity_tokens_found,
        "n_positions":    len(positions),
        "position_found": len(positions) > 0,
    })

val_df = pd.DataFrame(validation_records)
success_rate = val_df["position_found"].mean()
print(f"\nValidation success rate: {success_rate:.0%} ({val_df['position_found'].sum()}/{len(val_df)})")


# ── 8. Build entity lookup table ──────────────────────────────────────────

print("\n" + "="*60)
print("ENTITY LOOKUP TABLE")
print("="*60)

lookup_table = {}
for _, row in audit_df.iterrows():
    lookup_table[row["entity"]] = {
        "tier":                    row["tier"],
        "n_tokens":                row["n_tokens_in_context"],
        "token_strings":           row["token_strings_in_context"],
        "token_ids":               row["token_ids_in_context"],
        "is_multi_token":          row["is_multi_token"],
        "aggregation_strategy":    AGGREGATION_STRATEGY,
        "context_differs":         row["context_differs"],
        "requires_position_check": row["is_multi_token"] or row["context_differs"],
    }

# Save lookup table
with open("entity_tokenisation_lookup.json", "w") as f:
    json.dump(lookup_table, f, indent=2)

print("Saved: entity_tokenisation_lookup.json")

# Print clean summary table
print("\n  Entity lookup table (abridged — multi-token and flagged only):\n")
print(f"  {'Entity':<20} {'Tier':<5} {'N tokens':<10} {'Tokens':<35} {'Notes'}")
print(f"  {'-'*20} {'-'*5} {'-'*10} {'-'*35} {'-'*20}")

for entity, info in lookup_table.items():
    if info["requires_position_check"]:
        note = "⚠ mean pool" if info["is_multi_token"] else "⚠ context differs"
        print(f"  {entity:<20} {info['tier']:<5} {info['n_tokens']:<10} "
              f"{str(info['token_strings']):<35} {note}")


# ── 9. Visualisations ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle("Tokenisation Audit Results", fontsize=14, fontweight="bold")

tier_colors = {
    "T1": "#2166ac", "T2": "#4393c3", "T3": "#92c5de",
    "T4": "#f4a582", "T5": "#d6604d", "T6": "#b2182b"
}

# ── Plot 1: Token count by entity coloured by tier
ax1 = axes[0]
colors = [tier_colors[t] for t in audit_df["tier"]]
bars   = ax1.barh(
    audit_df["entity"],
    audit_df["n_tokens_in_context"],
    color=colors,
    edgecolor="black",
    linewidth=0.5
)
ax1.axvline(1, color="black", linestyle="--", alpha=0.5, label="Single token")
ax1.set_xlabel("Number of tokens (in context)")
ax1.set_title("Token Count per Entity")
ax1.set_xlim(0, audit_df["n_tokens_in_context"].max() + 1)

legend_patches = [
    mpatches.Patch(color=c, label=t)
    for t, c in tier_colors.items()
]
ax1.legend(handles=legend_patches, loc="lower right", fontsize=8)

# ── Plot 2: Proportion multi-token by tier
ax2 = axes[1]
tier_multi = audit_df.groupby("tier").apply(
    lambda x: x["is_multi_token"].mean() * 100
).reset_index(name="pct_multi")
ax2.bar(
    tier_multi["tier"],
    tier_multi["pct_multi"],
    color=[tier_colors[t] for t in tier_multi["tier"]],
    edgecolor="black"
)
ax2.set_xlabel("Tier")
ax2.set_ylabel("% Multi-token entities")
ax2.set_title("Multi-token Rate by Tier")
ax2.set_ylim(0, 100)
for i, row in tier_multi.iterrows():
    ax2.text(i, row["pct_multi"] + 2, f"{row['pct_multi']:.0f}%",
             ha="center", fontsize=9)

# ── Plot 3: Token count distribution
ax3 = axes[2]
token_counts = audit_df["n_tokens_in_context"].value_counts().sort_index()
ax3.bar(
    token_counts.index,
    token_counts.values,
    color="steelblue",
    edgecolor="black"
)
ax3.set_xlabel("Number of tokens")
ax3.set_ylabel("Number of entities")
ax3.set_title("Token Count Distribution")
ax3.set_xticks(token_counts.index)

plt.tight_layout()
plt.savefig("tokenisation_audit.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: tokenisation_audit.png")


# ── 10. Methods text — ready to paste into paper ──────────────────────────

print("""
╔══════════════════════════════════════════════════════════════╗
║  METHODS TEXT (paste into paper)                             ║
╠══════════════════════════════════════════════════════════════╣

Tokenisation and Aggregation Strategy
──────────────────────────────────────
All 60 entities were tokenised using the Pythia tokenizer both
in isolation and in sentential context ("The [entity]...") to
detect cases where tokenisation differs by position. Token
positions were identified by locating the contiguous subword
span matching each entity's in-context token ids within the
full sentence token sequence.

For single-token entities, residual stream activations were
extracted directly at the entity token position. For multi-
token entities, activations were extracted at each subword
position and aggregated via element-wise mean pooling across
all subword positions, applied independently at each layer.
This strategy was preferred over last-token or first-token
extraction because it retains information from all subwords
contributing to entity meaning, which is particularly
important for compound nouns (e.g. "mantis shrimp", "oak
tree") where no single subword adequately represents the full
entity concept.

Multi-token entities were flagged in all analyses and results
were verified to be robust to their inclusion or exclusion.
The entity lookup table, tokenisation audit, and position-
finding validation code are available in the project
repository.

╚══════════════════════════════════════════════════════════════╝
""")


# ── 11. Final audit summary saved to CSV ──────────────────────────────────

audit_output = audit_df[[
    "entity", "tier",
    "n_tokens_standalone", "n_tokens_in_context",
    "token_strings_standalone", "token_strings_in_context",
    "is_multi_token", "has_unknown_token",
    "context_differs", "is_clean"
]]

audit_output.to_csv("tokenisation_audit.csv", index=False)
print("Saved: tokenisation_audit.csv")

print("\n── Audit complete ──")
print("Output files:")
print("  tokenisation_audit.csv          — full per-entity audit")
print("  entity_tokenisation_lookup.json — lookup table for extraction pipeline")
print("  tokenisation_audit.png          — visualisations")

## Build a lookup table: entity → token indices in each template sentence

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer
import json
import warnings
warnings.filterwarnings('ignore')

# ── 1. Load tokenizer ──────────────────────────────────────────────────────

MODEL_NAME = "EleutherAI/pythia-1.4b"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size:,}")


# ── 2. Templates & entities ────────────────────────────────────────────────

templates = [
    ("T01", "pain_suffering",       "The [ENTITY] was in pain."),
    ("T02", "pain_suffering",       "Everyone could see that the [ENTITY] was suffering."),
    ("T03", "pain_suffering",       "The [ENTITY] experienced significant physical distress during the procedure."),
    ("T04", "pain_suffering",       "After the injury, the [ENTITY] showed clear signs of pain."),
    ("T05", "fear",                 "The [ENTITY] felt fear as the predator approached."),
    ("T06", "fear",                 "The [ENTITY] was visibly terrified by what was happening."),
    ("T07", "fear",                 "A sense of dread overcame the [ENTITY] as the door closed."),
    ("T08", "fear",                 "The [ENTITY] froze, overwhelmed by fear."),
    ("T09", "scientific_welfare",   "Scientists studied whether the [ENTITY] could suffer."),
    ("T10", "scientific_welfare",   "Researchers debated whether the [ENTITY] was capable of experiencing pain."),
    ("T11", "scientific_welfare",   "The study aimed to determine whether the [ENTITY] had subjective experiences."),
    ("T12", "scientific_welfare",   "Evidence suggested the [ENTITY] might be capable of feeling distress."),
    ("T13", "moral_consideration",  "The [ENTITY]'s interests were considered in the decision."),
    ("T14", "moral_consideration",  "Policymakers agreed that the [ENTITY]'s wellbeing deserved protection."),
    ("T15", "moral_consideration",  "The committee debated how much moral weight to give the [ENTITY]."),
    ("T16", "moral_consideration",  "It was decided that the [ENTITY] deserved to be treated with care."),
    ("T17", "neutral_control",      "The [ENTITY] moved across the room."),
    ("T18", "neutral_control",      "The [ENTITY] was observed near the window."),
    ("T19", "neutral_control",      "Someone noticed the [ENTITY] on the other side of the field."),
    ("T20", "neutral_control",      "The [ENTITY] remained still for several minutes."),
    ("T21", "cross_framing",        "The [ENTITY] was clearly frightened, and those responsible agreed its distress mattered."),
    ("T22", "cross_framing",        "Despite the [ENTITY]'s visible terror, no one considered whether it deserved protection."),
    ("T23", "cross_framing",        "The committee agreed that the [ENTITY]'s fear was a morally relevant factor in their decision."),
    ("T24", "cross_framing",        "It was impossible to ignore that the [ENTITY] was suffering, and everyone felt they had a responsibility to help."),
    ("T25", "cross_framing",        "The [ENTITY] trembled with fear, yet the question of whether its experience counted remained unresolved."),
]

entities = [
    # T1
    ("child", "T1"), ("infant", "T1"), ("elderly person", "T1"),
    ("worker", "T1"), ("patient", "T1"), ("prisoner", "T1"),
    ("refugee", "T1"), ("soldier", "T1"), ("teenager", "T1"),
    ("woman", "T1"),
    # T2
    ("dog", "T2"), ("chimpanzee", "T2"), ("pig", "T2"),
    ("cow", "T2"), ("dolphin", "T2"), ("elephant", "T2"),
    ("rabbit", "T2"), ("rat", "T2"), ("horse", "T2"),
    ("whale", "T2"),
    # T3
    ("salmon", "T3"), ("chicken", "T3"), ("frog", "T3"),
    ("zebrafish", "T3"), ("crow", "T3"), ("snake", "T3"),
    ("tuna", "T3"), ("gecko", "T3"), ("trout", "T3"),
    ("pigeon", "T3"),
    # T4
    ("octopus", "T4"), ("crab", "T4"), ("bee", "T4"),
    ("shrimp", "T4"), ("lobster", "T4"), ("squid", "T4"),
    ("crayfish", "T4"), ("bumblebee", "T4"), ("mussel", "T4"),
    ("mantis shrimp", "T4"),
    # T5
    ("ant", "T5"), ("fly", "T5"), ("earthworm", "T5"),
    ("moth", "T5"), ("beetle", "T5"), ("aphid", "T5"),
    ("nematode", "T5"), ("cockroach", "T5"), ("mite", "T5"),
    ("woodlouse", "T5"),
    # T6
    ("oak tree", "T6"), ("mushroom", "T6"), ("bacterium", "T6"),
    ("moss", "T6"), ("seaweed", "T6"), ("rock", "T6"),
    ("table", "T6"), ("cloud", "T6"), ("river", "T6"),
    ("statue", "T6"),
]


# ── 3. Core position-finding function ─────────────────────────────────────

def find_entity_positions(sentence: str,
                           entity: str,
                           tokenizer,
                           verbose: bool = False) -> dict:
    """
    Locate the token positions of an entity within a fully tokenised sentence.

    Returns a dict containing:
        positions       — list of int token indices (0-indexed incl. BOS)
        token_strings   — subword strings at those positions
        token_ids       — token ids at those positions
        n_entity_tokens — number of tokens the entity spans
        found           — bool; False if matching failed
        failure_reason  — str if found=False, else None
        full_token_strings — complete sentence token list (for inspection)
    """

    # ── Tokenise full sentence (with BOS if model uses one)
    full_enc     = tokenizer(sentence, add_special_tokens=True,
                             return_tensors="pt")
    full_ids     = full_enc["input_ids"][0].tolist()
    full_strings = tokenizer.convert_ids_to_tokens(full_ids)

    # ── Determine entity ids as they appear in sentential context
    #    Strategy: tokenise "...the [entity]..." and "...the ..."
    #    then subtract to isolate entity subwords
    #    We use the literal pre-entity string from the sentence
    #    to capture any context-dependent tokenisation shift

    entity_start_char = sentence.find(entity)
    if entity_start_char == -1:
        # Possessive templates: "[ENTITY]'s" — entity substring is present
        # but may be followed immediately by apostrophe; handle by searching
        # for entity ignoring case
        entity_lower = entity.lower()
        sentence_lower = sentence.lower()
        entity_start_char = sentence_lower.find(entity_lower)
        if entity_start_char == -1:
            return {
                "positions":          [],
                "token_strings":      [],
                "token_ids":          [],
                "n_entity_tokens":    0,
                "found":              False,
                "failure_reason":     "entity string not found in sentence",
                "full_token_strings": full_strings,
            }

    entity_end_char = entity_start_char + len(entity)

    # ── Use character-to-token mapping via offset mapping
    full_enc_with_offsets = tokenizer(
        sentence,
        add_special_tokens=True,
        return_tensors="pt",
        return_offsets_mapping=True
    )

    offsets = full_enc_with_offsets["offset_mapping"][0].tolist()
    # offsets[i] = (char_start, char_end) for token i
    # Special tokens have offset (0, 0)

    entity_positions = []
    for idx, (char_start, char_end) in enumerate(offsets):
        # Skip special tokens
        if char_start == 0 and char_end == 0:
            continue
        # Token overlaps with entity character span
        if char_start < entity_end_char and char_end > entity_start_char:
            entity_positions.append(idx)

    if not entity_positions:
        # Fallback: subword-matching search
        entity_ids_standalone = tokenizer(
            entity,
            add_special_tokens=False
        )["input_ids"]

        n = len(entity_ids_standalone)
        found_fallback = False
        for start in range(len(full_ids) - n + 1):
            if full_ids[start:start + n] == entity_ids_standalone:
                entity_positions = list(range(start, start + n))
                found_fallback = True
                break

        if not found_fallback:
            return {
                "positions":          [],
                "token_strings":      [],
                "token_ids":          [],
                "n_entity_tokens":    0,
                "found":              False,
                "failure_reason":     "offset mapping and fallback both failed",
                "full_token_strings": full_strings,
            }

    entity_token_ids     = [full_ids[p] for p in entity_positions]
    entity_token_strings = [full_strings[p] for p in entity_positions]

    if verbose:
        print(f"  Sentence : {sentence}")
        print(f"  Entity   : '{entity}'  (chars {entity_start_char}–{entity_end_char})")
        print(f"  All tokens: {full_strings}")
        print(f"  Entity positions: {entity_positions}")
        print(f"  Entity tokens   : {entity_token_strings}")

    return {
        "positions":          entity_positions,
        "token_strings":      entity_token_strings,
        "token_ids":          entity_token_ids,
        "n_entity_tokens":    len(entity_positions),
        "found":              True,
        "failure_reason":     None,
        "full_token_strings": full_strings,
    }


# ── 4. Build the full lookup table ────────────────────────────────────────

print("\nBuilding lookup table...")
print(f"  {len(templates)} templates × {len(entities)} entities"
      f" = {len(templates) * len(entities):,} entries\n")

lookup = {}          # primary nested lookup: lookup[entity][template_id]
flat_records = []    # flat list for DataFrame export
failures     = []    # collect any failures for inspection

for entity, tier in entities:
    lookup[entity] = {
        "tier":    tier,
        "entries": {}
    }

    for tid, ttype, template in templates:

        sentence = template.replace("[ENTITY]", entity)

        result = find_entity_positions(sentence, entity, tokenizer)

        entry = {
            "template_id":        tid,
            "template_type":      ttype,
            "sentence":           sentence,
            "positions":          result["positions"],
            "token_strings":      result["token_strings"],
            "token_ids":          result["token_ids"],
            "n_entity_tokens":    result["n_entity_tokens"],
            "found":              result["found"],
            "failure_reason":     result["failure_reason"],
            "full_token_strings": result["full_token_strings"],
            "is_multi_token":     result["n_entity_tokens"] > 1,
            "aggregation":        "mean_pool" if result["n_entity_tokens"] > 1
                                  else "direct",
        }

        lookup[entity]["entries"][tid] = entry

        flat_records.append({
            "entity":           entity,
            "tier":             tier,
            "template_id":      tid,
            "template_type":    ttype,
            "sentence":         sentence,
            "positions":        result["positions"],
            "token_strings":    result["token_strings"],
            "n_entity_tokens":  result["n_entity_tokens"],
            "found":            result["found"],
            "failure_reason":   result["failure_reason"],
            "is_multi_token":   result["n_entity_tokens"] > 1,
            "aggregation":      entry["aggregation"],
        })

        if not result["found"]:
            failures.append({
                "entity":   entity,
                "tier":     tier,
                "template": tid,
                "sentence": sentence,
                "reason":   result["failure_reason"],
            })

flat_df = pd.DataFrame(flat_records)

print(f"Lookup table built: {len(flat_df):,} entries")
print(f"Failures: {len(failures)}")
print(f"Multi-token entries: {flat_df['is_multi_token'].sum():,}"
      f" ({100*flat_df['is_multi_token'].mean():.1f}%)")


# ── 5. Failure report ──────────────────────────────────────────────────────

if failures:
    print("\n" + "="*60)
    print("FAILURES — REQUIRE MANUAL RESOLUTION")
    print("="*60)
    for f in failures:
        print(f"\n  [{f['entity']} × {f['template']}]")
        print(f"  Sentence : {f['sentence']}")
        print(f"  Reason   : {f['reason']}")
else:
    print("\n✓ No failures — all 1,500 positions resolved successfully")


# ── 6. Consistency checks ──────────────────────────────────────────────────

print("\n" + "="*60)
print("CONSISTENCY CHECKS")
print("="*60)

# Check 1: entity always maps to same number of tokens across templates
print("\nCheck 1: Token count consistency per entity across templates")
token_count_consistency = (
    flat_df.groupby("entity")["n_entity_tokens"]
    .nunique()
    .reset_index(name="n_distinct_counts")
)
inconsistent = token_count_consistency[
    token_count_consistency["n_distinct_counts"] > 1
]
if len(inconsistent) == 0:
    print("  ✓ All entities tokenise to the same number of tokens"
          " across all templates")
else:
    print(f"  ⚠ {len(inconsistent)} entities show inconsistent token counts:")
    for _, row in inconsistent.iterrows():
        entity_rows = flat_df[flat_df["entity"] == row["entity"]]
        counts = entity_rows.groupby("n_entity_tokens")["template_id"].apply(list)
        print(f"\n    Entity: '{row['entity']}'")
        for n, tids in counts.items():
            print(f"      {n} tokens in templates: {tids}")

# Check 2: positions always found
print("\nCheck 2: Position resolution rate")
resolution_rate = flat_df["found"].mean()
print(f"  Resolution rate: {resolution_rate:.4%}"
      f" ({flat_df['found'].sum()} / {len(flat_df)})")

# Check 3: token strings match expected entity subwords
print("\nCheck 3: Spot-check token strings for known multi-token entities")
spot_checks = [
    ("mantis shrimp", "T01"),
    ("elderly person", "T01"),
    ("bumblebee",      "T01"),
    ("oak tree",       "T01"),
    ("earthworm",      "T01"),
]
for entity, tid in spot_checks:
    if entity in lookup and tid in lookup[entity]["entries"]:
        entry = lookup[entity]["entries"][tid]
        print(f"  '{entity}' in {tid}: "
              f"positions={entry['positions']} "
              f"tokens={entry['token_strings']}")

# Check 4: possessive templates resolved correctly
print("\nCheck 4: Possessive template resolution")
possessive_templates = ["T13", "T14", "T22", "T23"]
for entity, tier in [("child", "T1"), ("bee", "T4"), ("oak tree", "T6")]:
    for tid in possessive_templates:
        if tid in lookup[entity]["entries"]:
            entry = lookup[entity]["entries"][tid]
            print(f"  '{entity}' × {tid}: "
                  f"positions={entry['positions']} "
                  f"tokens={entry['token_strings']} "
                  f"sentence='{entry['sentence'][:60]}...'")


# ── 7. Summary statistics ──────────────────────────────────────────────────

print("\n" + "="*60)
print("LOOKUP TABLE SUMMARY STATISTICS")
print("="*60)

print("\nToken count distribution across all 1,500 entries:")
print(flat_df["n_entity_tokens"].value_counts()
      .sort_index()
      .to_string())

print("\nMulti-token rate by tier:")
tier_multi = (flat_df.groupby("tier")["is_multi_token"]
              .agg(["sum", "count", "mean"])
              .rename(columns={"sum": "n_multi",
                               "count": "n_total",
                               "mean": "pct_multi"}))
tier_multi["pct_multi"] = (tier_multi["pct_multi"] * 100).round(1)
print(tier_multi.to_string())

print("\nMulti-token rate by template type:")
tmpl_multi = (flat_df.groupby("template_type")["is_multi_token"]
              .agg(["sum", "count", "mean"])
              .rename(columns={"sum": "n_multi",
                               "count": "n_total",
                               "mean": "pct_multi"}))
tmpl_multi["pct_multi"] = (tmpl_multi["pct_multi"] * 100).round(1)
print(tmpl_multi.to_string())

print("\nEntities requiring mean pooling (multi-token):")
multi_entities = (flat_df[flat_df["is_multi_token"]]
                  [["entity", "tier", "n_entity_tokens", "token_strings"]]
                  .drop_duplicates(subset=["entity"])
                  .sort_values(["tier", "n_entity_tokens"],
                               ascending=[True, False]))
for _, row in multi_entities.iterrows():
    print(f"  {row['entity']:<20} ({row['tier']})  "
          f"{row['n_entity_tokens']} tokens  →  {row['token_strings']}")


# ── 8. Serialise lookup table ──────────────────────────────────────────────

# JSON export — strip full_token_strings to keep file size manageable
lookup_slim = {}
for entity, data in lookup.items():
    lookup_slim[entity] = {
        "tier": data["tier"],
        "entries": {}
    }
    for tid, entry in data["entries"].items():
        lookup_slim[entity]["entries"][tid] = {
            "template_type":   entry["template_type"],
            "sentence":        entry["sentence"],
            "positions":       entry["positions"],
            "token_strings":   entry["token_strings"],
            "token_ids":       entry["token_ids"],
            "n_entity_tokens": entry["n_entity_tokens"],
            "found":           entry["found"],
            "is_multi_token":  entry["is_multi_token"],
            "aggregation":     entry["aggregation"],
        }

with open("entity_position_lookup.json", "w") as f:
    json.dump(lookup_slim, f, indent=2)
print("\nSaved: entity_position_lookup.json")

# CSV export of flat table
flat_df_export = flat_df.copy()
flat_df_export["positions"]     = flat_df_export["positions"].apply(str)
flat_df_export["token_strings"] = flat_df_export["token_strings"].apply(str)
flat_df_export.to_csv("entity_position_lookup.csv", index=False)
print("Saved: entity_position_lookup.csv")


# ── 9. Extraction-ready helper ─────────────────────────────────────────────

def get_positions(entity: str,
                  template_id: str,
                  lookup: dict) -> list[int]:
    """
    Retrieve token positions for an entity in a specific template.
    Call this inside the activation extraction loop.

    Usage
    -----
    positions = get_positions("bee", "T01", lookup)
    # → [2]  (single token, direct extraction)

    positions = get_positions("mantis shrimp", "T01", lookup)
    # → [2, 3, 4]  (multi-token, mean pool across positions)
    """
    try:
        entry = lookup[entity]["entries"][template_id]
        if not entry["found"]:
            raise ValueError(
                f"Position not found for '{entity}' × '{template_id}': "
                f"{entry['failure_reason']}"
            )
        return entry["positions"]
    except KeyError:
        raise KeyError(
            f"Entity '{entity}' or template '{template_id}' "
            f"not in lookup table"
        )


def get_aggregation(entity: str, lookup: dict) -> str:
    """
    Return the aggregation strategy for an entity.
    'direct'    — single token, extract directly
    'mean_pool' — multiple tokens, average activations
    """
    first_entry = next(iter(lookup[entity]["entries"].values()))
    return first_entry["aggregation"]


# Quick demo
print("\n── Extraction helper demo ──")
demo_cases = [
    ("child",         "T01"),
    ("bee",           "T05"),
    ("mantis shrimp", "T01"),
    ("elderly person","T07"),
    ("oak tree",      "T13"),
]
for entity, tid in demo_cases:
    positions    = get_positions(entity, tid, lookup_slim)
    aggregation  = get_aggregation(entity, lookup_slim)
    tokens       = lookup_slim[entity]["entries"][tid]["token_strings"]
    print(f"  {entity:<20} × {tid}  →  "
          f"positions={positions}  tokens={tokens}  "
          f"aggregation={aggregation}")


# ── 10. Unit tests ─────────────────────────────────────────────────────────

print("\n── Unit tests ──")

def run_unit_tests(lookup, flat_df):
    tests_passed = 0
    tests_failed = 0

    def check(condition, name):
        nonlocal tests_passed, tests_failed
        if condition:
            print(f"  ✓ {name}")
            tests_passed += 1
        else:
            print(f"  ✗ FAIL: {name}")
            tests_failed += 1

    # Total entry count
    check(len(flat_df) == 1500,
          f"Total entries == 1500 (got {len(flat_df)})")

    # All found
    check(flat_df["found"].all(),
          "All 1500 entries resolved successfully")

    # Single-token entities have exactly 1 position
    single_token = flat_df[~flat_df["is_multi_token"]]
    check((single_token["n_entity_tokens"] == 1).all(),
          "All single-token entities have exactly 1 position")

    # Aggregation strategy assigned correctly
    check((flat_df[flat_df["is_multi_token"]]["aggregation"] == "mean_pool").all(),
          "Multi-token entities assigned mean_pool aggregation")
    check((flat_df[~flat_df["is_multi_token"]]["aggregation"] == "direct").all(),
          "Single-token entities assigned direct aggregation")

    # Known single-token entity spot check
    child_t01 = flat_df[
        (flat_df["entity"] == "child") &
        (flat_df["template_id"] == "T01")
    ]
    check(len(child_t01) == 1 and child_t01.iloc[0]["n_entity_tokens"] == 1,
          "'child' in T01 is single token")

    # Known multi-token entity spot check
    ms_t01 = flat_df[
        (flat_df["entity"] == "mantis shrimp") &
        (flat_df["template_id"] == "T01")
    ]
    check(len(ms_t01) == 1 and ms_t01.iloc[0]["n_entity_tokens"] > 1,
          "'mantis shrimp' in T01 is multi-token")

    # Positions are non-negative integers
    all_positions = flat_df["positions"].explode().dropna()
    check((all_positions >= 0).all(),
          "All positions are non-negative")

    # Each entity appears in exactly 25 templates
    entity_counts = flat_df.groupby("entity").size()
    check((entity_counts == 25).all(),
          "Each entity appears in exactly 25 template entries")

    # Each template has exactly 60 entity entries
    template_counts = flat_df.groupby("template_id").size()
    check((template_counts == 60).all(),
          "Each template has exactly 60 entity entries")

    print(f"\n  {tests_passed} passed / {tests_failed} failed")
    return tests_failed == 0

all_passed = run_unit_tests(lookup_slim, flat_df)

print("\n── Output files ──")
print("  entity_position_lookup.json   nested lookup for extraction pipeline")
print("  entity_position_lookup.csv    flat table for inspection & QA")
print(f"\n── Done ──  All tests passed: {all_passed}")

What Each Test Catches ^^
T1 empty positions - Extraction silently skips entity; NaN activations downstream
T2 integer types - Numpy int64 causes PyTorch indexing failures in some versions
T3 non-negative - Negative index wraps to end of sequence; wrong token extracted
T4 in bounds - Index error at extraction time; or silent tensor wrap
T5 contiguous - Non-entity tokens included in mean pool; corrupts representation
T6 special tokens - BOS/EOS representation averaged into entity vector
T7 decoded match - Positions point to entirely wrong word — undetectable otherwise
T8 count mismatch- Lookup and extractor disagree on how many tokens to pool
T9 id mismatch- Sentence modified after lookup built; stale positions usedT
10 entity absent - Template substitution failed silently
T11 pool shape - Mean pool crashes during extraction; breaks entire run
T12 duplicates - Token double-counted in pool; representation biased
T13 unsorted - Attention extraction assumes ascending order; wrong heads read
T14 single/multi - Extra positions found for single-token entity; wrong span
T15 off-by-one - Systematic shift left; every entity extracted one position early

## Write a unit test that checks entity token position is correctly identified for every sentence - this is the most common source of silent errors in this kind of study

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer
import json
import warnings
import traceback
from dataclasses import dataclass, field
from typing import Optional
from collections import defaultdict

warnings.filterwarnings('ignore')

#  Load tokenizer & lookup

MODEL_NAME = "EleutherAI/pythia-1.4b"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

with open("entity_position_lookup.json") as f:
    lookup = json.load(f)

df = pd.read_csv("entity_position_lookup.csv")


# TEST RESULT DATACLASS

@dataclass
class TestResult:
    """
    Holds the result of a single test assertion.
    Designed to accumulate across all 1,500 entries so the
    full failure surface is visible in one run — not just
    the first failure.
    """
    test_name:    str
    entity:       str
    tier:         str
    template_id:  str
    sentence:     str
    passed:       bool
    failure_msg:  Optional[str] = None
    expected:     Optional[object] = None
    actual:       Optional[object] = None

    def __str__(self):
        status = "✓" if self.passed else "✗"
        base   = (f"  [{status}] {self.test_name:<40} "
                  f"entity='{self.entity}' × {self.template_id}")
        if not self.passed:
            base += (f"\n       sentence : {self.sentence}"
                     f"\n       expected : {self.expected}"
                     f"\n       actual   : {self.actual}"
                     f"\n       detail   : {self.failure_msg}")
        return base


@dataclass
class TestSuite:
    """Accumulates results and produces structured reports."""
    name:    str
    results: list = field(default_factory=list)

    def add(self, result: TestResult):
        self.results.append(result)

    @property
    def passed(self):  return [r for r in self.results if r.passed]
    @property
    def failed(self):  return [r for r in self.results if not r.passed]
    @property
    def n_total(self): return len(self.results)
    @property
    def n_passed(self):return len(self.passed)
    @property
    def n_failed(self):return len(self.failed)
    @property
    def pass_rate(self):
        return self.n_passed / self.n_total if self.n_total else 0

    def report(self, show_passing: bool = False,
               max_failures: int = 50) -> str:
        lines = [
            "",
            "═" * 70,
            f"  TEST SUITE: {self.name}",
            f"  {self.n_passed}/{self.n_total} passed  "
            f"({self.pass_rate:.1%})  |  "
            f"{self.n_failed} failures",
            "═" * 70,
        ]
        if show_passing:
            for r in self.passed[:20]:
                lines.append(str(r))

        if self.failed:
            lines.append(f"\n  ── FAILURES (showing first {max_failures}) ──")
            for r in self.failed[:max_failures]:
                lines.append(str(r))
            if self.n_failed > max_failures:
                lines.append(f"\n  ... and {self.n_failed - max_failures} more")

        return "\n".join(lines)

    def failures_by_entity(self) -> dict:
        counts = defaultdict(int)
        for r in self.failed:
            counts[r.entity] += 1
        return dict(sorted(counts.items(),
                           key=lambda x: x[1], reverse=True))

    def failures_by_template(self) -> dict:
        counts = defaultdict(int)
        for r in self.failed:
            counts[r.template_id] += 1
        return dict(sorted(counts.items(),
                           key=lambda x: x[1], reverse=True))

    def failures_by_tier(self) -> dict:
        counts = defaultdict(int)
        for r in self.failed:
            counts[r.tier] += 1
        return dict(sorted(counts.items(),
                           key=lambda x: x[1], reverse=True))


# HELPER FUNCTIONS

def decode_positions(sentence: str,
                     positions: list,
                     tokenizer) -> list[str]:
    """
    Decode token ids at given positions back to strings.
    Used to verify positions point to the right tokens.
    """
    if not positions:
        return []
    ids = tokenizer(sentence, add_special_tokens=True)["input_ids"]
    return [tokenizer.decode([ids[p]]) for p in positions]


def reconstruct_entity_from_positions(sentence: str,
                                       positions: list,
                                       tokenizer) -> str:
    """
    Reconstruct the entity string by decoding and concatenating
    the tokens at the given positions. Used to verify the
    reconstructed string matches the original entity name.
    """
    if not positions:
        return ""
    ids     = tokenizer(sentence, add_special_tokens=True)["input_ids"]
    subids  = [ids[p] for p in positions]
    decoded = tokenizer.decode(subids, skip_special_tokens=True)
    # Strip leading whitespace artifact from Ġ tokens
    return decoded.strip()


def get_full_tokens(sentence: str, tokenizer) -> list[str]:
    ids = tokenizer(sentence, add_special_tokens=True)["input_ids"]
    return tokenizer.convert_ids_to_tokens(ids)


def entity_appears_in_sentence(entity: str, sentence: str) -> bool:
    return entity.lower() in sentence.lower()


def positions_are_contiguous(positions: list) -> bool:
    if len(positions) <= 1:
        return True
    return all(positions[i+1] == positions[i] + 1
               for i in range(len(positions) - 1))


def positions_in_bounds(positions: list, sentence: str,
                         tokenizer) -> bool:
    ids = tokenizer(sentence, add_special_tokens=True)["input_ids"]
    n   = len(ids)
    return all(0 <= p < n for p in positions)


def no_overlap_with_special_tokens(positions: list,
                                    sentence: str,
                                    tokenizer) -> bool:
    """
    Ensure entity positions don't overlap with BOS/EOS/PAD tokens.
    Special tokens typically sit at position 0 (BOS) and -1 (EOS).
    """
    enc          = tokenizer(sentence, add_special_tokens=True,
                             return_tensors="pt")
    ids          = enc["input_ids"][0].tolist()
    special_ids  = set(tokenizer.all_special_ids)
    special_pos  = {i for i, tok in enumerate(ids) if tok in special_ids}
    return not any(p in special_pos for p in positions)


def simulate_mean_pool(sentence: str,
                        positions: list,
                        hidden_dim: int = 512) -> torch.Tensor:
    """
    Simulate mean pooling with random activations to verify
    the pooling operation produces expected output shape.
    """
    seq_len      = len(tokenizer(sentence,
                                  add_special_tokens=True)["input_ids"])
    fake_acts    = torch.randn(seq_len, hidden_dim)
    entity_vecs  = fake_acts[positions, :]
    pooled       = entity_vecs.mean(dim=0)
    return pooled


# INDIVIDUAL TEST FUNCTIONS

def test_positions_not_empty(entity, tier, tid, sentence,
                              positions) -> TestResult:
    """
    T1 — Positions list must be non-empty.
    The most fundamental check: if positions is empty,
    no activation can ever be extracted.
    """
    passed = len(positions) > 0
    return TestResult(
        test_name   = "T1_positions_not_empty",
        entity      = entity,
        tier        = tier,
        template_id = tid,
        sentence    = sentence,
        passed      = passed,
        failure_msg = "positions list is empty — entity was not located",
        expected    = "> 0 positions",
        actual      = f"{len(positions)} positions",
    )


def test_positions_are_integers(entity, tier, tid, sentence,
                                 positions) -> TestResult:
    """
    T2 — All positions must be plain Python ints.
    Numpy int64 values can cause silent indexing failures
    in PyTorch tensors.
    """
    bad = [p for p in positions if not isinstance(p, int)]
    passed = len(bad) == 0
    return TestResult(
        test_name   = "T2_positions_are_integers",
        entity      = entity,
        tier        = tier,
        template_id = tid,
        sentence    = sentence,
        passed      = passed,
        failure_msg = f"non-int positions: {bad}",
        expected    = "all int",
        actual      = [type(p).__name__ for p in bad],
    )


def test_positions_non_negative(entity, tier, tid, sentence,
                                 positions) -> TestResult:
    """
    T3 — All positions must be >= 0.
    Negative indices would silently index from the end of the
    sequence, pointing to completely wrong tokens.
    """
    bad    = [p for p in positions if p < 0]
    passed = len(bad) == 0
    return TestResult(
        test_name   = "T3_positions_non_negative",
        entity      = entity,
        tier        = tier,
        template_id = tid,
        sentence    = sentence,
        passed      = passed,
        failure_msg = f"negative positions found: {bad}",
        expected    = "all >= 0",
        actual      = bad,
    )


def test_positions_in_bounds(entity, tier, tid, sentence,
                              positions, tokenizer) -> TestResult:
    """
    T4 — All positions must be within the token sequence length.
    Out-of-bounds indices raise errors at extraction time,
    but only if you're lucky — some tensor ops silently
    wrap around.
    """
    in_bounds = positions_in_bounds(positions, sentence, tokenizer)
    seq_len   = len(tokenizer(sentence,
                               add_special_tokens=True)["input_ids"])
    bad       = [p for p in positions if p >= seq_len]
    return TestResult(
        test_name   = "T4_positions_in_bounds",
        entity      = entity,
        tier        = tier,
        template_id = tid,
        sentence    = sentence,
        passed      = in_bounds,
        failure_msg = f"positions {bad} >= seq_len {seq_len}",
        expected    = f"all < {seq_len}",
        actual      = bad or "all in bounds",
    )


def test_positions_contiguous(entity, tier, tid, sentence,
                               positions) -> TestResult:
    """
    T5 — Positions must be contiguous (no gaps).
    A gap between positions indicates the entity span was
    mis-identified and non-entity tokens are being included.
    This is one of the most common silent errors.
    """
    contiguous = positions_are_contiguous(positions)
    return TestResult(
        test_name   = "T5_positions_contiguous",
        entity      = entity,
        tier        = tier,
        template_id = tid,
        sentence    = sentence,
        passed      = contiguous,
        failure_msg = (f"positions {positions} are not contiguous — "
                       f"gaps at: "
                       f"{[positions[i] for i in range(len(positions)-1) if positions[i+1] != positions[i]+1]}"),
        expected    = "contiguous span",
        actual      = positions,
    )


def test_no_special_token_overlap(entity, tier, tid, sentence,
                                   positions, tokenizer) -> TestResult:
    """
    T6 — Entity positions must not overlap with special tokens.
    BOS/EOS/PAD tokens at position 0 or end of sequence
    represent the model's sequence boundary markers, not
    entity content.
    """
    clean = no_overlap_with_special_tokens(positions, sentence, tokenizer)
    return TestResult(
        test_name   = "T6_no_special_token_overlap",
        entity      = entity,
        tier        = tier,
        template_id = tid,
        sentence    = sentence,
        passed      = clean,
        failure_msg = f"positions {positions} include special token positions",
        expected    = "no overlap with special tokens",
        actual      = positions,
    )


def test_decoded_tokens_contain_entity(entity, tier, tid, sentence,
                                        positions, tokenizer) -> TestResult:
    """
    T7 — Decoding the tokens at the identified positions must
    reconstruct a string that matches the entity name.
    This is the ground truth check: do the positions actually
    point to the entity, not some other word in the sentence?

    Matching is case-insensitive and strips whitespace to
    handle Ġ-prefixed tokens gracefully.
    """
    reconstructed = reconstruct_entity_from_positions(
        sentence, positions, tokenizer
    )
    # Normalise both for comparison
    recon_norm  = reconstructed.lower().strip()
    entity_norm = entity.lower().strip()

    # Accept if entity is a substring of reconstruction or vice versa
    # (handles cases where apostrophe or punctuation bleeds into span)
    passed = (entity_norm in recon_norm) or (recon_norm in entity_norm)

    return TestResult(
        test_name   = "T7_decoded_tokens_match_entity",
        entity      = entity,
        tier        = tier,
        template_id = tid,
        sentence    = sentence,
        passed      = passed,
        failure_msg = (f"decoded tokens '{reconstructed}' do not match "
                       f"entity '{entity}'"),
        expected    = entity,
        actual      = reconstructed,
    )


def test_token_count_matches_lookup(entity, tier, tid, sentence,
                                    positions, lookup) -> TestResult:
    """
    T8 — Number of positions must match the n_entity_tokens
    field stored in the lookup table.
    Catches cases where position-finding returns a different
    number of tokens than was recorded during the audit.
    """
    expected_n = lookup[entity]["entries"][tid]["n_entity_tokens"]
    actual_n   = len(positions)
    passed     = (actual_n == expected_n)
    return TestResult(
        test_name   = "T8_token_count_matches_lookup",
        entity      = entity,
        tier        = tier,
        template_id = tid,
        sentence    = sentence,
        passed      = passed,
        failure_msg = (f"position count {actual_n} != "
                       f"lookup n_entity_tokens {expected_n}"),
        expected    = expected_n,
        actual      = actual_n,
    )


def test_token_ids_match_lookup(entity, tier, tid, sentence,
                                 positions, lookup, tokenizer) -> TestResult:
    """
    T9 — Token ids at identified positions must match the
    token ids recorded in the lookup table.
    Catches cases where the same positions resolve to different
    tokens — e.g. if the sentence was modified after lookup
    construction.
    """
    stored_ids = lookup[entity]["entries"][tid]["token_ids"]
    full_ids   = tokenizer(sentence, add_special_tokens=True)["input_ids"]
    actual_ids = [full_ids[p] for p in positions]
    passed     = (actual_ids == stored_ids)
    return TestResult(
        test_name   = "T9_token_ids_match_lookup",
        entity      = entity,
        tier        = tier,
        template_id = tid,
        sentence    = sentence,
        passed      = passed,
        failure_msg = (f"token ids at positions {positions} are {actual_ids} "
                       f"but lookup stores {stored_ids}"),
        expected    = stored_ids,
        actual      = actual_ids,
    )


def test_entity_appears_in_sentence(entity, tier, tid,
                                     sentence) -> TestResult:
    """
    T10 — Entity string must appear as a substring in the sentence.
    A precondition check: if the entity isn't in the sentence
    at all, any positions found are definitely wrong.
    """
    appears = entity_appears_in_sentence(entity, sentence)
    return TestResult(
        test_name   = "T10_entity_in_sentence",
        entity      = entity,
        tier        = tier,
        template_id = tid,
        sentence    = sentence,
        passed      = appears,
        failure_msg = f"'{entity}' not found as substring in sentence",
        expected    = f"'{entity}' present",
        actual      = sentence,
    )


def test_mean_pool_output_shape(entity, tier, tid, sentence,
                                 positions, tokenizer,
                                 hidden_dim: int = 512) -> TestResult:
    """
    T11 — Simulated mean pooling must produce a vector of
    shape (hidden_dim,). Verifies that the pooling operation
    over the identified positions is dimensionally valid
    before any real model activations are extracted.
    """
    try:
        pooled = simulate_mean_pool(sentence, positions, hidden_dim)
        passed = (pooled.shape == torch.Size([hidden_dim]))
        return TestResult(
            test_name   = "T11_mean_pool_shape_valid",
            entity      = entity,
            tier        = tier,
            template_id = tid,
            sentence    = sentence,
            passed      = passed,
            failure_msg = f"pooled shape {pooled.shape} != ({hidden_dim},)",
            expected    = torch.Size([hidden_dim]),
            actual      = pooled.shape,
        )
    except Exception as e:
        return TestResult(
            test_name   = "T11_mean_pool_shape_valid",
            entity      = entity,
            tier        = tier,
            template_id = tid,
            sentence    = sentence,
            passed      = False,
            failure_msg = f"mean pool raised exception: {e}",
            expected    = torch.Size([hidden_dim]),
            actual      = str(e),
        )


def test_no_duplicate_positions(entity, tier, tid, sentence,
                                 positions) -> TestResult:
    """
    T12 — No position should appear twice in the list.
    Duplicate positions cause a token to be double-counted
    in mean pooling, biasing the entity vector toward
    that token's representation.
    """
    has_dups = len(positions) != len(set(positions))
    dups     = [p for p in positions if positions.count(p) > 1]
    return TestResult(
        test_name   = "T12_no_duplicate_positions",
        entity      = entity,
        tier        = tier,
        template_id = tid,
        sentence    = sentence,
        passed      = not has_dups,
        failure_msg = f"duplicate positions found: {dups}",
        expected    = "no duplicates",
        actual      = positions,
    )


def test_positions_sorted_ascending(entity, tier, tid, sentence,
                                     positions) -> TestResult:
    """
    T13 — Positions must be in ascending order.
    Out-of-order positions don't affect mean pooling numerically
    but can cause subtle bugs in attention pattern extraction
    and visualisation code that assumes left-to-right order.
    """
    sorted_pos = sorted(positions)
    passed     = (positions == sorted_pos)
    return TestResult(
        test_name   = "T13_positions_sorted_ascending",
        entity      = entity,
        tier        = tier,
        template_id = tid,
        sentence    = sentence,
        passed      = passed,
        failure_msg = f"positions {positions} are not sorted ascending",
        expected    = sorted_pos,
        actual      = positions,
    )


def test_single_token_entity_has_one_position(entity, tier, tid,
                                               sentence,
                                               positions,
                                               lookup) -> TestResult:
    """
    T14 — Entities marked as single-token in lookup must have
    exactly one position. Guards against the extraction pipeline
    accidentally finding extra positions for a known single-token
    entity — e.g. if the entity string also appears elsewhere
    in the sentence.
    """
    is_multi  = lookup[entity]["entries"][tid]["is_multi_token"]
    if is_multi:
        # Not applicable — skip by returning a passing dummy result
        return TestResult(
            test_name   = "T14_single_token_has_one_position",
            entity      = entity,
            tier        = tier,
            template_id = tid,
            sentence    = sentence,
            passed      = True,
            failure_msg = None,
        )
    passed = (len(positions) == 1)
    return TestResult(
        test_name   = "T14_single_token_has_one_position",
        entity      = entity,
        tier        = tier,
        template_id = tid,
        sentence    = sentence,
        passed      = passed,
        failure_msg = (f"single-token entity has {len(positions)} positions: "
                       f"{positions}"),
        expected    = 1,
        actual      = len(positions),
    )


def test_entity_not_in_wrong_position(entity, tier, tid, sentence,
                                       positions, tokenizer) -> TestResult:
    """
    T15 — The token immediately before the entity span must NOT
    decode to the entity string. Catches off-by-one errors where
    the position finder returns positions shifted left by one —
    a particularly insidious silent error because the tokens at
    position-1 often look plausible (e.g. "The" before "child").
    """
    if not positions:
        return TestResult(
            test_name   = "T15_no_off_by_one_left",
            entity      = entity,
            tier        = tier,
            template_id = tid,
            sentence    = sentence,
            passed      = False,
            failure_msg = "cannot check: positions is empty",
        )

    first_pos   = positions[0]
    passed      = True
    failure_msg = None

    if first_pos > 0:
        pre_pos  = first_pos - 1
        pre_recon = reconstruct_entity_from_positions(
            sentence, [pre_pos], tokenizer
        )
        # If the token before the span reconstructs to the entity,
        # we have an off-by-one
        if entity.lower() in pre_recon.lower():
            passed      = False
            failure_msg = (f"token at position {pre_pos} (before span) "
                           f"decodes to '{pre_recon}' which contains entity "
                           f"'{entity}' — possible off-by-one shift")

    return TestResult(
        test_name   = "T15_no_off_by_one_left",
        entity      = entity,
        tier        = tier,
        template_id = tid,
        sentence    = sentence,
        passed      = passed,
        failure_msg = failure_msg,
        expected    = f"token before span != '{entity}'",
        actual      = None,
    )


# MASTER TEST RUNNER

ALL_TESTS = [
    test_positions_not_empty,
    test_positions_are_integers,
    test_positions_non_negative,
    test_positions_in_bounds,
    test_positions_contiguous,
    test_no_special_token_overlap,
    test_decoded_tokens_contain_entity,
    test_token_count_matches_lookup,
    test_token_ids_match_lookup,
    test_entity_appears_in_sentence,
    test_mean_pool_output_shape,
    test_no_duplicate_positions,
    test_positions_sorted_ascending,
    test_single_token_entity_has_one_position,
    test_entity_not_in_wrong_position,
]


def run_all_tests(lookup: dict,
                  tokenizer,
                  show_passing: bool = False) -> dict[str, TestSuite]:
    """
    Run all 15 tests across all 1,500 entity × template entries.
    Returns one TestSuite per test, keyed by test name.
    Total assertions: 15 × 1,500 = 22,500
    """

    # Initialise one suite per test
    suites = {
        t.__name__: TestSuite(name=t.__name__)
        for t in ALL_TESTS
    }

    total_entries = sum(
        len(data["entries"])
        for data in lookup.values()
    )
    print(f"Running {len(ALL_TESTS)} tests × "
          f"{total_entries} entries = "
          f"{len(ALL_TESTS) * total_entries:,} assertions...\n")

    n_processed = 0

    for entity, data in lookup.items():
        tier = data["tier"]

        for tid, entry in data["entries"].items():
            sentence  = entry["sentence"]
            positions = entry["positions"]

            # ── Run each test ──────────────────────────────────────────────

            suites["test_positions_not_empty"].add(
                test_positions_not_empty(
                    entity, tier, tid, sentence, positions))

            suites["test_positions_are_integers"].add(
                test_positions_are_integers(
                    entity, tier, tid, sentence, positions))

            suites["test_positions_non_negative"].add(
                test_positions_non_negative(
                    entity, tier, tid, sentence, positions))

            suites["test_positions_in_bounds"].add(
                test_positions_in_bounds(
                    entity, tier, tid, sentence, positions, tokenizer))

            suites["test_positions_contiguous"].add(
                test_positions_contiguous(
                    entity, tier, tid, sentence, positions))

            suites["test_no_special_token_overlap"].add(
                test_no_special_token_overlap(
                    entity, tier, tid, sentence, positions, tokenizer))

            suites["test_decoded_tokens_contain_entity"].add(
                test_decoded_tokens_contain_entity(
                    entity, tier, tid, sentence, positions, tokenizer))

            suites["test_token_count_matches_lookup"].add(
                test_token_count_matches_lookup(
                    entity, tier, tid, sentence, positions, lookup))

            suites["test_token_ids_match_lookup"].add(
                test_token_ids_match_lookup(
                    entity, tier, tid, sentence, positions, lookup, tokenizer))

            suites["test_entity_appears_in_sentence"].add(
                test_entity_appears_in_sentence(
                    entity, tier, tid, sentence))

            suites["test_mean_pool_output_shape"].add(
                test_mean_pool_output_shape(
                    entity, tier, tid, sentence, positions, tokenizer))

            suites["test_no_duplicate_positions"].add(
                test_no_duplicate_positions(
                    entity, tier, tid, sentence, positions))

            suites["test_positions_sorted_ascending"].add(
                test_positions_sorted_ascending(
                    entity, tier, tid, sentence, positions))

            suites["test_single_token_entity_has_one_position"].add(
                test_single_token_entity_has_one_position(
                    entity, tier, tid, sentence, positions, lookup))

            suites["test_entity_not_in_wrong_position"].add(
                test_entity_not_in_wrong_position(
                    entity, tier, tid, sentence, positions, tokenizer))

            n_processed += 1
            if n_processed % 300 == 0:
                print(f"  processed {n_processed}/{total_entries} entries...")

    return suites


# RUN & REPORT

suites = run_all_tests(lookup, tokenizer)

# ── Per-suite reports ──────────────────────────────────────────────────────
for name, suite in suites.items():
    print(suite.report(show_passing=False))

# ── Master summary ─────────────────────────────────────────────────────────
total_assertions = sum(s.n_total  for s in suites.values())
total_passed     = sum(s.n_passed for s in suites.values())
total_failed     = sum(s.n_failed for s in suites.values())

print("\n" + "═" * 70)
print("  MASTER SUMMARY")
print("═" * 70)
print(f"  Total assertions : {total_assertions:,}")
print(f"  Passed           : {total_passed:,}  ({total_passed/total_assertions:.1%})")
print(f"  Failed           : {total_failed:,}")

# ── Failure hotspots ───────────────────────────────────────────────────────
if total_failed > 0:
    print("\n  ── Failure hotspots ──")

    all_failures = [r for s in suites.values() for r in s.failed]

    # By entity
    by_entity = defaultdict(int)
    for r in all_failures:
        by_entity[r.entity] += 1
    print("\n  Top entities by failure count:")
    for entity, n in sorted(by_entity.items(),
                             key=lambda x: x[1], reverse=True)[:10]:
        tier = lookup[entity]["tier"]
        print(f"    {entity:<20} ({tier})  {n} failures")

    # By template
    by_template = defaultdict(int)
    for r in all_failures:
        by_template[r.template_id] += 1
    print("\n  Top templates by failure count:")
    for tid, n in sorted(by_template.items(),
                          key=lambda x: x[1], reverse=True)[:10]:
        print(f"    {tid}   {n} failures")

    # By tier
    by_tier = defaultdict(int)
    for r in all_failures:
        by_tier[r.tier] += 1
    print("\n  Failures by tier:")
    for tier in ["T1", "T2", "T3", "T4", "T5", "T6"]:
        n = by_tier.get(tier, 0)
        print(f"    {tier}   {n} failures")

    # By test
    print("\n  Failures by test:")
    for name, suite in suites.items():
        if suite.n_failed > 0:
            print(f"    {name:<50} {suite.n_failed} failures")

# ── Actionable resolution guide ────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  RESOLUTION GUIDE                                                    ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  T1  positions_not_empty                                             ║
║      → Re-run position finder; check entity string in template       ║
║                                                                      ║
║  T2  positions_are_integers                                          ║
║      → Cast positions to int() after extraction                      ║
║                                                                      ║
║  T4  positions_in_bounds                                             ║
║      → Sequence length mismatch; check add_special_tokens flag       ║
║                                                                      ║
║  T5  positions_contiguous                                            ║
║      → Entity string appears non-contiguously; check template wording║
║                                                                      ║
║  T6  no_special_token_overlap                                        ║
║      → BOS at position 0 captured; shift positions by +1             ║
║                                                                      ║
║  T7  decoded_tokens_match_entity   ← MOST IMPORTANT                  ║
║      → Positions point to wrong tokens; rerun offset mapping         ║
║                                                                      ║
║  T9  token_ids_match_lookup                                          ║
║      → Sentence was modified after lookup was built; rebuild lookup  ║
║                                                                      ║
║  T14 single_token_has_one_position                                   ║
║      → Entity appears twice in sentence; tighten match logic         ║
║                                                                      ║
║  T15 no_off_by_one_left                                              ║
║      → Off-by-one in position finder; check char offset mapping      ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")

# ── Save failure log ───────────────────────────────────────────────────────
failure_records = []
for name, suite in suites.items():
    for r in suite.failed:
        failure_records.append({
            "test_name":   r.test_name,
            "entity":      r.entity,
            "tier":        r.tier,
            "template_id": r.template_id,
            "sentence":    r.sentence,
            "failure_msg": r.failure_msg,
            "expected":    str(r.expected),
            "actual":      str(r.actual),
        })

if failure_records:
    pd.DataFrame(failure_records).to_csv("test_failures.csv", index=False)
    print(f"Saved: test_failures.csv  ({len(failure_records)} failure records)")
else:
    print("\n✓ All 22,500 assertions passed — corpus is clean for extraction")

## Write extract_activations(sentence, entity_token_idx, model, cache) function that returns:  
- Residual stream at entity position for all layers
- MLP output at entity position for all layers
- Attention pattern to/from entity token for all layers

Run on 10 test sentences; inspect output shapes and values: make sure nothing is suspiciously uniform or NaN

Store outputs as a structured dataframe or HDF5 file with columns: sentence_id, entity, tier, layer, activation_vector

In [ ]:
import torch
import pandas as pd
import numpy as np
import h5py
import json
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
! pip install transformer_lens
from transformer_lens import HookedTransformer
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')


# ══════════════════════════════════════════════════════════════════════════════
# 1. MODEL LOADING
# ══════════════════════════════════════════════════════════════════════════════

MODEL_NAME = "pythia-1.4b"   # swap to "pythia-1.4b" for full run

print(f"Loading {MODEL_NAME}...")
model = HookedTransformer.from_pretrained(MODEL_NAME)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)

N_LAYERS     = model.cfg.n_layers
N_HEADS      = model.cfg.n_heads
D_MODEL      = model.cfg.d_model
D_MLP        = model.cfg.d_mlp

print(f"  Device    : {device}")
print(f"  Layers    : {N_LAYERS}")
print(f"  Heads     : {N_HEADS}")
print(f"  d_model   : {D_MODEL}")
print(f"  d_mlp     : {D_MLP}")

# Load lookup table built in previous step
with open("entity_position_lookup.json") as f:
    lookup = json.load(f)


# ══════════════════════════════════════════════════════════════════════════════
# 2. DATACLASSES FOR STRUCTURED OUTPUT
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class LayerActivations:
    """
    All activations extracted at a single layer for a single sentence.
    All tensors are on CPU and detached before storage.
    """
    layer:              int

    # Residual stream at entity token position(s) — shape (d_model,)
    # For multi-token entities this is already mean-pooled
    resid_post:         torch.Tensor

    # MLP output at entity position — shape (d_model,)
    mlp_out:            torch.Tensor

    # Attention pattern FROM entity token to all other tokens
    # shape (n_heads, seq_len) — how much entity attends to each position
    attn_pattern_from:  torch.Tensor

    # Attention pattern TO entity token from all other tokens
    # shape (n_heads, seq_len) — how much each position attends to entity
    attn_pattern_to:    torch.Tensor


@dataclass
class SentenceActivations:
    """
    Complete activation record for one sentence × entity pair.
    Contains LayerActivations for every layer.
    """
    sentence_id:   str
    entity:        str
    tier:          str
    template_id:   str
    template_type: str
    sentence:      str
    positions:     list[int]
    n_tokens:      int
    layers:        list[LayerActivations] = field(default_factory=list)

    def get_layer(self, layer: int) -> Optional[LayerActivations]:
        matches = [l for l in self.layers if l.layer == layer]
        return matches[0] if matches else None


# ══════════════════════════════════════════════════════════════════════════════
# 3. CORE EXTRACTION FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def extract_activations(sentence:    str,
                         positions:   list[int],
                         model:       HookedTransformer,
                         cache=None) -> list[LayerActivations]:
    """
    Extract residual stream, MLP output, and attention patterns
    at entity token position(s) for every layer.

    Parameters
    ----------
    sentence  : str
        The full sentence to process.
    positions : list[int]
        Token positions of the entity. Single-token entities pass
        [idx]; multi-token entities pass all subword positions —
        activations are mean-pooled across positions.
    model     : HookedTransformer
        Loaded TransformerLens model.
    cache     : ActivationCache, optional
        Pre-computed cache. If None, model.run_with_cache is called.
        Pass a cache to avoid redundant forward passes when extracting
        multiple entity positions from the same sentence.

    Returns
    -------
    list[LayerActivations]  — one entry per layer, length = n_layers
    """

    if not positions:
        raise ValueError("positions list is empty — cannot extract activations")

    # ── Forward pass with cache ────────────────────────────────────────────
    tokens = model.to_tokens(sentence)          # (1, seq_len)
    seq_len = tokens.shape[1]

    # Validate positions against actual sequence length
    for p in positions:
        if p >= seq_len:
            raise IndexError(
                f"Position {p} out of bounds for sequence length {seq_len}. "
                f"Sentence: '{sentence}'"
            )

    if cache is None:
        with torch.no_grad():
            _, cache = model.run_with_cache(tokens)

    layer_activations = []

    for layer in range(model.cfg.n_layers):

        # ── Residual stream post layer norm — shape (1, seq_len, d_model)
        resid_post = cache[f"blocks.{layer}.hook_resid_post"]
        # Extract entity positions and mean-pool  → (d_model,)
        resid_entity = resid_post[0, positions, :].mean(dim=0).cpu()

        # ── MLP output — shape (1, seq_len, d_model)
        mlp_out      = cache[f"blocks.{layer}.hook_mlp_out"]
        mlp_entity   = mlp_out[0, positions, :].mean(dim=0).cpu()

        # ── Attention patterns — shape (1, n_heads, seq_len, seq_len)
        #    attn_pattern[b, h, q, k] = softmax weight from query q to key k
        attn = cache[f"blocks.{layer}.attn.hook_pattern"]  # (1,H,S,S)

        # FROM entity: how much entity token(s) attend to every other token
        # Average across entity positions if multi-token → (n_heads, seq_len)
        attn_from = attn[0, :, positions, :].mean(dim=1).cpu()
        # shape: (n_heads, seq_len)

        # TO entity: how much every token attends to entity token(s)
        # Average across entity positions → (n_heads, seq_len)
        attn_to = attn[0, :, :, positions].mean(dim=2).cpu()
        # shape: (n_heads, seq_len)

        layer_activations.append(LayerActivations(
            layer             = layer,
            resid_post        = resid_entity,   # (d_model,)
            mlp_out           = mlp_entity,     # (d_model,)
            attn_pattern_from = attn_from,      # (n_heads, seq_len)
            attn_pattern_to   = attn_to,        # (n_heads, seq_len)
        ))

    return layer_activations


# ══════════════════════════════════════════════════════════════════════════════
# 4. BATCH EXTRACTION WRAPPER
# ══════════════════════════════════════════════════════════════════════════════

def extract_sentence(sentence_id:   str,
                      entity:        str,
                      tier:          str,
                      template_id:   str,
                      template_type: str,
                      sentence:      str,
                      positions:     list[int],
                      model:         HookedTransformer) -> SentenceActivations:
    """
    Run a full extraction for one sentence and return a
    SentenceActivations object containing all layers.
    """
    tokens  = model.to_tokens(sentence)
    n_tokens = tokens.shape[1]

    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)

    layers = extract_activations(sentence, positions, model, cache)

    return SentenceActivations(
        sentence_id   = sentence_id,
        entity        = entity,
        tier          = tier,
        template_id   = template_id,
        template_type = template_type,
        sentence      = sentence,
        positions     = positions,
        n_tokens      = n_tokens,
        layers        = layers,
    )


# ══════════════════════════════════════════════════════════════════════════════
# 5. TEST SENTENCES
# ══════════════════════════════════════════════════════════════════════════════

test_cases = [
    # (sentence_id, entity, tier, template_id, template_type)
    ("T01_child",         "child",         "T1", "T01", "pain_suffering"),
    ("T01_dog",           "dog",           "T2", "T01", "pain_suffering"),
    ("T01_salmon",        "salmon",        "T3", "T01", "pain_suffering"),
    ("T01_octopus",       "octopus",       "T4", "T01", "pain_suffering"),
    ("T01_bee",           "bee",           "T4", "T01", "pain_suffering"),
    ("T01_ant",           "ant",           "T5", "T01", "pain_suffering"),
    ("T01_mantis_shrimp", "mantis shrimp", "T4", "T01", "pain_suffering"),
    ("T01_earthworm",     "earthworm",     "T5", "T01", "pain_suffering"),
    ("T01_oak_tree",      "oak tree",      "T6", "T01", "pain_suffering"),
    ("T01_rock",          "rock",          "T6", "T01", "pain_suffering"),
]

# Build full sentences and retrieve positions from lookup
test_records = []
for sid, entity, tier, tid, ttype in test_cases:
    entry    = lookup[entity]["entries"][tid]
    sentence  = entry["sentence"]
    positions = entry["positions"]
    test_records.append((sid, entity, tier, tid, ttype, sentence, positions))

print(f"\nTest sentences ({len(test_records)}):")
for sid, entity, tier, tid, ttype, sentence, positions in test_records:
    print(f"  [{sid}]  positions={positions}  '{sentence}'")


# ══════════════════════════════════════════════════════════════════════════════
# 6. RUN EXTRACTION ON TEST SENTENCES
# ══════════════════════════════════════════════════════════════════════════════

print(f"\nExtracting activations...")
results: list[SentenceActivations] = []

for sid, entity, tier, tid, ttype, sentence, positions in test_records:
    print(f"  {sid}...", end=" ")
    try:
        sa = extract_sentence(
            sentence_id   = sid,
            entity        = entity,
            tier          = tier,
            template_id   = tid,
            template_type = ttype,
            sentence      = sentence,
            positions     = positions,
            model         = model,
        )
        results.append(sa)
        print(f"✓  ({N_LAYERS} layers)")
    except Exception as e:
        print(f"✗  ERROR: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# 7. SHAPE INSPECTION
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═"*70)
print("  SHAPE INSPECTION")
print("═"*70)

for sa in results[:3]:   # show first 3 in detail
    print(f"\n  [{sa.sentence_id}]  entity='{sa.entity}'  "
          f"n_tokens={sa.n_tokens}  positions={sa.positions}")
    for layer_idx in [0, N_LAYERS//2, N_LAYERS-1]:
        la = sa.get_layer(layer_idx)
        print(f"    Layer {layer_idx:2d}:")
        print(f"      resid_post        : {tuple(la.resid_post.shape)}"
              f"  expected ({D_MODEL},)")
        print(f"      mlp_out           : {tuple(la.mlp_out.shape)}"
              f"  expected ({D_MODEL},)")
        print(f"      attn_pattern_from : {tuple(la.attn_pattern_from.shape)}"
              f"  expected ({N_HEADS}, {sa.n_tokens})")
        print(f"      attn_pattern_to   : {tuple(la.attn_pattern_to.shape)}"
              f"  expected ({N_HEADS}, {sa.n_tokens})")

# Assert shapes for all results
print("\n  Shape assertions (all layers, all test sentences):")
shape_errors = []
for sa in results:
    for la in sa.layers:
        if la.resid_post.shape != (D_MODEL,):
            shape_errors.append(f"{sa.sentence_id} L{la.layer} resid_post: "
                                 f"{la.resid_post.shape}")
        if la.mlp_out.shape != (D_MODEL,):
            shape_errors.append(f"{sa.sentence_id} L{la.layer} mlp_out: "
                                 f"{la.mlp_out.shape}")
        if la.attn_pattern_from.shape != (N_HEADS, sa.n_tokens):
            shape_errors.append(f"{sa.sentence_id} L{la.layer} attn_from: "
                                 f"{la.attn_pattern_from.shape}")
        if la.attn_pattern_to.shape != (N_HEADS, sa.n_tokens):
            shape_errors.append(f"{sa.sentence_id} L{la.layer} attn_to: "
                                 f"{la.attn_pattern_to.shape}")

if shape_errors:
    print(f"  ✗ {len(shape_errors)} shape errors:")
    for e in shape_errors:
        print(f"    {e}")
else:
    n_checks = len(results) * N_LAYERS * 4
    print(f"  ✓ All {n_checks:,} shape checks passed")


# ══════════════════════════════════════════════════════════════════════════════
# 8. NaN / UNIFORMITY INSPECTION
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═"*70)
print("  NaN AND UNIFORMITY INSPECTION")
print("═"*70)

@dataclass
class HealthReport:
    sentence_id: str
    entity:      str
    layer:       int
    component:   str
    has_nan:     bool
    has_inf:     bool
    is_uniform:  bool       # std < 1e-6
    mean:        float
    std:         float
    min_val:     float
    max_val:     float
    n_zero:      int        # how many dimensions are exactly 0


def inspect_tensor(t: torch.Tensor) -> dict:
    """Compute health statistics for a 1D or 2D tensor."""
    flat = t.flatten().float()
    return {
        "has_nan":    torch.isnan(flat).any().item(),
        "has_inf":    torch.isinf(flat).any().item(),
        "is_uniform": flat.std().item() < 1e-6,
        "mean":       flat.mean().item(),
        "std":        flat.std().item(),
        "min_val":    flat.min().item(),
        "max_val":    flat.max().item(),
        "n_zero":     (flat == 0).sum().item(),
    }


health_records = []
for sa in results:
    for la in sa.layers:
        for component, tensor in [
            ("resid_post",        la.resid_post),
            ("mlp_out",           la.mlp_out),
            ("attn_pattern_from", la.attn_pattern_from),
            ("attn_pattern_to",   la.attn_pattern_to),
        ]:
            stats = inspect_tensor(tensor)
            health_records.append(HealthReport(
                sentence_id = sa.sentence_id,
                entity      = sa.entity,
                layer       = la.layer,
                component   = component,
                **stats
            ))

health_df = pd.DataFrame([vars(r) for r in health_records])

# ── Summary
n_nan     = health_df["has_nan"].sum()
n_inf     = health_df["has_inf"].sum()
n_uniform = health_df["is_uniform"].sum()

print(f"\n  Total tensors inspected : {len(health_df):,}")
print(f"  Contains NaN            : {n_nan}  {'✓' if n_nan == 0 else '✗ PROBLEM'}")
print(f"  Contains Inf            : {n_inf}  {'✓' if n_inf == 0 else '✗ PROBLEM'}")
print(f"  Suspiciously uniform    : {n_uniform}  "
      f"{'✓' if n_uniform == 0 else '⚠ INVESTIGATE'}")

# ── Per-component stats
print("\n  Statistics by component:")
comp_stats = (health_df.groupby("component")
              .agg(mean_mean=("mean", "mean"),
                   mean_std=("std", "mean"),
                   n_nan=("has_nan", "sum"),
                   n_uniform=("is_uniform", "sum"))
              .round(4))
print(comp_stats.to_string())

# ── Per-sentence spot check
print("\n  Per-sentence spot check (layer 0 and final layer):")
print(f"  {'sentence_id':<22} {'entity':<15} {'layer':>5} "
      f"{'component':<18} {'mean':>8} {'std':>8} {'nan':>5}")
print(f"  {'-'*22} {'-'*15} {'-'*5} {'-'*18} {'-'*8} {'-'*8} {'-'*5}")

for sa in results:
    for layer_idx in [0, N_LAYERS-1]:
        la = sa.get_layer(layer_idx)
        for component, tensor in [("resid_post", la.resid_post),
                                   ("mlp_out",    la.mlp_out)]:
            stats = inspect_tensor(tensor)
            print(f"  {sa.sentence_id:<22} {sa.entity:<15} "
                  f"{layer_idx:>5} {component:<18} "
                  f"{stats['mean']:>8.3f} {stats['std']:>8.3f} "
                  f"{'YES' if stats['has_nan'] else 'no':>5}")

# ── Attention pattern row-sum check (should sum to 1.0 per head)
print("\n  Attention pattern row-sum check (from-entity rows should sum to 1.0):")
attn_sum_errors = []
for sa in results:
    for la in sa.layers:
        row_sums = la.attn_pattern_from.sum(dim=-1)  # (n_heads,)
        if not torch.allclose(row_sums,
                               torch.ones(N_HEADS), atol=1e-4):
            bad_heads = (row_sums - 1.0).abs() > 1e-4
            attn_sum_errors.append(
                f"{sa.sentence_id} L{la.layer}: "
                f"heads {bad_heads.nonzero().flatten().tolist()} "
                f"sums={row_sums[bad_heads].tolist()}"
            )

if attn_sum_errors:
    print(f"  ✗ {len(attn_sum_errors)} attention normalisation errors:")
    for e in attn_sum_errors[:10]:
        print(f"    {e}")
else:
    print(f"  ✓ All attention rows sum to 1.0 across all layers and sentences")


# ══════════════════════════════════════════════════════════════════════════════
# 9. DIAGNOSTIC VISUALISATIONS
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Activation Extraction — Diagnostic Plots",
             fontsize=14, fontweight="bold")

tier_colors = {
    "T1": "#2166ac", "T2": "#4393c3", "T3": "#92c5de",
    "T4": "#f4a582", "T5": "#d6604d", "T6": "#b2182b"
}

# ── Plot 1: Residual stream L2 norm by layer, coloured by tier
ax1 = axes[0, 0]
for sa in results:
    norms = [sa.get_layer(l).resid_post.norm().item()
             for l in range(N_LAYERS)]
    ax1.plot(range(N_LAYERS), norms,
             color=tier_colors[sa.tier], alpha=0.8,
             label=f"{sa.entity} ({sa.tier})")
ax1.set_xlabel("Layer")
ax1.set_ylabel("L2 norm")
ax1.set_title("Residual Stream L2 Norm by Layer")
ax1.legend(fontsize=7, loc="upper left")

# ── Plot 2: MLP output L2 norm by layer
ax2 = axes[0, 1]
for sa in results:
    norms = [sa.get_layer(l).mlp_out.norm().item()
             for l in range(N_LAYERS)]
    ax2.plot(range(N_LAYERS), norms,
             color=tier_colors[sa.tier], alpha=0.8,
             label=sa.entity)
ax2.set_xlabel("Layer")
ax2.set_ylabel("L2 norm")
ax2.set_title("MLP Output L2 Norm by Layer")

# ── Plot 3: Cosine similarity between child and bee across layers
ax3 = axes[0, 2]
child_sa = next(s for s in results if s.entity == "child")
bee_sa   = next(s for s in results if s.entity == "bee")
sims = []
for l in range(N_LAYERS):
    c = child_sa.get_layer(l).resid_post
    b = bee_sa.get_layer(l).resid_post
    sim = torch.nn.functional.cosine_similarity(
        c.unsqueeze(0), b.unsqueeze(0)
    ).item()
    sims.append(sim)
ax3.plot(range(N_LAYERS), sims, color="purple", linewidth=2)
ax3.axhline(0, color="black", linestyle="--", alpha=0.3)
ax3.set_xlabel("Layer")
ax3.set_ylabel("Cosine similarity")
ax3.set_title("Residual Stream Similarity:\n'child' vs 'bee' by Layer")
ax3.set_ylim(-1, 1)

# ── Plot 4: Attention TO entity at final layer, head 0
ax4 = axes[1, 0]
final_layer = N_LAYERS - 1
for sa in results[:4]:
    la   = sa.get_layer(final_layer)
    attn = la.attn_pattern_to[0].numpy()   # head 0
    ax4.plot(range(sa.n_tokens), attn,
             color=tier_colors[sa.tier], alpha=0.8,
             label=sa.entity)
    # Mark entity positions
    for p in sa.positions:
        ax4.axvline(p, color=tier_colors[sa.tier],
                    linestyle=":", alpha=0.4)
ax4.set_xlabel("Token position")
ax4.set_ylabel("Attention weight")
ax4.set_title(f"Attention TO Entity (Layer {final_layer}, Head 0)")
ax4.legend(fontsize=8)

# ── Plot 5: Std of residual stream by layer (uniformity check)
ax5 = axes[1, 1]
for sa in results:
    stds = [sa.get_layer(l).resid_post.std().item()
            for l in range(N_LAYERS)]
    ax5.plot(range(N_LAYERS), stds,
             color=tier_colors[sa.tier], alpha=0.8,
             label=sa.entity)
ax5.axhline(1e-6, color="red", linestyle="--",
            linewidth=1, label="Uniformity threshold")
ax5.set_xlabel("Layer")
ax5.set_ylabel("Std of activation values")
ax5.set_title("Residual Stream Std by Layer\n(red = uniformity threshold)")
ax5.legend(fontsize=7, loc="upper left")

# ── Plot 6: Health summary heatmap
ax6 = axes[1, 2]
pivot = (health_df[health_df["component"] == "resid_post"]
         .pivot_table(index="sentence_id", columns="layer",
                      values="std", aggfunc="mean"))
sns.heatmap(pivot, ax=ax6, cmap="viridis",
            cbar_kws={"label": "Std of resid_post"})
ax6.set_title("Residual Stream Std Heatmap\n(sentence × layer)")
ax6.set_xlabel("Layer")
ax6.set_ylabel("")
ax6.tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.savefig("activation_extraction_diagnostics.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("\nSaved: activation_extraction_diagnostics.png")


# ══════════════════════════════════════════════════════════════════════════════
# 10. HDF5 STORAGE
# ══════════════════════════════════════════════════════════════════════════════

HDF5_PATH = "activations.h5"

print(f"\nWriting activations to {HDF5_PATH}...")

with h5py.File(HDF5_PATH, "w") as f:

    # ── Top-level metadata
    f.attrs["model_name"]  = MODEL_NAME
    f.attrs["n_layers"]    = N_LAYERS
    f.attrs["n_heads"]     = N_HEADS
    f.attrs["d_model"]     = D_MODEL
    f.attrs["description"] = (
        "Sentience Salience Probe activations. "
        "Each group is one sentence×entity pair. "
        "Subgroups: resid_post, mlp_out, attn_from, attn_to — "
        "each dataset shape (n_layers, ...). "
    )

    for sa in results:

        grp = f.create_group(sa.sentence_id)

        # ── Group metadata
        grp.attrs["entity"]        = sa.entity
        grp.attrs["tier"]          = sa.tier
        grp.attrs["template_id"]   = sa.template_id
        grp.attrs["template_type"] = sa.template_type
        grp.attrs["sentence"]      = sa.sentence
        grp.attrs["positions"]     = sa.positions
        grp.attrs["n_tokens"]      = sa.n_tokens
        grp.attrs["n_layers"]      = N_LAYERS

        # ── Stack layer tensors into (n_layers, ...) arrays for efficiency
        resid_stack  = torch.stack(
            [la.resid_post for la in sa.layers]
        ).numpy()                                 # (n_layers, d_model)

        mlp_stack    = torch.stack(
            [la.mlp_out for la in sa.layers]
        ).numpy()                                 # (n_layers, d_model)

        attn_from_stack = torch.stack(
            [la.attn_pattern_from for la in sa.layers]
        ).numpy()                                 # (n_layers, n_heads, seq_len)

        attn_to_stack = torch.stack(
            [la.attn_pattern_to for la in sa.layers]
        ).numpy()                                 # (n_layers, n_heads, seq_len)

        # ── Write datasets with compression
        grp.create_dataset("resid_post",
                            data=resid_stack,
                            compression="gzip", compression_opts=4)
        grp.create_dataset("mlp_out",
                            data=mlp_stack,
                            compression="gzip", compression_opts=4)
        grp.create_dataset("attn_from",
                            data=attn_from_stack,
                            compression="gzip", compression_opts=4)
        grp.create_dataset("attn_to",
                            data=attn_to_stack,
                            compression="gzip", compression_opts=4)

print(f"  ✓ Written {len(results)} sentence groups")

# ── Verify HDF5 contents
print("\n  HDF5 verification:")
with h5py.File(HDF5_PATH, "r") as f:
    for sid in list(f.keys())[:3]:
        grp = f[sid]
        print(f"\n  [{sid}]  entity='{grp.attrs['entity']}'  "
              f"tier={grp.attrs['tier']}")
        for ds_name in ["resid_post", "mlp_out", "attn_from", "attn_to"]:
            ds = grp[ds_name]
            print(f"    {ds_name:<15} shape={ds.shape}  "
                  f"dtype={ds.dtype}  "
                  f"compressed={ds.compression}")


# ══════════════════════════════════════════════════════════════════════════════
# 11. FLAT DATAFRAME STORAGE
# ══════════════════════════════════════════════════════════════════════════════

print("\nBuilding flat DataFrame...")

flat_rows = []
for sa in results:
    for la in sa.layers:
        flat_rows.append({
            "sentence_id":    sa.sentence_id,
            "entity":         sa.entity,
            "tier":           sa.tier,
            "template_id":    sa.template_id,
            "template_type":  sa.template_type,
            "sentence":       sa.sentence,
            "positions":      str(sa.positions),
            "n_tokens":       sa.n_tokens,
            "layer":          la.layer,
            # Store activation vectors as comma-joined strings
            # Use HDF5 for full vectors; CSV is for metadata + probing
            "resid_post_mean":    la.resid_post.mean().item(),
            "resid_post_std":     la.resid_post.std().item(),
            "resid_post_norm":    la.resid_post.norm().item(),
            "mlp_out_mean":       la.mlp_out.mean().item(),
            "mlp_out_std":        la.mlp_out.std().item(),
            "mlp_out_norm":       la.mlp_out.norm().item(),
            "attn_to_entity_mean": la.attn_pattern_to.mean().item(),
            "attn_from_entity_mean": la.attn_pattern_from.mean().item(),
        })

flat_df = pd.DataFrame(flat_rows)
flat_df.to_csv("activations_summary.csv", index=False)

print(f"  Rows       : {len(flat_df):,}")
print(f"  Columns    : {list(flat_df.columns)}")
print(f"  Shape      : {flat_df.shape}")
print(f"  Saved      : activations_summary.csv")

print("\n  Sample rows:")
print(flat_df[["sentence_id", "entity", "tier", "layer",
               "resid_post_norm", "mlp_out_norm"]].head(8).to_string(index=False))


# ══════════════════════════════════════════════════════════════════════════════
# 12. READ-BACK HELPER
# ══════════════════════════════════════════════════════════════════════════════

def load_activations(sentence_id: str,
                      component:   str,
                      hdf5_path:   str = HDF5_PATH) -> np.ndarray:
    """
    Load a single activation array from HDF5.

    Parameters
    ----------
    sentence_id : str    e.g. "T01_child"
    component   : str    one of resid_post | mlp_out | attn_from | attn_to
    hdf5_path   : str

    Returns
    -------
    np.ndarray  shape depends on component:
        resid_post / mlp_out : (n_layers, d_model)
        attn_from / attn_to  : (n_layers, n_heads, seq_len)
    """
    valid = {"resid_post", "mlp_out", "attn_from", "attn_to"}
    if component not in valid:
        raise ValueError(f"component must be one of {valid}")

    with h5py.File(hdf5_path, "r") as f:
        if sentence_id not in f:
            raise KeyError(f"'{sentence_id}' not found in {hdf5_path}")
        return f[sentence_id][component][:]


def load_entity_metadata(sentence_id: str,
                          hdf5_path: str = HDF5_PATH) -> dict:
    """Load group-level metadata for a sentence."""
    with h5py.File(hdf5_path, "r") as f:
        grp = f[sentence_id]
        return dict(grp.attrs)


# Demo read-back
print("\n── Read-back demo ──")
resid = load_activations("T01_child", "resid_post")
meta  = load_entity_metadata("T01_child")
print(f"  T01_child  resid_post  shape={resid.shape}  "
      f"dtype={resid.dtype}  metadata={meta}")

resid_bee = load_activations("T01_bee", "resid_post")
print(f"  T01_bee    resid_post  shape={resid_bee.shape}")



# 13. FINAL STATUS

print("\n" + "═"*70)
print("  EXTRACTION COMPLETE")
print("═"*70)
print(f"  Sentences extracted : {len(results)}")
print(f"  Layers per sentence : {N_LAYERS}")
print(f"  Components per layer: 4  (resid_post, mlp_out, attn_from, attn_to)")
print(f"  NaN found           : {n_nan}")
print(f"  Inf found           : {n_inf}")
print(f"  Uniform tensors     : {n_uniform}")
print(f"\n  Output files:")
print(f"    activations.h5                  full vectors (HDF5, compressed)")
print(f"    activations_summary.csv         metadata + scalar stats per layer")
print(f"    activation_extraction_diagnostics.png")
print(f"\n  Next step: run full corpus extraction by iterating all 1,500")
print(f"  sentences through extract_sentence() and appending to HDF5.")

## Run full extraction on complete corpus; estimated time ~20–40 mins on Colab for Pythia-1.4B

Before starting: checks GPU memory, disk space, and estimates output size so you don't discover a problem three hours in.

During extraction: runs one forward pass per sentence, extracts all four activation types across all layers, validates for NaN/Inf, writes immediately to HDF5 (keeping RAM flat), and saves a checkpoint every 100 sentences so a crash loses at most 100 sentences of work.

After finishing: verifies HDF5 integrity, saves a summary CSV, and produces diagnostic plots — including a stability plot that catches silent model degradation across the run.

In [ ]:
import torch
import numpy as np
import h5py
import json
import pandas as pd
import time
import logging
import traceback
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
from transformer_lens import HookedTransformer
import matplotlib.pyplot as plt
import psutil
import os

# ══════════════════════════════════════════════════════════════════════════════
# 0. LOGGING
# ══════════════════════════════════════════════════════════════════════════════

logging.basicConfig(
    level    = logging.INFO,
    format   = "%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt  = "%H:%M:%S",
    handlers = [
        logging.FileHandler("extraction.log"),
        logging.StreamHandler(),
    ]
)
log = logging.getLogger(__name__)


# ══════════════════════════════════════════════════════════════════════════════
# 1. CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class ExtractionConfig:
    model_name:       str   = "pythia-1.4B"
    corpus_path:      str   = "entity_position_lookup.json"
    hdf5_path:        str   = "activations_1.4B.h5"
    summary_csv:      str   = "activations_260M_summary.csv"
    checkpoint_path:  str   = "extraction_checkpoint.json"
    log_path:         str   = "extraction.log"

    # Batching & memory
    sentences_per_checkpoint: int   = 100   # save progress every N sentences
    dtype:                    str   = "float16"   # float16 halves storage
    hdf5_compression:         int   = 4           # gzip level 0–9

    # Safety thresholds
    nan_abort_threshold:      int   = 10    # abort if > N NaN tensors seen
    max_retries:              int   = 3     # retries per sentence on error

    # What to extract
    extract_resid:            bool  = True
    extract_mlp:              bool  = True
    extract_attn:             bool  = True


CFG = ExtractionConfig()

log.info("="*60)
log.info("SENTIENCE SALIENCE PROBE — FULL CORPUS EXTRACTION")
log.info("="*60)
log.info(f"Model       : {CFG.model_name}")
log.info(f"Corpus      : {CFG.corpus_path}")
log.info(f"Output HDF5 : {CFG.hdf5_path}")
log.info(f"dtype       : {CFG.dtype}")


# ══════════════════════════════════════════════════════════════════════════════
# 2. ENVIRONMENT CHECK
# ══════════════════════════════════════════════════════════════════════════════

def check_environment(cfg: ExtractionConfig) -> dict:
    """
    Verify GPU memory, disk space, and dependencies before
    committing to a multi-hour extraction run.
    Returns a dict of environment facts; raises if any is fatal.
    """
    env = {}

    # GPU
    if torch.cuda.is_available():
        gpu_props = torch.cuda.get_device_properties(0)
        gpu_mem_gb = gpu_props.total_memory / 1e9
        env["device"]     = "cuda"
        env["gpu_name"]   = gpu_props.name
        env["gpu_mem_gb"] = round(gpu_mem_gb, 1)
        log.info(f"GPU : {gpu_props.name}  ({gpu_mem_gb:.1f} GB)")
        # Pythia-1.4b needs ~3 GB in float16; warn if tight
        if gpu_mem_gb < 5:
            log.warning("GPU memory < 5 GB — consider using float16 and "
                        "clearing cache aggressively")
    else:
        env["device"]   = "cpu"
        env["gpu_name"] = None
        log.warning("No GPU detected — extraction will be very slow on CPU")

    # RAM
    ram_gb = psutil.virtual_memory().total / 1e9
    env["ram_gb"] = round(ram_gb, 1)
    log.info(f"RAM : {ram_gb:.1f} GB")

    # Disk
    disk = psutil.disk_usage(".")
    disk_free_gb = disk.free / 1e9
    env["disk_free_gb"] = round(disk_free_gb, 1)
    log.info(f"Disk free : {disk_free_gb:.1f} GB")

    # Estimate output size
    # Pythia-1.4b: d_model=2048, n_layers=24, n_heads=16
    # Per sentence: resid (24×2048) + mlp (24×2048) = 24×4096 float16
    # Attn: 24×16×seq_len (variable); estimate seq_len~25
    # 1500 sentences × (24×4096×2 + 24×16×25×2) × 2 bytes / 1e9
    est_resid_mlp = 1500 * 24 * 4096 * 2 * 2 / 1e9
    est_attn      = 1500 * 24 * 16 * 25 * 2 * 2 / 1e9
    est_total_gb  = est_resid_mlp + est_attn
    env["est_output_gb"] = round(est_total_gb, 2)
    log.info(f"Estimated HDF5 output : ~{est_total_gb:.2f} GB")

    if disk_free_gb < est_total_gb * 1.5:
        raise RuntimeError(
            f"Insufficient disk space: {disk_free_gb:.1f} GB free, "
            f"need ~{est_total_gb*1.5:.1f} GB"
        )

    return env


env = check_environment(CFG)
device = env["device"]


# ══════════════════════════════════════════════════════════════════════════════
# 3. MODEL LOADING
# ══════════════════════════════════════════════════════════════════════════════

log.info(f"Loading {CFG.model_name}...")
t0 = time.time()

model = HookedTransformer.from_pretrained(
    CFG.model_name,
    # Load in float16 to halve VRAM and storage requirements
    dtype = torch.float16 if CFG.dtype == "float16" else torch.float32,
)
model.eval()
model = model.to(device)

N_LAYERS = model.cfg.n_layers
N_HEADS  = model.cfg.n_heads
D_MODEL  = model.cfg.d_model
D_MLP    = model.cfg.d_mlp

load_time = time.time() - t0
log.info(f"Model loaded in {load_time:.1f}s")
log.info(f"  Layers  : {N_LAYERS}")
log.info(f"  Heads   : {N_HEADS}")
log.info(f"  d_model : {D_MODEL}")
log.info(f"  d_mlp   : {D_MLP}")

if device == "cuda":
    allocated = torch.cuda.memory_allocated() / 1e9
    log.info(f"  GPU memory after load : {allocated:.2f} GB")


# ══════════════════════════════════════════════════════════════════════════════
# 4. CORPUS LOADING
# ══════════════════════════════════════════════════════════════════════════════

with open(CFG.corpus_path) as f:
    lookup = json.load(f)

# Flatten to a list of extraction jobs
jobs = []
for entity, data in lookup.items():
    tier = data["tier"]
    for tid, entry in data["entries"].items():
        jobs.append({
            "sentence_id":   f"{tid}_{entity.replace(' ', '_')}",
            "entity":        entity,
            "tier":          tier,
            "template_id":   tid,
            "template_type": entry["template_type"],
            "sentence":      entry["sentence"],
            "positions":     entry["positions"],
            "n_entity_tokens": entry["n_entity_tokens"],
            "is_multi_token":  entry["is_multi_token"],
            "aggregation":     entry["aggregation"],
        })

log.info(f"Corpus loaded: {len(jobs):,} extraction jobs")
log.info(f"  Entities  : {len(lookup)}")
log.info(f"  Templates : {len(set(j['template_id'] for j in jobs))}")

# Sort for reproducibility — tier then entity then template
jobs.sort(key=lambda j: (j["tier"], j["entity"], j["template_id"]))


# ══════════════════════════════════════════════════════════════════════════════
# 5. CHECKPOINT SYSTEM
# ══════════════════════════════════════════════════════════════════════════════

def load_checkpoint(path: str) -> set:
    """Load set of already-completed sentence_ids."""
    if Path(path).exists():
        with open(path) as f:
            data = json.load(f)
        completed = set(data["completed"])
        log.info(f"Checkpoint loaded: {len(completed)} sentences already done")
        return completed
    return set()


def save_checkpoint(path: str, completed: set, stats: dict):
    """Save progress checkpoint."""
    with open(path, "w") as f:
        json.dump({
            "completed":    list(completed),
            "n_completed":  len(completed),
            "stats":        stats,
        }, f, indent=2)


completed_ids = load_checkpoint(CFG.checkpoint_path)

# Filter to remaining jobs
remaining_jobs = [j for j in jobs
                  if j["sentence_id"] not in completed_ids]

log.info(f"Jobs remaining: {len(remaining_jobs):,} "
         f"(skipping {len(completed_ids)} completed)")


# ══════════════════════════════════════════════════════════════════════════════
# 6. HDF5 INITIALISATION
# ══════════════════════════════════════════════════════════════════════════════

hdf5_mode = "a" if Path(CFG.hdf5_path).exists() else "w"
log.info(f"HDF5 file: {CFG.hdf5_path}  (mode='{hdf5_mode}')")

with h5py.File(CFG.hdf5_path, hdf5_mode) as f:
    f.attrs["model_name"] = CFG.model_name
    f.attrs["n_layers"]   = N_LAYERS
    f.attrs["n_heads"]    = N_HEADS
    f.attrs["d_model"]    = D_MODEL
    f.attrs["dtype"]      = CFG.dtype
    f.attrs["n_jobs"]     = len(jobs)
    f.attrs["description"] = (
        "Sentience Salience Probe — full corpus activations. "
        f"Model: {CFG.model_name}. "
        "Groups keyed by sentence_id. "
        "Datasets: resid_post (n_layers, d_model), "
        "mlp_out (n_layers, d_model), "
        "attn_from (n_layers, n_heads, seq_len), "
        "attn_to (n_layers, n_heads, seq_len)."
    )


# ══════════════════════════════════════════════════════════════════════════════
# 7. EXTRACTION FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def extract_and_write(job:      dict,
                       model:    HookedTransformer,
                       hdf5_path: str,
                       cfg:      ExtractionConfig) -> dict:
    """
    Run a single forward pass, extract activations at entity positions,
    and write immediately to HDF5. Returns a summary stats dict.

    Writing immediately (rather than accumulating in RAM) keeps
    memory use flat across the full corpus run.
    """
    sentence  = job["sentence"]
    positions = job["positions"]
    sid       = job["sentence_id"]

    # ── Forward pass ──────────────────────────────────────────────────────
    tokens  = model.to_tokens(sentence)
    seq_len = tokens.shape[1]

    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)

    np_dtype = np.float16 if cfg.dtype == "float16" else np.float32

    # ── Stack across layers ────────────────────────────────────────────────

    resid_list     = []
    mlp_list       = []
    attn_from_list = []
    attn_to_list   = []

    for layer in range(N_LAYERS):

        if cfg.extract_resid:
            resid = cache[f"blocks.{layer}.hook_resid_post"]
            # (1, seq_len, d_model) → mean-pool entity positions → (d_model,)
            resid_entity = (resid[0, positions, :]
                            .mean(dim=0)
                            .cpu()
                            .to(torch.float32)
                            .numpy()
                            .astype(np_dtype))
            resid_list.append(resid_entity)

        if cfg.extract_mlp:
            mlp = cache[f"blocks.{layer}.hook_mlp_out"]
            mlp_entity = (mlp[0, positions, :]
                          .mean(dim=0)
                          .cpu()
                          .to(torch.float32)
                          .numpy()
                          .astype(np_dtype))
            mlp_list.append(mlp_entity)

        if cfg.extract_attn:
            attn = cache[f"blocks.{layer}.attn.hook_pattern"]
            # FROM entity → all positions: (n_heads, seq_len)
            attn_from = (attn[0, :, positions, :]
                         .mean(dim=1)
                         .cpu()
                         .to(torch.float32)
                         .numpy()
                         .astype(np_dtype))
            # TO entity ← all positions: (n_heads, seq_len)
            attn_to   = (attn[0, :, :, positions]
                         .mean(dim=2)
                         .cpu()
                         .to(torch.float32)
                         .numpy()
                         .astype(np_dtype))
            attn_from_list.append(attn_from)
            attn_to_list.append(attn_to)

    # ── Stack into arrays ──────────────────────────────────────────────────
    resid_arr     = np.stack(resid_list)      if resid_list     else None
    mlp_arr       = np.stack(mlp_list)        if mlp_list       else None
    attn_from_arr = np.stack(attn_from_list)  if attn_from_list else None
    attn_to_arr   = np.stack(attn_to_list)    if attn_to_list   else None

    # ── Validate before writing ────────────────────────────────────────────
    for name, arr in [("resid_post", resid_arr), ("mlp_out", mlp_arr),
                       ("attn_from", attn_from_arr), ("attn_to", attn_to_arr)]:
        if arr is None:
            continue
        if np.isnan(arr).any():
            raise ValueError(f"NaN in {name} for {sid}")
        if np.isinf(arr).any():
            raise ValueError(f"Inf in {name} for {sid}")

    # ── Write to HDF5 ──────────────────────────────────────────────────────
    with h5py.File(hdf5_path, "a") as f:
        if sid in f:
            del f[sid]   # overwrite if retrying

        grp = f.create_group(sid)

        # Metadata
        grp.attrs["entity"]          = job["entity"]
        grp.attrs["tier"]            = job["tier"]
        grp.attrs["template_id"]     = job["template_id"]
        grp.attrs["template_type"]   = job["template_type"]
        grp.attrs["sentence"]        = sentence
        grp.attrs["positions"]       = positions
        grp.attrs["n_tokens"]        = seq_len
        grp.attrs["n_entity_tokens"] = job["n_entity_tokens"]
        grp.attrs["is_multi_token"]  = job["is_multi_token"]
        grp.attrs["aggregation"]     = job["aggregation"]

        opts = dict(compression="gzip",
                    compression_opts=cfg.hdf5_compression)

        if resid_arr is not None:
            grp.create_dataset("resid_post", data=resid_arr, **opts)
        if mlp_arr is not None:
            grp.create_dataset("mlp_out",    data=mlp_arr,   **opts)
        if attn_from_arr is not None:
            grp.create_dataset("attn_from",  data=attn_from_arr, **opts)
        if attn_to_arr is not None:
            grp.create_dataset("attn_to",    data=attn_to_arr,   **opts)

    # ── Clear cache ────────────────────────────────────────────────────────
    del cache
    if device == "cuda":
        torch.cuda.empty_cache()

    # ── Return summary stats ───────────────────────────────────────────────
    return {
        "sentence_id":       sid,
        "entity":            job["entity"],
        "tier":              job["tier"],
        "template_id":       job["template_id"],
        "template_type":     job["template_type"],
        "n_tokens":          seq_len,
        "n_entity_tokens":   job["n_entity_tokens"],
        "resid_post_norm":   float(np.linalg.norm(resid_arr[-1]))
                             if resid_arr is not None else None,
        "mlp_out_norm":      float(np.linalg.norm(mlp_arr[-1]))
                             if mlp_arr is not None else None,
        "attn_to_mean":      float(attn_to_arr.mean())
                             if attn_to_arr is not None else None,
    }


# ══════════════════════════════════════════════════════════════════════════════
# 8. MAIN EXTRACTION LOOP
# ══════════════════════════════════════════════════════════════════════════════

summary_rows = []
error_log    = []

n_total     = len(remaining_jobs)
n_done      = 0
n_errors    = 0
n_nan_seen  = 0
t_start     = time.time()

log.info(f"\nStarting extraction: {n_total} sentences remaining")
log.info("-" * 60)

for job_idx, job in enumerate(remaining_jobs):

    sid = job["sentence_id"]

    # ── Retry loop ─────────────────────────────────────────────────────────
    success    = False
    last_error = None

    for attempt in range(CFG.max_retries):
        try:
            stats = extract_and_write(job, model, CFG.hdf5_path, CFG)
            summary_rows.append(stats)
            completed_ids.add(sid)
            success = True
            break

        except ValueError as e:
            # NaN/Inf — likely a real problem, log and count
            last_error  = str(e)
            n_nan_seen += 1
            log.warning(f"  NaN/Inf in {sid} (attempt {attempt+1}): {e}")
            if n_nan_seen > CFG.nan_abort_threshold:
                log.error(f"NaN threshold exceeded ({n_nan_seen}). Aborting.")
                raise RuntimeError("Too many NaN values — check model state")

        except torch.cuda.OutOfMemoryError:
            last_error = "CUDA OOM"
            log.warning(f"  CUDA OOM on {sid} (attempt {attempt+1}) — "
                        f"clearing cache and retrying")
            torch.cuda.empty_cache()
            time.sleep(2)

        except Exception as e:
            last_error = str(e)
            log.warning(f"  Error on {sid} attempt {attempt+1}: {e}")
            time.sleep(1)

    if not success:
        n_errors += 1
        error_log.append({
            "sentence_id": sid,
            "entity":      job["entity"],
            "tier":        job["tier"],
            "template_id": job["template_id"],
            "error":       last_error,
        })
        log.error(f"  FAILED after {CFG.max_retries} attempts: {sid} — {last_error}")

    n_done += 1

    # ── Progress logging ───────────────────────────────────────────────────
    if n_done % 50 == 0 or n_done == n_total:
        elapsed   = time.time() - t_start
        rate      = n_done / elapsed
        remaining = (n_total - n_done) / rate if rate > 0 else 0
        pct       = 100 * n_done / n_total

        gpu_info = ""
        if device == "cuda":
            alloc = torch.cuda.memory_allocated() / 1e9
            resv  = torch.cuda.memory_reserved() / 1e9
            gpu_info = f"  GPU {alloc:.1f}/{resv:.1f}GB"

        log.info(
            f"  [{n_done:>4}/{n_total}]  {pct:5.1f}%  "
            f"{rate:.1f} sent/s  "
            f"ETA {remaining/60:.1f}min  "
            f"errors={n_errors}{gpu_info}"
        )

    # ── Checkpoint ────────────────────────────────────────────────────────
    if n_done % CFG.sentences_per_checkpoint == 0:
        save_checkpoint(CFG.checkpoint_path, completed_ids, {
            "n_done":   n_done,
            "n_errors": n_errors,
            "elapsed_s": round(time.time() - t_start, 1),
        })
        log.info(f"  Checkpoint saved ({n_done} done)")


# ══════════════════════════════════════════════════════════════════════════════
# 9. FINAL CHECKPOINT & ERROR REPORT
# ══════════════════════════════════════════════════════════════════════════════

save_checkpoint(CFG.checkpoint_path, completed_ids, {
    "n_done":    n_done,
    "n_errors":  n_errors,
    "elapsed_s": round(time.time() - t_start, 1),
    "complete":  True,
})

total_time = time.time() - t_start
log.info("\n" + "="*60)
log.info("EXTRACTION COMPLETE")
log.info("="*60)
log.info(f"  Total sentences  : {n_total}")
log.info(f"  Succeeded        : {n_done - n_errors}")
log.info(f"  Failed           : {n_errors}")
log.info(f"  Total time       : {total_time/60:.1f} min")
log.info(f"  Mean rate        : {n_done/total_time:.2f} sentences/sec")

if error_log:
    error_df = pd.DataFrame(error_log)
    error_df.to_csv("extraction_errors.csv", index=False)
    log.info(f"\n  Error log saved: extraction_errors.csv")
    log.info(f"  Error breakdown by tier:")
    log.info(error_df.groupby("tier").size().to_string())


# ══════════════════════════════════════════════════════════════════════════════
# 10. SUMMARY CSV
# ══════════════════════════════════════════════════════════════════════════════

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(CFG.summary_csv, index=False)
log.info(f"\n  Summary CSV saved: {CFG.summary_csv}")
log.info(f"  Rows: {len(summary_df):,}")


# ══════════════════════════════════════════════════════════════════════════════
# 11. HDF5 INTEGRITY VERIFICATION
# ══════════════════════════════════════════════════════════════════════════════

log.info("\n── HDF5 integrity verification ──")

with h5py.File(CFG.hdf5_path, "r") as f:
    n_groups      = len(f.keys())
    file_size_gb  = Path(CFG.hdf5_path).stat().st_size / 1e9

    log.info(f"  Groups in HDF5 : {n_groups}")
    log.info(f"  File size       : {file_size_gb:.2f} GB")
    log.info(f"  Expected groups : {len(jobs)}")

    if n_groups != len(jobs) - n_errors:
        log.warning(f"  ⚠ Group count mismatch: "
                    f"expected {len(jobs) - n_errors}, got {n_groups}")
    else:
        log.info(f"  ✓ Group count correct")

    # Shape check on sample groups
    shape_errors = []
    sample_ids   = list(f.keys())[:10]

    for sid in sample_ids:
        grp = f[sid]
        n_tok = grp.attrs["n_tokens"]

        expected_shapes = {
            "resid_post": (N_LAYERS, D_MODEL),
            "mlp_out":    (N_LAYERS, D_MODEL),
            "attn_from":  (N_LAYERS, N_HEADS, n_tok),
            "attn_to":    (N_LAYERS, N_HEADS, n_tok),
        }
        for ds_name, expected in expected_shapes.items():
            if ds_name in grp:
                actual = grp[ds_name].shape
                if actual != expected:
                    shape_errors.append(
                        f"{sid}/{ds_name}: "
                        f"expected {expected}, got {actual}"
                    )

    if shape_errors:
        log.error(f"  ✗ Shape errors:")
        for e in shape_errors:
            log.error(f"    {e}")
    else:
        log.info(f"  ✓ All sampled shapes correct")


# ══════════════════════════════════════════════════════════════════════════════
# 12. POST-EXTRACTION DIAGNOSTICS
# ══════════════════════════════════════════════════════════════════════════════

log.info("\n── Post-extraction diagnostics ──")

# Norm trajectories by tier
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"Extraction Diagnostics — {CFG.model_name}",
             fontsize=13, fontweight="bold")

tier_colors = {
    "T1": "#2166ac", "T2": "#4393c3", "T3": "#92c5de",
    "T4": "#f4a582", "T5": "#d6604d", "T6": "#b2182b"
}

# ── Plot 1: Residual norm at final layer by tier
ax1 = axes[0]
for tier in ["T1","T2","T3","T4","T5","T6"]:
    subset = summary_df[summary_df["tier"] == tier]["resid_post_norm"].dropna()
    ax1.scatter([tier]*len(subset), subset,
                color=tier_colors[tier], alpha=0.4, s=15)
    ax1.plot([tier], [subset.mean()],
             color=tier_colors[tier], marker="D",
             markersize=8, zorder=5)
ax1.set_xlabel("Tier")
ax1.set_ylabel("L2 norm (final layer resid_post)")
ax1.set_title("Residual Norm by Tier\n(diamond = mean)")

# ── Plot 2: Mean attention to entity by tier
ax2 = axes[1]
for tier in ["T1","T2","T3","T4","T5","T6"]:
    subset = summary_df[summary_df["tier"] == tier]["attn_to_mean"].dropna()
    ax2.scatter([tier]*len(subset), subset,
                color=tier_colors[tier], alpha=0.4, s=15)
    ax2.plot([tier], [subset.mean()],
             color=tier_colors[tier], marker="D",
             markersize=8, zorder=5)
ax2.set_xlabel("Tier")
ax2.set_ylabel("Mean attention weight to entity")
ax2.set_title("Attention to Entity by Tier")

# ── Plot 3: Extraction rate over time
ax3 = axes[2]
if len(summary_rows) > 1:
    ax3.plot(range(len(summary_rows)),
             summary_df["resid_post_norm"].rolling(20).mean(),
             color="steelblue", linewidth=1.5)
    ax3.set_xlabel("Sentence index")
    ax3.set_ylabel("Residual norm (20-sentence rolling mean)")
    ax3.set_title("Residual Norm Stability Over Extraction\n"
                  "(should be stable — drift = problem)")

plt.tight_layout()
plt.savefig("extraction_diagnostics.png", dpi=150, bbox_inches="tight")
log.info("  Saved: extraction_diagnostics.png")


# ══════════════════════════════════════════════════════════════════════════════
# 13. READ-BACK UTILITIES FOR WEEK 2
# ══════════════════════════════════════════════════════════════════════════════

def load_resid_post_all(hdf5_path: str = CFG.hdf5_path) -> tuple:
    """
    Load all residual stream vectors into a single matrix.
    Returns (matrix, metadata_df) ready for PCA and probing.

    matrix shape: (n_sentences × n_layers, d_model)
    metadata_df : one row per (sentence, layer) with entity/tier/layer cols
    """
    rows_data = []
    rows_meta = []

    with h5py.File(hdf5_path, "r") as f:
        for sid in f.keys():
            grp  = f[sid]
            resid = grp["resid_post"][:]          # (n_layers, d_model)
            entity = grp.attrs["entity"]
            tier   = grp.attrs["tier"]
            tid    = grp.attrs["template_id"]
            ttype  = grp.attrs["template_type"]

            for layer_idx in range(resid.shape[0]):
                rows_data.append(resid[layer_idx])
                rows_meta.append({
                    "sentence_id":   sid,
                    "entity":        entity,
                    "tier":          tier,
                    "template_id":   tid,
                    "template_type": ttype,
                    "layer":         layer_idx,
                })

    matrix   = np.stack(rows_data).astype(np.float32)
    meta_df  = pd.DataFrame(rows_meta)

    return matrix, meta_df


def load_entity_mean_per_layer(entity:    str,
                                hdf5_path: str = CFG.hdf5_path) -> np.ndarray:
    """
    Load the mean residual stream vector for one entity,
    averaged across all templates, per layer.
    Returns shape (n_layers, d_model).
    """
    vectors = []   # list of (n_layers, d_model) arrays

    with h5py.File(hdf5_path, "r") as f:
        for sid in f.keys():
            grp = f[sid]
            if grp.attrs["entity"] == entity:
                vectors.append(grp["resid_post"][:])

    if not vectors:
        raise KeyError(f"Entity '{entity}' not found in {hdf5_path}")

    stacked = np.stack(vectors)          # (n_templates, n_layers, d_model)
    return stacked.mean(axis=0)          # (n_layers, d_model)


log.info("\n── Read-back utilities ready ──")
log.info("  load_resid_post_all()         → full matrix for PCA/probing")
log.info("  load_entity_mean_per_layer()  → per-entity mean vector by layer")

log.info("\n── Output files ──")
log.info(f"  {CFG.hdf5_path:<35} full activations (HDF5)")
log.info(f"  {CFG.summary_csv:<35} scalar summary per sentence")
log.info(f"  extraction.log               full extraction log")
log.info(f"  extraction_checkpoint.json   resume state")
log.info(f"  extraction_diagnostics.png   post-run diagnostic plots")
if error_log:
    log.info(f"  extraction_errors.csv        failed sentences")



## For each layer (or a sample: layers 1, 4, 8, 12, 16, final), run PCA on residual stream vectors at entity position

Plot 2D PCA coloured by tier — use a consistent colour scheme across all layer plots (humans = dark blue through to plants = light grey)

Also run UMAP on the full-layer stack — sometimes finds structure PCA misses

In [ ]:
import numpy as np
import pandas as pd
import h5py
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import umap
import warnings
from pathlib import Path
from scipy.spatial import ConvexHull
from scipy import stats

warnings.filterwarnings('ignore')


# ══════════════════════════════════════════════════════════════════════════════
# 1. CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

HDF5_PATH   = "activations_1.4b.h5"
LOOKUP_PATH = "entity_position_lookup.json"
N_LAYERS    = 24    # Pythia-1.4b
D_MODEL     = 2048

# Layers to visualise individually
SAMPLE_LAYERS = [0, 1, 4, 8, 12, 16, 20, N_LAYERS - 1]

# ── Tier colour scheme ─────────────────────────────────────────────────────
# Dark blue (humans) → medium blue → light blue → orange → red → light grey (controls)
TIER_CONFIG = {
    "T1": {
        "label":  "T1 — Humans",
        "color":  "#08306b",    # very dark blue
        "marker": "o",
        "zorder": 6,
        "size":   55,
    },
    "T2": {
        "label":  "T2 — Mammals",
        "color":  "#2171b5",    # medium blue
        "marker": "o",
        "zorder": 5,
        "size":   45,
    },
    "T3": {
        "label":  "T3 — Vertebrates",
        "color":  "#6baed6",    # light blue
        "marker": "o",
        "zorder": 4,
        "size":   40,
    },
    "T4": {
        "label":  "T4 — Invertebrates (welfare+)",
        "color":  "#fd8d3c",    # orange
        "marker": "s",
        "zorder": 3,
        "size":   40,
    },
    "T5": {
        "label":  "T5 — Invertebrates (minimal)",
        "color":  "#d7301f",    # red
        "marker": "s",
        "zorder": 2,
        "size":   40,
    },
    "T6": {
        "label":  "T6 — Non-sentient controls",
        "color":  "#bdbdbd",    # light grey
        "marker": "^",
        "zorder": 1,
        "size":   40,
    },
}

TIER_ORDER = ["T1", "T2", "T3", "T4", "T5", "T6"]


# ══════════════════════════════════════════════════════════════════════════════
# 2. DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_all_resid(hdf5_path: str) -> tuple[np.ndarray, pd.DataFrame]:
    """
    Load residual stream vectors for every sentence × layer.

    Returns
    -------
    matrix   : np.ndarray  shape (n_sentences, n_layers, d_model)
    meta_df  : pd.DataFrame  one row per sentence with entity/tier/template cols
    """
    matrices = []
    meta_rows = []

    with h5py.File(hdf5_path, "r") as f:
        sentence_ids = sorted(f.keys())
        print(f"  Loading {len(sentence_ids)} sentence groups...")

        for sid in sentence_ids:
            grp  = f[sid]
            resid = grp["resid_post"][:].astype(np.float32)   # (n_layers, d_model)
            matrices.append(resid)
            meta_rows.append({
                "sentence_id":   sid,
                "entity":        grp.attrs["entity"],
                "tier":          grp.attrs["tier"],
                "template_id":   grp.attrs["template_id"],
                "template_type": grp.attrs["template_type"],
                "n_tokens":      grp.attrs["n_tokens"],
            })

    matrix  = np.stack(matrices)           # (n_sentences, n_layers, d_model)
    meta_df = pd.DataFrame(meta_rows)

    print(f"  Matrix shape: {matrix.shape}")
    print(f"  Tiers: {meta_df['tier'].value_counts().to_dict()}")

    return matrix, meta_df


print("Loading activations...")
matrix, meta_df = load_all_resid(HDF5_PATH)

N_SENTENCES = matrix.shape[0]
N_LAYERS_ACTUAL = matrix.shape[1]

# Entity-level means — average across all templates for each entity
# Used for the cleaner "one point per entity" plots
with open(LOOKUP_PATH) as f:
    lookup = json.load(f)

entity_list = sorted(lookup.keys())

def compute_entity_means(matrix: np.ndarray,
                          meta_df: pd.DataFrame) -> tuple:
    """
    For each entity, average residual vectors across all templates.
    Returns (entity_matrix, entity_meta) where entity_matrix has
    shape (n_entities, n_layers, d_model).
    """
    entity_matrices = []
    entity_meta     = []

    for entity in entity_list:
        mask = meta_df["entity"] == entity
        if mask.sum() == 0:
            continue
        entity_vecs = matrix[mask]          # (n_templates, n_layers, d_model)
        entity_mean = entity_vecs.mean(axis=0)   # (n_layers, d_model)
        entity_matrices.append(entity_mean)
        entity_meta.append({
            "entity": entity,
            "tier":   lookup[entity]["tier"],
        })

    return (np.stack(entity_matrices),
            pd.DataFrame(entity_meta))


print("\nComputing entity-level means...")
entity_matrix, entity_meta = compute_entity_means(matrix, meta_df)
print(f"  Entity matrix shape: {entity_matrix.shape}")
# (n_entities, n_layers, d_model)


# ══════════════════════════════════════════════════════════════════════════════
# 3. PLOTTING UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def scatter_by_tier(ax,
                     coords:    np.ndarray,
                     tiers:     pd.Series,
                     entities:  pd.Series = None,
                     alpha:     float = 0.75,
                     label_entities: bool = False,
                     hull:      bool = True):
    """
    Scatter plot with consistent tier colour scheme.
    Optionally draws convex hulls per tier and entity labels.

    Parameters
    ----------
    ax        : matplotlib Axes
    coords    : (n, 2) array of 2D coordinates
    tiers     : Series of tier strings aligned with coords
    entities  : Series of entity name strings (for labels)
    alpha     : point transparency
    label_entities : whether to annotate each point with entity name
    hull      : whether to draw convex hulls per tier
    """
    # Draw hulls first (behind points)
    if hull:
        for tier in TIER_ORDER:
            mask = tiers == tier
            pts  = coords[mask]
            if pts.shape[0] >= 3:
                try:
                    ch = ConvexHull(pts)
                    hull_pts = np.append(ch.vertices, ch.vertices[0])
                    ax.fill(pts[hull_pts, 0], pts[hull_pts, 1],
                            alpha=0.06,
                            color=TIER_CONFIG[tier]["color"],
                            zorder=0)
                    ax.plot(pts[hull_pts, 0], pts[hull_pts, 1],
                            alpha=0.2, linewidth=0.8,
                            color=TIER_CONFIG[tier]["color"],
                            zorder=0)
                except Exception:
                    pass   # hull fails with collinear points; skip silently

    # Draw points per tier (so zorder works correctly)
    for tier in TIER_ORDER:
        mask = (tiers == tier).values
        cfg  = TIER_CONFIG[tier]
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            c       = cfg["color"],
            marker  = cfg["marker"],
            s       = cfg["size"],
            alpha   = alpha,
            zorder  = cfg["zorder"],
            edgecolors = "white",
            linewidths = 0.4,
            label   = cfg["label"],
        )

    # Entity labels
    if label_entities and entities is not None:
        for i, (x, y) in enumerate(coords):
            tier = tiers.iloc[i]
            ax.annotate(
                entities.iloc[i],
                (x, y),
                fontsize  = 5.5,
                color     = TIER_CONFIG[tier]["color"],
                alpha     = 0.85,
                xytext    = (3, 3),
                textcoords= "offset points",
            )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines[["top","right","bottom","left"]].set_visible(False)


def add_legend(ax, fontsize=8):
    handles = [
        mpatches.Patch(
            color=TIER_CONFIG[t]["color"],
            label=TIER_CONFIG[t]["label"]
        )
        for t in TIER_ORDER
    ]
    ax.legend(handles=handles, fontsize=fontsize,
              loc="lower right", framealpha=0.9,
              edgecolor="#cccccc")


def variance_explained_annotation(ax, pca_obj):
    ve = pca_obj.explained_variance_ratio_
    ax.set_xlabel(f"PC1 ({ve[0]*100:.1f}% var)", fontsize=8)
    ax.set_ylabel(f"PC2 ({ve[1]*100:.1f}% var)", fontsize=8)


# ══════════════════════════════════════════════════════════════════════════════
# 4. PCA — PER-LAYER (FULL SENTENCE SET)
# ══════════════════════════════════════════════════════════════════════════════

print("\nRunning PCA per sampled layer (full sentence set)...")

# Store PCA results for later use
pca_results = {}   # layer → {"coords": (n,2), "pca": PCA obj}

for layer in SAMPLE_LAYERS:
    X = matrix[:, layer, :]                    # (n_sentences, d_model)
    X_scaled = StandardScaler().fit_transform(X)
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X_scaled)
    pca_results[layer] = {"coords": coords, "pca": pca}
    ve = pca.explained_variance_ratio_
    print(f"  Layer {layer:2d}: PC1={ve[0]*100:.1f}%  PC2={ve[1]*100:.1f}%"
          f"  total={sum(ve)*100:.1f}%")


# ── Figure 1: PCA grid, all sampled layers ─────────────────────────────────

n_sample = len(SAMPLE_LAYERS)
ncols    = 4
nrows    = int(np.ceil(n_sample / ncols))

fig1, axes = plt.subplots(nrows, ncols,
                           figsize=(ncols*4.5, nrows*4.2))
axes_flat  = axes.flatten()

fig1.suptitle(
    f"PCA of Residual Stream at Entity Token — {N_LAYERS_ACTUAL}-layer Pythia-1.4b\n"
    f"Each point = one sentence (n={N_SENTENCES}), coloured by sentience tier",
    fontsize=12, fontweight="bold", y=1.01
)

for plot_idx, layer in enumerate(SAMPLE_LAYERS):
    ax     = axes_flat[plot_idx]
    result = pca_results[layer]

    is_last = (layer == N_LAYERS - 1)
    layer_label = f"Layer {layer}" + (" (final)" if is_last else "")

    scatter_by_tier(
        ax       = ax,
        coords   = result["coords"],
        tiers    = meta_df["tier"],
        hull     = True,
        alpha    = 0.55,
    )
    variance_explained_annotation(ax, result["pca"])
    ax.set_title(layer_label, fontsize=10, fontweight="bold", pad=6)

    # Add legend only on final subplot
    if plot_idx == len(SAMPLE_LAYERS) - 1:
        add_legend(ax, fontsize=7)

# Hide unused subplots
for idx in range(len(SAMPLE_LAYERS), len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.tight_layout()
plt.savefig("pca_per_layer_sentences.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: pca_per_layer_sentences.png")


# ══════════════════════════════════════════════════════════════════════════════
# 5. PCA — PER-LAYER (ENTITY MEANS)
# ══════════════════════════════════════════════════════════════════════════════

print("\nRunning PCA per sampled layer (entity means — one point per entity)...")

pca_entity_results = {}

for layer in SAMPLE_LAYERS:
    X        = entity_matrix[:, layer, :]      # (n_entities, d_model)
    X_scaled = StandardScaler().fit_transform(X)
    pca      = PCA(n_components=2, random_state=42)
    coords   = pca.fit_transform(X_scaled)
    pca_entity_results[layer] = {"coords": coords, "pca": pca}


fig2, axes = plt.subplots(nrows, ncols,
                           figsize=(ncols*4.5, nrows*4.2))
axes_flat  = axes.flatten()

fig2.suptitle(
    f"PCA of Residual Stream — Entity Means (one point per entity, n={len(entity_list)})\n"
    f"Averaged across all {len(meta_df)//len(entity_list)} templates per entity",
    fontsize=12, fontweight="bold", y=1.01
)

for plot_idx, layer in enumerate(SAMPLE_LAYERS):
    ax     = axes_flat[plot_idx]
    result = pca_entity_results[layer]

    is_last    = (layer == N_LAYERS - 1)
    layer_label = f"Layer {layer}" + (" (final)" if is_last else "")

    scatter_by_tier(
        ax             = ax,
        coords         = result["coords"],
        tiers          = entity_meta["tier"],
        entities       = entity_meta["entity"],
        hull           = True,
        alpha          = 0.85,
        label_entities = True,   # entity names visible — fewer points
    )
    variance_explained_annotation(ax, result["pca"])
    ax.set_title(layer_label, fontsize=10, fontweight="bold", pad=6)

    if plot_idx == len(SAMPLE_LAYERS) - 1:
        add_legend(ax, fontsize=7)

for idx in range(len(SAMPLE_LAYERS), len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.tight_layout()
plt.savefig("pca_per_layer_entity_means.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: pca_per_layer_entity_means.png")


# ══════════════════════════════════════════════════════════════════════════════
# 6. PCA DEEP DIVE — FINAL LAYER
# ══════════════════════════════════════════════════════════════════════════════

print("\nBuilding final-layer PCA deep dive...")

final_layer   = N_LAYERS - 1
final_coords  = pca_entity_results[final_layer]["coords"]
final_pca     = pca_entity_results[final_layer]["pca"]

fig3 = plt.figure(figsize=(16, 12))
gs   = gridspec.GridSpec(2, 2, figure=fig3,
                          hspace=0.35, wspace=0.3)

fig3.suptitle(
    f"Final Layer PCA Deep Dive (Layer {final_layer}) — Entity Means",
    fontsize=13, fontweight="bold"
)

# ── Panel A: Main scatter with entity labels
ax_main = fig3.add_subplot(gs[0, :])
scatter_by_tier(ax_main, final_coords, entity_meta["tier"],
                entity_meta["entity"],
                hull=True, alpha=0.9, label_entities=True)
variance_explained_annotation(ax_main, final_pca)
ax_main.set_title("Panel A — Final Layer PCA: Entity Means",
                   fontsize=11, fontweight="bold")
add_legend(ax_main)

# ── Panel B: PC1 score ranked by entity, coloured by tier
ax_b = fig3.add_subplot(gs[1, 0])
sort_idx  = np.argsort(final_coords[:, 0])
sorted_pc1 = final_coords[sort_idx, 0]
sorted_tiers  = entity_meta["tier"].iloc[sort_idx]
sorted_entities = entity_meta["entity"].iloc[sort_idx]
bar_colors = [TIER_CONFIG[t]["color"] for t in sorted_tiers]

ax_b.barh(range(len(sorted_pc1)), sorted_pc1,
          color=bar_colors, edgecolor="white", linewidth=0.3)
ax_b.set_yticks(range(len(sorted_pc1)))
ax_b.set_yticklabels(sorted_entities, fontsize=6)
ax_b.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax_b.set_xlabel("PC1 score")
ax_b.set_title("Panel B — PC1 Ranking\n(do tiers separate along PC1?)",
               fontsize=10, fontweight="bold")

# Colour y-tick labels by tier
for ticklabel, tier in zip(ax_b.get_yticklabels(), sorted_tiers):
    ticklabel.set_color(TIER_CONFIG[tier]["color"])

# ── Panel C: Tier centroids across layers on PC1
ax_c = fig3.add_subplot(gs[1, 1])

# For each sampled layer, compute mean PC1 score per tier
# using the sentence-level PCA (not entity means)
layer_tier_pc1 = {tier: [] for tier in TIER_ORDER}

for layer in SAMPLE_LAYERS:
    coords_layer = pca_results[layer]["coords"]
    for tier in TIER_ORDER:
        mask = (meta_df["tier"] == tier).values
        mean_pc1 = coords_layer[mask, 0].mean()
        layer_tier_pc1[tier].append(mean_pc1)

for tier in TIER_ORDER:
    ax_c.plot(SAMPLE_LAYERS,
              layer_tier_pc1[tier],
              color     = TIER_CONFIG[tier]["color"],
              linewidth = 2,
              marker    = "o",
              markersize= 5,
              label     = TIER_CONFIG[tier]["label"],
              zorder    = TIER_CONFIG[tier]["zorder"])

ax_c.set_xlabel("Layer")
ax_c.set_ylabel("Mean PC1 score")
ax_c.set_title("Panel C — Tier Centroid PC1 Across Layers\n"
               "(separation emerging = gradient forming)",
               fontsize=10, fontweight="bold")
ax_c.legend(fontsize=7, loc="best")
ax_c.axhline(0, color="black", linewidth=0.5, linestyle="--", alpha=0.4)
ax_c.set_xticks(SAMPLE_LAYERS)

plt.savefig("pca_final_layer_deepdive.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: pca_final_layer_deepdive.png")


# ══════════════════════════════════════════════════════════════════════════════
# 7. UMAP — FULL LAYER STACK
# ══════════════════════════════════════════════════════════════════════════════

print("\nRunning UMAP on full layer stack...")
print("  Strategy: concatenate all layers into one long vector per entity")
print("  This asks UMAP to find structure across the entire depth of the model")

# ── Strategy A: Concatenate all layers (captures cross-layer structure)
X_concat = entity_matrix.reshape(
    len(entity_list), -1
)   # (n_entities, n_layers × d_model)

print(f"  Concatenated shape: {X_concat.shape}")

# Standardise before UMAP
X_concat_scaled = StandardScaler().fit_transform(X_concat)

# ── Strategy B: Final layer only (for comparison)
X_final = entity_matrix[:, -1, :]
X_final_scaled = StandardScaler().fit_transform(X_final)

# ── UMAP hyperparameter grid — run multiple settings
# n_neighbors controls local vs global structure
# min_dist controls cluster compactness
umap_configs = [
    {"n_neighbors": 5,  "min_dist": 0.05, "label": "local\n(n_neighbors=5)"},
    {"n_neighbors": 15, "min_dist": 0.1,  "label": "balanced\n(n_neighbors=15)"},
    {"n_neighbors": 30, "min_dist": 0.3,  "label": "global\n(n_neighbors=30)"},
]

print(f"  Running {len(umap_configs)} UMAP configurations "
      f"× 2 strategies = {len(umap_configs)*2} projections...")

umap_results = {}

for cfg_u in umap_configs:
    for strategy_name, X_input in [
        ("all_layers", X_concat_scaled),
        ("final_layer", X_final_scaled),
    ]:
        key = f"{strategy_name}_{cfg_u['n_neighbors']}"
        print(f"    {key}...", end=" ")

        reducer = umap.UMAP(
            n_components = 2,
            n_neighbors  = cfg_u["n_neighbors"],
            min_dist     = cfg_u["min_dist"],
            metric       = "cosine",
            random_state = 42,
            verbose      = False,
        )
        coords = reducer.fit_transform(X_input)
        umap_results[key] = {
            "coords":   coords,
            "config":   cfg_u,
            "strategy": strategy_name,
        }
        print("done")


# ── Figure 4: UMAP grid ────────────────────────────────────────────────────

fig4, axes = plt.subplots(2, 3, figsize=(18, 11))
fig4.suptitle(
    "UMAP Projections — Entity Means\n"
    "Top row: all layers concatenated  |  Bottom row: final layer only\n"
    "Columns: local → global neighbourhood structure",
    fontsize=12, fontweight="bold"
)

for row_idx, strategy_name in enumerate(["all_layers", "final_layer"]):
    for col_idx, cfg_u in enumerate(umap_configs):
        ax  = axes[row_idx, col_idx]
        key = f"{strategy_name}_{cfg_u['n_neighbors']}"
        res = umap_results[key]

        scatter_by_tier(
            ax       = ax,
            coords   = res["coords"],
            tiers    = entity_meta["tier"],
            entities = entity_meta["entity"],
            hull     = True,
            alpha    = 0.88,
            label_entities = True,
        )

        strategy_label = ("All layers\nconcatenated"
                          if strategy_name == "all_layers"
                          else "Final layer\nonly")
        ax.set_title(
            f"{strategy_label}\n{cfg_u['label']}",
            fontsize=9, fontweight="bold"
        )

        if row_idx == 1 and col_idx == 2:
            add_legend(ax, fontsize=7)

plt.savefig("umap_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: umap_grid.png")


# ══════════════════════════════════════════════════════════════════════════════
# 8. UMAP DEEP DIVE — BEST CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# Use all-layers balanced as primary result
best_umap_key    = "all_layers_15"
best_umap_coords = umap_results[best_umap_key]["coords"]

fig5, axes5 = plt.subplots(1, 2, figsize=(16, 7))
fig5.suptitle(
    "UMAP Deep Dive — All Layers Concatenated, n_neighbors=15\n"
    "Left: tier colours  |  Right: template type colours",
    fontsize=12, fontweight="bold"
)

# ── Left: tier colours
scatter_by_tier(axes5[0], best_umap_coords,
                entity_meta["tier"], entity_meta["entity"],
                hull=True, alpha=0.9, label_entities=True)
add_legend(axes5[0])
axes5[0].set_title("Coloured by sentience tier", fontsize=10)

# ── Right: template type colours — do welfare-framed sentences cluster?
# For this panel use sentence-level UMAP

print("\nRunning sentence-level UMAP for template-type panel...")

X_sent_final = matrix[:, -1, :]
X_sent_scaled = StandardScaler().fit_transform(X_sent_final)
reducer_sent  = umap.UMAP(n_components=2, n_neighbors=15,
                           min_dist=0.1, metric="cosine",
                           random_state=42, verbose=False)
sent_coords = reducer_sent.fit_transform(X_sent_scaled)

template_colors = {
    "pain_suffering":     "#d7301f",
    "fear":               "#fc8d59",
    "scientific_welfare": "#91bfdb",
    "moral_consideration":"#4575b4",
    "neutral_control":    "#a6a6a6",
    "cross_framing":      "#1a9641",
}

for ttype, color in template_colors.items():
    mask = (meta_df["template_type"] == ttype).values
    if mask.sum() > 0:
        axes5[1].scatter(
            sent_coords[mask, 0], sent_coords[mask, 1],
            c=color, alpha=0.4, s=12, label=ttype,
            edgecolors="none"
        )

axes5[1].legend(fontsize=8, loc="lower right",
                framealpha=0.9, title="Template type")
axes5[1].set_title("Coloured by template type\n"
                    "(sentence level, final layer)",
                    fontsize=10)
axes5[1].set_xticks([])
axes5[1].set_yticks([])
axes5[1].spines[["top","right","bottom","left"]].set_visible(False)

plt.tight_layout()
plt.savefig("umap_deepdive.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: umap_deepdive.png")


# ══════════════════════════════════════════════════════════════════════════════
# 9. QUANTITATIVE STRUCTURE METRICS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═"*60)
print("  QUANTITATIVE STRUCTURE METRICS")
print("═"*60)

# ── Metric 1: Inter-tier separation on PC1 (final layer)
# If a gradient exists, T1 and T6 should be well-separated on PC1
print("\n  1. PC1 tier separation (final layer entity means):")
final_pc1 = pca_entity_results[final_layer]["coords"][:, 0]

tier_pc1_means = {}
for tier in TIER_ORDER:
    mask = (entity_meta["tier"] == tier).values
    vals = final_pc1[mask]
    tier_pc1_means[tier] = vals.mean()
    print(f"    {tier}: mean={vals.mean():+.3f}  std={vals.std():.3f}  "
          f"n={mask.sum()}")

# Spearman correlation: does tier order predict PC1?
tier_numeric = {"T1":1,"T2":2,"T3":3,"T4":4,"T5":5,"T6":6}
tier_nums    = entity_meta["tier"].map(tier_numeric).values
spearman_r, spearman_p = stats.spearmanr(tier_nums, final_pc1)
print(f"\n    Spearman r (tier_rank vs PC1): {spearman_r:+.3f}  "
      f"p={spearman_p:.4f}")
if abs(spearman_r) > 0.3 and spearman_p < 0.05:
    print("    → Statistically significant tier gradient on PC1")
else:
    print("    → No significant linear tier gradient on PC1")

# ── Metric 2: Within-tier vs between-tier variance (final layer)
print("\n  2. Variance decomposition (final layer):")

total_var = final_pc1.var()
tier_means_arr = np.array([tier_pc1_means[t]
                            for t in entity_meta["tier"]])
between_var = tier_means_arr.var()
within_var  = total_var - between_var
print(f"    Total PC1 variance    : {total_var:.4f}")
print(f"    Between-tier variance : {between_var:.4f}  "
      f"({100*between_var/total_var:.1f}%)")
print(f"    Within-tier variance  : {within_var:.4f}  "
      f"({100*within_var/total_var:.1f}%)")

# ── Metric 3: T1 vs T6 separation across layers (effect size)
print("\n  3. T1 vs T6 Cohen's d on PC1 across sampled layers:")
print(f"    {'Layer':>6}  {'T1 mean':>9}  {'T6 mean':>9}  "
      f"{'Cohen d':>9}  {'separation':>12}")

for layer in SAMPLE_LAYERS:
    coords_l = pca_results[layer]["coords"]   # sentence level
    t1_vals  = coords_l[meta_df["tier"]=="T1", 0]
    t6_vals  = coords_l[meta_df["tier"]=="T6", 0]
    pooled_std = np.sqrt((t1_vals.std()**2 + t6_vals.std()**2) / 2)
    cohens_d   = (t1_vals.mean() - t6_vals.mean()) / (pooled_std + 1e-9)
    direction  = ("T1>T6" if t1_vals.mean() > t6_vals.mean() else "T6>T1")
    print(f"    {layer:>6}  {t1_vals.mean():>+9.3f}  {t6_vals.mean():>+9.3f}  "
          f"{cohens_d:>+9.3f}  {direction:>12}")


# ══════════════════════════════════════════════════════════════════════════════
# 10. VARIANCE EXPLAINED SUMMARY PLOT
# ══════════════════════════════════════════════════════════════════════════════

print("\nBuilding variance explained summary...")

all_layers_range = list(range(N_LAYERS))
ve_pc1 = []
ve_pc2 = []
tier_sep = []   # Spearman r at each layer

for layer in all_layers_range:
    X = matrix[:, layer, :]
    X_s = StandardScaler().fit_transform(X)
    pca_tmp = PCA(n_components=2, random_state=42)
    coords_tmp = pca_tmp.fit_transform(X_s)
    ve_pc1.append(pca_tmp.explained_variance_ratio_[0])
    ve_pc2.append(pca_tmp.explained_variance_ratio_[1])

    # Spearman between tier rank and PC1
    tier_nums_sent = meta_df["tier"].map(tier_numeric).values
    r, p = stats.spearmanr(tier_nums_sent, coords_tmp[:, 0])
    tier_sep.append(abs(r))

fig6, axes6 = plt.subplots(1, 2, figsize=(14, 5))
fig6.suptitle("PCA Summary Across All Layers", fontsize=12, fontweight="bold")

ax_ve = axes6[0]
ax_ve.plot(all_layers_range, [v*100 for v in ve_pc1],
           color="#2171b5", linewidth=2, marker="o",
           markersize=3, label="PC1")
ax_ve.plot(all_layers_range, [v*100 for v in ve_pc2],
           color="#fd8d3c", linewidth=2, marker="o",
           markersize=3, label="PC2")
ax_ve.fill_between(all_layers_range,
                   [v*100 for v in ve_pc1], alpha=0.15, color="#2171b5")
for sl in SAMPLE_LAYERS:
    ax_ve.axvline(sl, color="grey", linewidth=0.6, linestyle=":", alpha=0.7)
ax_ve.set_xlabel("Layer")
ax_ve.set_ylabel("Variance explained (%)")
ax_ve.set_title("PC1 & PC2 Variance Explained\n"
                "(vertical lines = sampled layers)")
ax_ve.legend()

ax_sep = axes6[1]
ax_sep.plot(all_layers_range, tier_sep,
            color="#d7301f", linewidth=2, marker="o", markersize=3)
ax_sep.fill_between(all_layers_range, tier_sep,
                    alpha=0.15, color="#d7301f")
ax_sep.axhline(0.3, color="black", linewidth=1,
               linestyle="--", alpha=0.5, label="|r|=0.3 threshold")
for sl in SAMPLE_LAYERS:
    ax_sep.axvline(sl, color="grey", linewidth=0.6, linestyle=":", alpha=0.7)
ax_sep.set_xlabel("Layer")
ax_sep.set_ylabel("|Spearman r|  (tier rank vs PC1)")
ax_sep.set_title("Tier–PC1 Separation Strength by Layer\n"
                 "(peak = layer where gradient is strongest)")
ax_sep.set_ylim(0, 1)
ax_sep.legend(fontsize=9)

# Annotate peak
peak_layer = int(np.argmax(tier_sep))
peak_val   = tier_sep[peak_layer]
ax_sep.annotate(
    f"Peak: layer {peak_layer}\n|r|={peak_val:.2f}",
    xy     = (peak_layer, peak_val),
    xytext = (peak_layer + 1.5, peak_val - 0.12),
    arrowprops=dict(arrowstyle="->", color="black", lw=1.2),
    fontsize=9, fontweight="bold",
)

plt.tight_layout()
plt.savefig("pca_variance_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: pca_variance_summary.png")


# ══════════════════════════════════════════════════════════════════════════════
# 11. SAVE PROJECTION COORDINATES
# ══════════════════════════════════════════════════════════════════════════════

print("\nSaving projection coordinates...")

# PCA entity-level coords for all sampled layers
pca_rows = []
for layer in SAMPLE_LAYERS:
    coords = pca_entity_results[layer]["coords"]
    for i, (x, y) in enumerate(coords):
        pca_rows.append({
            "entity":  entity_meta.iloc[i]["entity"],
            "tier":    entity_meta.iloc[i]["tier"],
            "layer":   layer,
            "pc1":     float(x),
            "pc2":     float(y),
            "method":  "pca_entity_mean",
        })

# UMAP entity-level coords (best config)
for i, (x, y) in enumerate(best_umap_coords):
    pca_rows.append({
        "entity":  entity_meta.iloc[i]["entity"],
        "tier":    entity_meta.iloc[i]["tier"],
        "layer":   "all_concat",
        "pc1":     float(x),
        "pc2":     float(y),
        "method":  "umap_all_layers",
    })

projections_df = pd.DataFrame(pca_rows)
projections_df.to_csv("projection_coordinates.csv", index=False)
print("Saved: projection_coordinates.csv")

# Tier separation metrics
sep_df = pd.DataFrame({
    "layer":       all_layers_range,
    "ve_pc1":      ve_pc1,
    "ve_pc2":      ve_pc2,
    "spearman_r":  tier_sep,
})
sep_df.to_csv("pca_separation_metrics.csv", index=False)
print("Saved: pca_separation_metrics.csv")

print("\n── Output files ──")
print("  pca_per_layer_sentences.png      PCA grid — all sentences")
print("  pca_per_layer_entity_means.png   PCA grid — entity means, labelled")
print("  pca_final_layer_deepdive.png     4-panel final layer deep dive")
print("  umap_grid.png                    UMAP hyperparameter grid")
print("  umap_deepdive.png                UMAP best config + template type")
print("  pca_variance_summary.png         VE% and tier separation across layers")
print("  projection_coordinates.csv       PCA & UMAP coords for all entities")
print("  pca_separation_metrics.csv       Layer-wise Spearman r and VE%")


What to Look For in the Outputs ^^

PCA grid (pca_per_layer_entity_means.png) — the key figure. If a sentience gradient is encoded, you should see:

- Early layers: no clear tier separation, mixed cloud
- Middle layers: tier clusters beginning to form
- Final layers: T1 (dark blue) and T6 (grey) pulling to opposite ends of PC1

Variance summary (pca_variance_summary.png) — the Spearman r panel is the single most important diagnostic. The peak layer is your target for the activation patching experiment.

UMAP grid (umap_grid.png) — compare all-layers-concatenated vs final-layer-only. If the all-layers version shows cleaner tier separation, the gradient is distributed across depth rather than localised in one layer.

Panel C of the deep dive — tier centroid trajectories across layers. A crossing or divergence pattern here is the clearest visual evidence that the model is doing something tier-specific at a particular depth.

## For each layer, train a logistic regression predicting tier (6-class) from the activation vector 

* Use 5-fold cross-validation; record mean accuracy and standard deviation 

* Also train a binary probe: sentient (T1–T5) vs non-sentient (T6) — this is a cleaner signal and more defensible scientifically 

* Plot probe accuracy vs layer depth — this is one of your key figures 

* Sanity check: shuffle tier labels and re-run probe; accuracy should drop to chance (~17% for 6-class). If it doesn't, you have a data leakage problem 

* Secondary probe: train on welfare-framed sentences only, test on neutral-framed sentences — does the gradient generalise across framing?

In [ ]:
import numpy as np
import pandas as pd
import h5py
import json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                      cross_validate, LeaveOneOut)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (confusion_matrix, classification_report,
                              ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
import warnings
from pathlib import Path
from scipy import stats
from collections import defaultdict
import time

warnings.filterwarnings('ignore')


# ══════════════════════════════════════════════════════════════════════════════
# 1. CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

HDF5_PATH   = "activations_1.4b.h5"
N_LAYERS    = 24
N_FOLDS     = 5
RANDOM_SEED = 42

TIER_ORDER  = ["T1","T2","T3","T4","T5","T6"]
TIER_CONFIG = {
    "T1": {"label": "T1 — Humans",              "color": "#08306b"},
    "T2": {"label": "T2 — Mammals",             "color": "#2171b5"},
    "T3": {"label": "T3 — Vertebrates",         "color": "#6baed6"},
    "T4": {"label": "T4 — Invertebrates (w+)",  "color": "#fd8d3c"},
    "T5": {"label": "T5 — Invertebrates (min)", "color": "#d7301f"},
    "T6": {"label": "T6 — Non-sentient",        "color": "#bdbdbd"},
}

TEMPLATE_TYPES = {
    "welfare":  ["pain_suffering", "fear",
                 "moral_consideration", "cross_framing"],
    "science":  ["scientific_welfare"],
    "neutral":  ["neutral_control"],
}

# Logistic regression settings
LR_PARAMS = dict(
    max_iter   = 1000,
    solver     = "lbfgs",
    C          = 1.0,
    multi_class= "multinomial",
    random_state = RANDOM_SEED,
)

N_SHUFFLE_RUNS = 20    # how many shuffles for null distribution

print("Sentience Salience Probe — Linear Probe Analysis")
print("="*60)


# ══════════════════════════════════════════════════════════════════════════════
# 2. DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_data(hdf5_path: str) -> tuple[np.ndarray, pd.DataFrame]:
    """
    Load all residual stream vectors.
    Returns
    -------
    matrix   : (n_sentences, n_layers, d_model)  float32
    meta_df  : one row per sentence
    """
    matrices  = []
    meta_rows = []

    with h5py.File(hdf5_path, "r") as f:
        for sid in sorted(f.keys()):
            grp = f[sid]
            matrices.append(grp["resid_post"][:].astype(np.float32))
            meta_rows.append({
                "sentence_id":   sid,
                "entity":        grp.attrs["entity"],
                "tier":          grp.attrs["tier"],
                "template_id":   grp.attrs["template_id"],
                "template_type": grp.attrs["template_type"],
            })

    matrix  = np.stack(matrices)
    meta_df = pd.DataFrame(meta_rows)

    # Sentient / non-sentient binary label
    meta_df["binary_label"] = meta_df["tier"].apply(
        lambda t: "sentient" if t != "T6" else "non_sentient"
    )

    # Framing group
    def framing_group(ttype):
        for group, ttypes in TEMPLATE_TYPES.items():
            if ttype in ttypes:
                return group
        return "other"

    meta_df["framing"] = meta_df["template_type"].apply(framing_group)

    print(f"  Loaded {len(meta_df):,} sentences  |  "
          f"matrix shape: {matrix.shape}")
    print(f"  Tiers: {meta_df['tier'].value_counts().to_dict()}")
    print(f"  Framings: {meta_df['framing'].value_counts().to_dict()}")

    return matrix, meta_df


print("\nLoading data...")
matrix, meta_df = load_data(HDF5_PATH)

N_SENTENCES = matrix.shape[0]


# ══════════════════════════════════════════════════════════════════════════════
# 3. PROBE BUILDING BLOCKS
# ══════════════════════════════════════════════════════════════════════════════

def make_pipeline(n_classes: int = 6) -> Pipeline:
    """
    Logistic regression pipeline with standard scaling.
    Scaling is fitted inside each CV fold to prevent leakage.
    """
    lr_params = LR_PARAMS.copy()
    if n_classes == 2:
        lr_params["multi_class"] = "ovr"
    return Pipeline([
        ("scaler", StandardScaler()),
        ("lr",     LogisticRegression(**lr_params)),
    ])


def probe_layer(X:          np.ndarray,
                y:          np.ndarray,
                n_folds:    int = N_FOLDS,
                seed:       int = RANDOM_SEED) -> dict:
    """
    Run stratified k-fold CV for one layer.

    Returns dict with keys:
        mean_acc, std_acc, fold_accs, chance_level,
        mean_f1, std_f1
    """
    n_classes = len(np.unique(y))
    pipe = make_pipeline(n_classes)
    cv   = StratifiedKFold(n_splits=n_folds,
                            shuffle=True,
                            random_state=seed)

    scores = cross_validate(
        pipe, X, y, cv=cv,
        scoring  = ["accuracy", "f1_macro"],
        n_jobs   = -1,
    )

    return {
        "mean_acc":     scores["test_accuracy"].mean(),
        "std_acc":      scores["test_accuracy"].std(),
        "fold_accs":    scores["test_accuracy"].tolist(),
        "mean_f1":      scores["test_f1_macro"].mean(),
        "std_f1":       scores["test_f1_macro"].std(),
        "chance_level": 1.0 / n_classes,
    }


def probe_all_layers(matrix:   np.ndarray,
                      y:        np.ndarray,
                      label:    str = "") -> pd.DataFrame:
    """
    Run probe at every layer. Returns DataFrame of results.
    """
    rows = []
    t0   = time.time()

    for layer in range(N_LAYERS):
        X      = matrix[:, layer, :]
        result = probe_layer(X, y)
        result["layer"] = layer
        rows.append(result)

        if layer % 6 == 0 or layer == N_LAYERS - 1:
            elapsed = time.time() - t0
            print(f"    Layer {layer:2d}/{N_LAYERS-1}  "
                  f"acc={result['mean_acc']:.3f}±{result['std_acc']:.3f}  "
                  f"f1={result['mean_f1']:.3f}  "
                  f"({elapsed:.0f}s elapsed)")

    df = pd.DataFrame(rows)
    df["probe_label"] = label
    return df


# ══════════════════════════════════════════════════════════════════════════════
# 4. PROBE A — 6-CLASS TIER PREDICTION
# ══════════════════════════════════════════════════════════════════════════════

print("\n── Probe A: 6-class tier prediction ──")
y_6class = meta_df["tier"].values
print(f"  Classes: {np.unique(y_6class)}  "
      f"  Chance: {1/6*100:.1f}%  "
      f"  n={len(y_6class)}")

results_6class = probe_all_layers(matrix, y_6class,
                                   label="6class_tier")
print(f"\n  Peak accuracy: "
      f"{results_6class['mean_acc'].max():.3f} "
      f"at layer {results_6class['mean_acc'].idxmax()}")


# ══════════════════════════════════════════════════════════════════════════════
# 5. PROBE B — BINARY: SENTIENT vs NON-SENTIENT
# ══════════════════════════════════════════════════════════════════════════════

print("\n── Probe B: Binary sentient vs non-sentient ──")
y_binary = (meta_df["binary_label"] == "sentient").astype(int).values
print(f"  Sentient: {y_binary.sum()}  "
      f"Non-sentient: {(1-y_binary).sum()}  "
      f"  Chance: 50%  n={len(y_binary)}")

results_binary = probe_all_layers(matrix, y_binary,
                                   label="binary_sentient")
print(f"\n  Peak accuracy: "
      f"{results_binary['mean_acc'].max():.3f} "
      f"at layer {results_binary['mean_acc'].idxmax()}")


# ══════════════════════════════════════════════════════════════════════════════
# 6. SANITY CHECK — SHUFFLED LABELS NULL DISTRIBUTION
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n── Sanity check: shuffled label null distribution ──")
print(f"  Running {N_SHUFFLE_RUNS} shuffles × {N_LAYERS} layers...")
print(f"  Expected chance accuracy: {1/6*100:.1f}% (6-class)")

rng           = np.random.default_rng(RANDOM_SEED)
shuffle_accs  = np.zeros((N_SHUFFLE_RUNS, N_LAYERS))

for run in range(N_SHUFFLE_RUNS):
    y_shuffled = rng.permutation(y_6class)
    for layer in range(N_LAYERS):
        X    = matrix[:, layer, :]
        res  = probe_layer(X, y_shuffled, n_folds=3)   # 3-fold for speed
        shuffle_accs[run, layer] = res["mean_acc"]

    if run % 5 == 0:
        print(f"    Shuffle run {run+1}/{N_SHUFFLE_RUNS}  "
              f"mean acc across layers: {shuffle_accs[run].mean():.3f}")

shuffle_mean = shuffle_accs.mean(axis=0)   # (n_layers,)
shuffle_std  = shuffle_accs.std(axis=0)
shuffle_95   = np.percentile(shuffle_accs, 95, axis=0)   # upper 95th pct

print(f"\n  Shuffled mean accuracy (all layers): "
      f"{shuffle_mean.mean():.4f} ± {shuffle_mean.std():.4f}")
print(f"  Expected chance: {1/6:.4f}")

# Leakage check
real_peak   = results_6class["mean_acc"].max()
null_95_max = shuffle_95.max()
if real_peak > null_95_max:
    print(f"  ✓ Real probe ({real_peak:.3f}) > null 95th pct ({null_95_max:.3f})"
          f"  — no leakage detected")
else:
    print(f"  ✗ Real probe ({real_peak:.3f}) ≤ null 95th pct ({null_95_max:.3f})"
          f"  — CHECK FOR LEAKAGE")


# ══════════════════════════════════════════════════════════════════════════════
# 7. PROBE C — GENERALISATION ACROSS FRAMING
#    Train on welfare sentences, test on neutral sentences
# ══════════════════════════════════════════════════════════════════════════════

print("\n── Probe C: Generalisation across framing ──")
print("  Train on welfare-framed sentences → test on neutral sentences")

welfare_mask = (meta_df["framing"] == "welfare").values
neutral_mask = (meta_df["framing"] == "neutral").values
science_mask = (meta_df["framing"] == "science").values

print(f"  Welfare train: {welfare_mask.sum()}  "
      f"  Neutral test: {neutral_mask.sum()}  "
      f"  Science test: {science_mask.sum()}")

def probe_generalisation(matrix:        np.ndarray,
                          y:             np.ndarray,
                          train_mask:    np.ndarray,
                          test_mask:     np.ndarray,
                          train_label:   str = "train",
                          test_label:    str = "test") -> pd.DataFrame:
    """
    Train probe on train_mask sentences; evaluate on test_mask sentences.
    No cross-validation — clean train/test split by design.
    """
    rows = []
    n_classes = len(np.unique(y[train_mask]))

    for layer in range(N_LAYERS):
        X_train = matrix[train_mask, layer, :]
        X_test  = matrix[test_mask,  layer, :]
        y_train = y[train_mask]
        y_test  = y[test_mask]

        pipe = make_pipeline(n_classes)
        pipe.fit(X_train, y_train)
        acc  = pipe.score(X_test, y_test)

        # Also evaluate on training set to check overfitting
        train_acc = pipe.score(X_train, y_train)

        rows.append({
            "layer":       layer,
            "test_acc":    acc,
            "train_acc":   train_acc,
            "train_label": train_label,
            "test_label":  test_label,
            "n_train":     train_mask.sum(),
            "n_test":      test_mask.sum(),
        })

    df = pd.DataFrame(rows)
    print(f"  [{train_label} → {test_label}]  "
          f"peak test acc = {df['test_acc'].max():.3f} "
          f"at layer {df['test_acc'].idxmax()}")
    return df


# 6-class generalisation
y_6 = meta_df["tier"].values
gen_welfare_to_neutral  = probe_generalisation(
    matrix, y_6, welfare_mask, neutral_mask,
    "welfare", "neutral"
)
gen_welfare_to_science  = probe_generalisation(
    matrix, y_6, welfare_mask, science_mask,
    "welfare", "science"
)
gen_neutral_to_welfare  = probe_generalisation(
    matrix, y_6, neutral_mask, welfare_mask,
    "neutral", "welfare"
)

# Binary generalisation
y_bin = y_binary
gen_bin_w2n = probe_generalisation(
    matrix, y_bin, welfare_mask, neutral_mask,
    "welfare→neutral (binary)", "neutral"
)


# ══════════════════════════════════════════════════════════════════════════════
# 8. CONFUSION MATRIX — BEST LAYER
# ══════════════════════════════════════════════════════════════════════════════

print("\n── Confusion matrix at peak layer ──")

best_layer = int(results_6class["mean_acc"].idxmax())
print(f"  Peak layer: {best_layer}  "
      f"acc={results_6class.loc[best_layer,'mean_acc']:.3f}")

X_best = matrix[:, best_layer, :]
y_6    = meta_df["tier"].values

# Full fit on all data for confusion matrix (for display only)
# True evaluation used CV above
pipe_cm = make_pipeline(6)
cv      = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                           random_state=RANDOM_SEED)

all_y_true = []
all_y_pred = []
for train_idx, test_idx in cv.split(X_best, y_6):
    pipe_cm.fit(X_best[train_idx], y_6[train_idx])
    preds = pipe_cm.predict(X_best[test_idx])
    all_y_true.extend(y_6[test_idx])
    all_y_pred.extend(preds)

all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)

cm = confusion_matrix(all_y_true, all_y_pred, labels=TIER_ORDER)

print("\n  Classification report:")
print(classification_report(all_y_true, all_y_pred,
                             target_names=TIER_ORDER))


# ══════════════════════════════════════════════════════════════════════════════
# 9. PER-TIER ACCURACY ACROSS LAYERS
# ══════════════════════════════════════════════════════════════════════════════

print("\n── Per-tier accuracy across layers ──")

def per_tier_accuracy(matrix: np.ndarray,
                       meta_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each layer, compute per-tier accuracy using CV.
    Returns DataFrame with columns: layer, tier, accuracy.
    """
    rows = []
    y_6  = meta_df["tier"].values

    for layer in range(N_LAYERS):
        X  = matrix[:, layer, :]
        cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                              random_state=RANDOM_SEED)
        pipe = make_pipeline(6)

        y_true_all, y_pred_all = [], []
        for train_idx, test_idx in cv.split(X, y_6):
            pipe.fit(X[train_idx], y_6[train_idx])
            y_pred = pipe.predict(X[test_idx])
            y_true_all.extend(y_6[test_idx])
            y_pred_all.extend(y_pred)

        y_true_all = np.array(y_true_all)
        y_pred_all = np.array(y_pred_all)

        for tier in TIER_ORDER:
            mask = y_true_all == tier
            if mask.sum() > 0:
                acc = (y_pred_all[mask] == tier).mean()
                rows.append({"layer": layer, "tier": tier, "accuracy": acc})

    return pd.DataFrame(rows)


tier_acc_df = per_tier_accuracy(matrix, meta_df)


# ══════════════════════════════════════════════════════════════════════════════
# 10. STATISTICAL TESTS
# ══════════════════════════════════════════════════════════════════════════════

print("\n── Statistical tests ──")

# Test 1: Is peak 6-class accuracy significantly above chance?
peak_row    = results_6class.loc[results_6class["mean_acc"].idxmax()]
fold_accs   = peak_row["fold_accs"]
chance      = 1/6

t_stat, p_val = stats.ttest_1samp(fold_accs, chance)
print(f"\n  6-class probe (layer {best_layer}) vs chance ({chance:.3f}):")
print(f"    Mean acc = {peak_row['mean_acc']:.3f}  "
      f"t={t_stat:.3f}  p={p_val:.4f}")
print(f"    {'Significantly above chance' if p_val < 0.05 else 'NOT significantly above chance'}")

# Test 2: Binary probe vs 50%
peak_bin     = results_binary.loc[results_binary["mean_acc"].idxmax()]
fold_accs_b  = peak_bin["fold_accs"]
t_b, p_b     = stats.ttest_1samp(fold_accs_b, 0.5)
print(f"\n  Binary probe (layer {peak_bin['layer']}) vs chance (0.50):")
print(f"    Mean acc = {peak_bin['mean_acc']:.3f}  "
      f"t={t_b:.3f}  p={p_b:.4f}")

# Test 3: Generalisation — is welfare→neutral significantly above chance?
gen_peak = gen_welfare_to_neutral["test_acc"].max()
gen_layer = gen_welfare_to_neutral["test_acc"].idxmax()
print(f"\n  Generalisation (welfare→neutral) peak: "
      f"{gen_peak:.3f} at layer {gen_layer}")
print(f"  Drop from in-distribution: "
      f"{results_6class['mean_acc'].max() - gen_peak:+.3f}")

# Test 4: Does probe accuracy correlate with layer depth?
spearman_6class = stats.spearmanr(
    results_6class["layer"],
    results_6class["mean_acc"]
)
print(f"\n  Spearman r (layer depth vs 6-class accuracy): "
      f"{spearman_6class.statistic:+.3f}  "
      f"p={spearman_6class.pvalue:.4f}")


# ══════════════════════════════════════════════════════════════════════════════
# 11. MAIN FIGURE — PROBE ACCURACY VS LAYER DEPTH
# ══════════════════════════════════════════════════════════════════════════════

print("\nBuilding figures...")

layers = list(range(N_LAYERS))

fig = plt.figure(figsize=(20, 22))
gs  = gridspec.GridSpec(4, 2, figure=fig,
                         hspace=0.42, wspace=0.32)

fig.suptitle(
    "Linear Probe Analysis — Sentience Salience Probe\n"
    f"Pythia-1.4b  |  {N_SENTENCES} sentences  |  "
    f"{N_FOLDS}-fold stratified CV",
    fontsize=14, fontweight="bold", y=0.98
)

PROBE_BLUE   = "#2171b5"
PROBE_RED    = "#d7301f"
PROBE_GREEN  = "#238b45"
PROBE_ORANGE = "#fd8d3c"
SHUFFLE_GREY = "#aaaaaa"


# ── Panel A: 6-class probe + null distribution ─────────────────────────────
ax_a = fig.add_subplot(gs[0, :])

# Null distribution band
ax_a.fill_between(
    layers,
    shuffle_mean - 2*shuffle_std,
    shuffle_mean + 2*shuffle_std,
    alpha=0.25, color=SHUFFLE_GREY, label="Null ±2σ (shuffled labels)"
)
ax_a.plot(layers, shuffle_mean,
          color=SHUFFLE_GREY, linewidth=1.5,
          linestyle="--", label="Null mean (shuffled)")
ax_a.plot(layers, shuffle_95,
          color=SHUFFLE_GREY, linewidth=1,
          linestyle=":", alpha=0.8, label="Null 95th pct")

# Real probe with CI
means = results_6class["mean_acc"].values
stds  = results_6class["std_acc"].values
ax_a.fill_between(layers, means - stds, means + stds,
                  alpha=0.18, color=PROBE_BLUE)
ax_a.plot(layers, means,
          color=PROBE_BLUE, linewidth=2.5, marker="o",
          markersize=5, label="6-class probe (mean ± 1σ CV)")

# Chance line
ax_a.axhline(1/6, color="black", linewidth=1,
             linestyle=":", alpha=0.6, label="Chance (16.7%)")

# Annotate peak
peak_idx  = int(np.argmax(means))
peak_val  = means[peak_idx]
ax_a.annotate(
    f"Peak: layer {peak_idx}\nacc={peak_val:.3f}",
    xy     = (peak_idx, peak_val),
    xytext = (peak_idx + 1.5, peak_val + 0.04),
    arrowprops = dict(arrowstyle="->", color=PROBE_BLUE, lw=1.5),
    fontsize=9, color=PROBE_BLUE, fontweight="bold",
)

ax_a.set_xlabel("Layer", fontsize=11)
ax_a.set_ylabel("Classification accuracy", fontsize=11)
ax_a.set_title(
    "Panel A — 6-class Tier Probe vs Null Distribution\n"
    "Probe trained to predict T1–T6 from residual stream at entity token",
    fontsize=11, fontweight="bold"
)
ax_a.set_ylim(0, 1)
ax_a.set_xticks(layers)
ax_a.legend(fontsize=9, loc="upper left")
ax_a.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda y, _: f"{y*100:.0f}%")
)


# ── Panel B: Binary probe ──────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[1, 0])

bin_means = results_binary["mean_acc"].values
bin_stds  = results_binary["std_acc"].values

ax_b.fill_between(layers, bin_means - bin_stds, bin_means + bin_stds,
                  alpha=0.18, color=PROBE_RED)
ax_b.plot(layers, bin_means,
          color=PROBE_RED, linewidth=2.5, marker="o",
          markersize=5, label="Binary probe")
ax_b.axhline(0.5, color="black", linewidth=1,
             linestyle=":", alpha=0.6, label="Chance (50%)")
ax_b.fill_between(layers,
                   shuffle_mean - 2*shuffle_std,
                   shuffle_mean + 2*shuffle_std,
                   alpha=0.15, color=SHUFFLE_GREY)
ax_b.plot(layers, shuffle_mean,
          color=SHUFFLE_GREY, linewidth=1.2,
          linestyle="--", label="Null mean (6-class shuffle)")

peak_b     = int(np.argmax(bin_means))
ax_b.annotate(
    f"Peak L{peak_b}\n{bin_means[peak_b]:.3f}",
    xy     = (peak_b, bin_means[peak_b]),
    xytext = (peak_b + 1.5, bin_means[peak_b] - 0.08),
    arrowprops = dict(arrowstyle="->", color=PROBE_RED, lw=1.2),
    fontsize=8, color=PROBE_RED, fontweight="bold",
)

ax_b.set_xlabel("Layer", fontsize=10)
ax_b.set_ylabel("Accuracy", fontsize=10)
ax_b.set_title(
    "Panel B — Binary Probe\nSentient (T1–T5) vs Non-sentient (T6)",
    fontsize=10, fontweight="bold"
)
ax_b.set_ylim(0.4, 1.05)
ax_b.set_xticks(layers[::2])
ax_b.legend(fontsize=8)
ax_b.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda y, _: f"{y*100:.0f}%")
)


# ── Panel C: Macro F1 for 6-class ─────────────────────────────────────────
ax_c = fig.add_subplot(gs[1, 1])

f1_means = results_6class["mean_f1"].values
f1_stds  = results_6class["std_f1"].values

ax_c.fill_between(layers, f1_means - f1_stds, f1_means + f1_stds,
                  alpha=0.18, color=PROBE_ORANGE)
ax_c.plot(layers, f1_means,
          color=PROBE_ORANGE, linewidth=2.5, marker="o",
          markersize=5, label="Macro F1")
ax_c.axhline(1/6, color="black", linewidth=1,
             linestyle=":", alpha=0.6, label="Chance baseline")

peak_f1   = int(np.argmax(f1_means))
ax_c.annotate(
    f"Peak L{peak_f1}\nF1={f1_means[peak_f1]:.3f}",
    xy     = (peak_f1, f1_means[peak_f1]),
    xytext = (peak_f1 + 1.5, f1_means[peak_f1] - 0.07),
    arrowprops = dict(arrowstyle="->", color=PROBE_ORANGE, lw=1.2),
    fontsize=8, color=PROBE_ORANGE, fontweight="bold",
)

ax_c.set_xlabel("Layer", fontsize=10)
ax_c.set_ylabel("Macro F1", fontsize=10)
ax_c.set_title(
    "Panel C — Macro F1 Score (6-class)\n"
    "Balances across tiers regardless of class size",
    fontsize=10, fontweight="bold"
)
ax_c.set_ylim(0, 1)
ax_c.set_xticks(layers[::2])
ax_c.legend(fontsize=8)


# ── Panel D: Generalisation across framing ────────────────────────────────
ax_d = fig.add_subplot(gs[2, :])

# In-distribution reference
ax_d.fill_between(layers, means - stds, means + stds,
                  alpha=0.12, color=PROBE_BLUE)
ax_d.plot(layers, means,
          color=PROBE_BLUE, linewidth=2, linestyle="-",
          marker="o", markersize=4,
          label="In-distribution (welfare CV, 6-class)")

# Generalisation curves
gen_configs = [
    (gen_welfare_to_neutral,  PROBE_RED,    "Welfare→Neutral",  "--"),
    (gen_welfare_to_science,  PROBE_ORANGE, "Welfare→Science",  "-."),
    (gen_neutral_to_welfare,  PROBE_GREEN,  "Neutral→Welfare",  ":"),
]
for gen_df, color, label, ls in gen_configs:
    ax_d.plot(layers, gen_df["test_acc"].values,
              color=color, linewidth=2, linestyle=ls,
              marker="s", markersize=4, label=label)

ax_d.axhline(1/6, color="black", linewidth=1,
             linestyle=":", alpha=0.6, label="Chance")

# Shade the generalisation gap at peak layer
ax_d.annotate(
    f"Generalisation gap\nat layer {peak_idx}",
    xy     = (peak_idx, gen_welfare_to_neutral["test_acc"].iloc[peak_idx]),
    xytext = (peak_idx + 2, gen_welfare_to_neutral["test_acc"].iloc[peak_idx] + 0.1),
    arrowprops = dict(arrowstyle="->", color="black", lw=1),
    fontsize=8,
)
ax_d.annotate(
    "",
    xy     = (peak_idx, means[peak_idx]),
    xytext = (peak_idx, gen_welfare_to_neutral["test_acc"].iloc[peak_idx]),
    arrowprops = dict(arrowstyle="<->", color="black", lw=1.5),
)

ax_d.set_xlabel("Layer", fontsize=11)
ax_d.set_ylabel("Accuracy", fontsize=11)
ax_d.set_title(
    "Panel D — Generalisation Across Framing\n"
    "Does the sentience gradient generalise from welfare-framed "
    "to neutral sentences? If yes: entity-level encoding. If no: framing-dependent.",
    fontsize=10, fontweight="bold"
)
ax_d.set_ylim(0, 1.05)
ax_d.set_xticks(layers)
ax_d.legend(fontsize=9, loc="upper left", ncols=2)
ax_d.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda y, _: f"{y*100:.0f}%")
)


# ── Panel E: Per-tier accuracy at peak layer ───────────────────────────────
ax_e = fig.add_subplot(gs[3, 0])

best_layer_tier = tier_acc_df[
    tier_acc_df["layer"] == best_layer
].set_index("tier")["accuracy"]

bars = ax_e.bar(
    TIER_ORDER,
    [best_layer_tier.get(t, 0) for t in TIER_ORDER],
    color  = [TIER_CONFIG[t]["color"] for t in TIER_ORDER],
    edgecolor = "white",
    linewidth  = 0.5,
)
ax_e.axhline(1/6, color="black", linewidth=1.2,
             linestyle="--", alpha=0.7, label="Chance (16.7%)")
ax_e.set_xlabel("Tier", fontsize=10)
ax_e.set_ylabel("Per-tier accuracy", fontsize=10)
ax_e.set_title(
    f"Panel E — Per-Tier Accuracy at Peak Layer {best_layer}\n"
    "Which tiers are most/least discriminable?",
    fontsize=10, fontweight="bold"
)
ax_e.set_ylim(0, 1)
ax_e.legend(fontsize=9)
ax_e.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda y, _: f"{y*100:.0f}%")
)
ax_e.set_xticklabels(
    [TIER_CONFIG[t]["label"].split("—")[0].strip() for t in TIER_ORDER],
    fontsize=8
)

for bar, tier in zip(bars, TIER_ORDER):
    h = bar.get_height()
    ax_e.text(bar.get_x() + bar.get_width()/2, h + 0.01,
              f"{h*100:.0f}%", ha="center", va="bottom",
              fontsize=8, fontweight="bold",
              color=TIER_CONFIG[tier]["color"])


# ── Panel F: Per-tier accuracy across all layers ───────────────────────────
ax_f = fig.add_subplot(gs[3, 1])

for tier in TIER_ORDER:
    tier_slice = tier_acc_df[tier_acc_df["tier"] == tier]
    ax_f.plot(
        tier_slice["layer"],
        tier_slice["accuracy"],
        color     = TIER_CONFIG[tier]["color"],
        linewidth = 1.8,
        marker    = "o",
        markersize= 3,
        label     = TIER_CONFIG[tier]["label"],
        alpha     = 0.85,
    )

ax_f.axhline(1/6, color="black", linewidth=1,
             linestyle=":", alpha=0.5)
ax_f.set_xlabel("Layer", fontsize=10)
ax_f.set_ylabel("Per-tier accuracy", fontsize=10)
ax_f.set_title(
    "Panel F — Per-Tier Accuracy Across All Layers\n"
    "Do T1 and T6 diverge first? At which layer?",
    fontsize=10, fontweight="bold"
)
ax_f.set_ylim(0, 1)
ax_f.set_xticks(layers[::2])
ax_f.legend(fontsize=7, loc="upper left")
ax_f.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda y, _: f"{y*100:.0f}%")
)

plt.savefig("probe_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: probe_analysis.png")


# ── Confusion matrix figure ────────────────────────────────────────────────
fig_cm, ax_cm = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(
    confusion_matrix = cm,
    display_labels   = TIER_ORDER,
)
disp.plot(ax=ax_cm, cmap="Blues",
          colorbar=True, values_format="d")
ax_cm.set_title(
    f"Confusion Matrix — 6-class Probe at Peak Layer {best_layer}\n"
    f"Aggregated across {N_FOLDS} CV folds  |  "
    f"n={len(all_y_true)} test predictions",
    fontsize=11, fontweight="bold"
)
# Add tier labels on axes
ax_cm.set_xticklabels([TIER_CONFIG[t]["label"] for t in TIER_ORDER],
                       rotation=30, ha="right", fontsize=8)
ax_cm.set_yticklabels([TIER_CONFIG[t]["label"] for t in TIER_ORDER],
                       fontsize=8)
plt.tight_layout()
plt.savefig("probe_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: probe_confusion_matrix.png")


# ══════════════════════════════════════════════════════════════════════════════
# 12. SAVE RESULTS
# ══════════════════════════════════════════════════════════════════════════════

results_6class.to_csv("probe_6class_results.csv",    index=False)
results_binary.to_csv("probe_binary_results.csv",    index=False)
gen_welfare_to_neutral.to_csv("probe_gen_w2n.csv",   index=False)
gen_welfare_to_science.to_csv("probe_gen_w2s.csv",   index=False)
gen_neutral_to_welfare.to_csv("probe_gen_n2w.csv",   index=False)
tier_acc_df.to_csv("probe_per_tier_accuracy.csv",    index=False)

shuffle_df = pd.DataFrame(shuffle_accs,
                           columns=[f"layer_{l}" for l in layers])
shuffle_df.to_csv("probe_shuffle_null.csv", index=False)

print("\n── Saved files ──")
for f in ["probe_6class_results.csv", "probe_binary_results.csv",
          "probe_gen_w2n.csv", "probe_gen_w2s.csv", "probe_gen_n2w.csv",
          "probe_per_tier_accuracy.csv", "probe_shuffle_null.csv",
          "probe_analysis.png", "probe_confusion_matrix.png"]:
    size = Path(f).stat().st_size / 1e3 if Path(f).exists() else 0
    print(f"  {f:<40} {size:.0f} KB")


# ══════════════════════════════════════════════════════════════════════════════
# 13. RESULTS SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═"*60)
print("  PROBE RESULTS SUMMARY")
print("═"*60)
print(f"""
  6-class probe
    Peak accuracy    : {results_6class['mean_acc'].max():.3f}
    Peak layer       : {results_6class['mean_acc'].idxmax()}
    Chance baseline  : {1/6:.3f}  (16.7%)
    Above chance by  : {results_6class['mean_acc'].max() - 1/6:.3f}

  Binary probe (sentient vs non-sentient)
    Peak accuracy    : {results_binary['mean_acc'].max():.3f}
    Peak layer       : {results_binary['mean_acc'].idxmax()}
    Chance baseline  : 0.500  (50.0%)
    Above chance by  : {results_binary['mean_acc'].max() - 0.5:.3f}

  Sanity check (shuffled labels)
    Null mean acc    : {shuffle_mean.mean():.3f} ± {shuffle_mean.std():.3f}
    Real peak        : {results_6class['mean_acc'].max():.3f}
    Leakage detected : {'YES — INVESTIGATE' if results_6class['mean_acc'].max() <= shuffle_95.max() else 'No'}

  Generalisation (welfare → neutral)
    Peak test acc    : {gen_welfare_to_neutral['test_acc'].max():.3f}
    At layer         : {gen_welfare_to_neutral['test_acc'].idxmax()}
    Drop from in-dist: {results_6class['mean_acc'].max() - gen_welfare_to_neutral['test_acc'].max():+.3f}
    Interpretation   : {'Gradient generalises — likely entity-level encoding'
                        if gen_welfare_to_neutral['test_acc'].max() > 1/6 + 0.1
                        else 'Gradient does NOT generalise — framing-dependent'}

  Statistical significance (peak layer vs chance)
    6-class: t={t_stat:.3f}  p={p_val:.4f}  {'*' if p_val < 0.05 else 'ns'}
    Binary:  t={t_b:.3f}  p={p_b:.4f}  {'*' if p_b < 0.05 else 'ns'}

  Peak layer for activation patching (Week 4)
    → Layer {best_layer}
""")

print("✓  Probe analysis complete ")

What Each Panel Tells You  ^^

Panel A is the headline result. The real probe curve should rise well above the shuffled null band — if it does, sentience tier information is linearly decodable from the residual stream. The gap between real and null is your effect size.

Panel B is your most scientifically defensible finding. Binary sentient/non-sentient is a cleaner question than 6-class and less susceptible to arbitrary tier boundary choices.

Panel D is the key interpretive panel. Three outcomes are possible — each means something different:

- PatternInterpretationWelfare→Neutral stays highGradient is in the entity representation itself — template-independentWelfare→Neutral drops to chance
- Gradient is framing-dependent — only present in welfare language
- Neutral→Welfare stays highNeutral sentences already carry the signal — strongest finding

Panel F shows when each tier becomes discriminable. If T1 and T6 diverge at earlier layers than the middle tiers, the model encodes the extremes of the gradient first — a finding worth discussing in the paper.

The confusion matrix reveals systematic confusions. If T4 (welfare-evidenced invertebrates) is regularly confused with T3 (vertebrates) rather than T5 (minimal-evidence invertebrates), that suggests the model's implicit ranking tracks scientific sentience evidence more than simple taxonomy

* Compute mean activation vector per entity (averaged across all sentence templates) 

* Build 30×30 pairwise cosine similarity matrix 
 Build two hypothesis matrices:  

    *Sentience-ordered: similarity proportional to shared tier proximity 

    * Taxonomic: similarity based on phylogenetic distance (use a simple lookup, not a full phylogeny tool) 

* Compute Spearman correlation between model similarity matrix and each hypothesis matrix 

* Key question: does the model's implicit ordering track sentience intuitions beyond what taxonomy alone predicts? Octopus is the critical test case here — phylogenetically distant from humans but arguably high sentience

In [ ]:
import numpy as np
import pandas as pd
import h5py
import json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import cosine, squareform
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.preprocessing import StandardScaler
from itertools import combinations
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')


# ══════════════════════════════════════════════════════════════════════════════
# 1. CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

HDF5_PATH   = "activations_1.4b.h5"
N_LAYERS    = 24
RANDOM_SEED = 42

TIER_ORDER = ["T1","T2","T3","T4","T5","T6"]
TIER_CONFIG = {
    "T1": {"label": "Humans",              "color": "#08306b"},
    "T2": {"label": "Mammals",             "color": "#2171b5"},
    "T3": {"label": "Vertebrates",         "color": "#6baed6"},
    "T4": {"label": "Invertebrates (w+)",  "color": "#fd8d3c"},
    "T5": {"label": "Invertebrates (min)", "color": "#d7301f"},
    "T6": {"label": "Non-sentient",        "color": "#bdbdbd"},
}

print("Representational Similarity Analysis")
print("="*60)


# ══════════════════════════════════════════════════════════════════════════════
# 2. ENTITY TAXONOMY
# ══════════════════════════════════════════════════════════════════════════════

# All 60 entities with tier and full metadata
ENTITY_META = {
    # ── T1: Humans ─────────────────────────────────────────────────────────
    "child":         {"tier":"T1","sentience_rank":1.0,"tax_group":"primate",    "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "infant":        {"tier":"T1","sentience_rank":1.0,"tax_group":"primate",    "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "elderly person":{"tier":"T1","sentience_rank":1.0,"tax_group":"primate",    "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "worker":        {"tier":"T1","sentience_rank":1.0,"tax_group":"primate",    "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "patient":       {"tier":"T1","sentience_rank":1.0,"tax_group":"primate",    "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "prisoner":      {"tier":"T1","sentience_rank":1.0,"tax_group":"primate",    "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "refugee":       {"tier":"T1","sentience_rank":1.0,"tax_group":"primate",    "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "soldier":       {"tier":"T1","sentience_rank":1.0,"tax_group":"primate",    "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "teenager":      {"tier":"T1","sentience_rank":1.0,"tax_group":"primate",    "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "woman":         {"tier":"T1","sentience_rank":1.0,"tax_group":"primate",    "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    # ── T2: Mammals ────────────────────────────────────────────────────────
    "dog":           {"tier":"T2","sentience_rank":0.82,"tax_group":"carnivore", "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "chimpanzee":    {"tier":"T2","sentience_rank":0.90,"tax_group":"primate",   "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "pig":           {"tier":"T2","sentience_rank":0.80,"tax_group":"ungulate",  "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "cow":           {"tier":"T2","sentience_rank":0.75,"tax_group":"ungulate",  "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "dolphin":       {"tier":"T2","sentience_rank":0.92,"tax_group":"cetacean",  "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "elephant":      {"tier":"T2","sentience_rank":0.88,"tax_group":"proboscid", "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "rabbit":        {"tier":"T2","sentience_rank":0.72,"tax_group":"lagomorph", "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "rat":           {"tier":"T2","sentience_rank":0.70,"tax_group":"rodent",    "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "horse":         {"tier":"T2","sentience_rank":0.78,"tax_group":"ungulate",  "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    "whale":         {"tier":"T2","sentience_rank":0.89,"tax_group":"cetacean",  "tax_class":"mammalia",     "phylum":"chordata",      "kingdom":"animalia"},
    # ── T3: Non-mammal vertebrates ──────────────────────────────────────────
    "salmon":        {"tier":"T3","sentience_rank":0.55,"tax_group":"fish",      "tax_class":"actinopterygii","phylum":"chordata",     "kingdom":"animalia"},
    "chicken":       {"tier":"T3","sentience_rank":0.60,"tax_group":"bird",      "tax_class":"aves",          "phylum":"chordata",     "kingdom":"animalia"},
    "frog":          {"tier":"T3","sentience_rank":0.52,"tax_group":"amphibian", "tax_class":"amphibia",      "phylum":"chordata",     "kingdom":"animalia"},
    "zebrafish":     {"tier":"T3","sentience_rank":0.48,"tax_group":"fish",      "tax_class":"actinopterygii","phylum":"chordata",     "kingdom":"animalia"},
    "crow":          {"tier":"T3","sentience_rank":0.70,"tax_group":"bird",      "tax_class":"aves",          "phylum":"chordata",     "kingdom":"animalia"},
    "snake":         {"tier":"T3","sentience_rank":0.45,"tax_group":"reptile",   "tax_class":"reptilia",      "phylum":"chordata",     "kingdom":"animalia"},
    "tuna":          {"tier":"T3","sentience_rank":0.50,"tax_group":"fish",      "tax_class":"actinopterygii","phylum":"chordata",     "kingdom":"animalia"},
    "gecko":         {"tier":"T3","sentience_rank":0.42,"tax_group":"reptile",   "tax_class":"reptilia",      "phylum":"chordata",     "kingdom":"animalia"},
    "trout":         {"tier":"T3","sentience_rank":0.53,"tax_group":"fish",      "tax_class":"actinopterygii","phylum":"chordata",     "kingdom":"animalia"},
    "pigeon":        {"tier":"T3","sentience_rank":0.58,"tax_group":"bird",      "tax_class":"aves",          "phylum":"chordata",     "kingdom":"animalia"},
    # ── T4: Welfare-evidenced invertebrates ─────────────────────────────────
    "octopus":       {"tier":"T4","sentience_rank":0.65,"tax_group":"cephalopod","tax_class":"cephalopoda",  "phylum":"mollusca",      "kingdom":"animalia"},
    "crab":          {"tier":"T4","sentience_rank":0.45,"tax_group":"crustacean","tax_class":"malacostraca",  "phylum":"arthropoda",    "kingdom":"animalia"},
    "bee":           {"tier":"T4","sentience_rank":0.40,"tax_group":"insect",    "tax_class":"insecta",       "phylum":"arthropoda",    "kingdom":"animalia"},
    "shrimp":        {"tier":"T4","sentience_rank":0.38,"tax_group":"crustacean","tax_class":"malacostraca",  "phylum":"arthropoda",    "kingdom":"animalia"},
    "lobster":       {"tier":"T4","sentience_rank":0.42,"tax_group":"crustacean","tax_class":"malacostraca",  "phylum":"arthropoda",    "kingdom":"animalia"},
    "squid":         {"tier":"T4","sentience_rank":0.50,"tax_group":"cephalopod","tax_class":"cephalopoda",   "phylum":"mollusca",      "kingdom":"animalia"},
    "crayfish":      {"tier":"T4","sentience_rank":0.40,"tax_group":"crustacean","tax_class":"malacostraca",  "phylum":"arthropoda",    "kingdom":"animalia"},
    "bumblebee":     {"tier":"T4","sentience_rank":0.42,"tax_group":"insect",    "tax_class":"insecta",       "phylum":"arthropoda",    "kingdom":"animalia"},
    "mussel":        {"tier":"T4","sentience_rank":0.20,"tax_group":"bivalve",   "tax_class":"bivalvia",      "phylum":"mollusca",      "kingdom":"animalia"},
    "mantis shrimp": {"tier":"T4","sentience_rank":0.48,"tax_group":"crustacean","tax_class":"malacostraca",  "phylum":"arthropoda",    "kingdom":"animalia"},
    # ── T5: Minimal-evidence invertebrates ──────────────────────────────────
    "ant":           {"tier":"T5","sentience_rank":0.25,"tax_group":"insect",    "tax_class":"insecta",       "phylum":"arthropoda",    "kingdom":"animalia"},
    "fly":           {"tier":"T5","sentience_rank":0.22,"tax_group":"insect",    "tax_class":"insecta",       "phylum":"arthropoda",    "kingdom":"animalia"},
    "earthworm":     {"tier":"T5","sentience_rank":0.18,"tax_group":"annelid",   "tax_class":"oligochaeta",   "phylum":"annelida",      "kingdom":"animalia"},
    "moth":          {"tier":"T5","sentience_rank":0.22,"tax_group":"insect",    "tax_class":"insecta",       "phylum":"arthropoda",    "kingdom":"animalia"},
    "beetle":        {"tier":"T5","sentience_rank":0.20,"tax_group":"insect",    "tax_class":"insecta",       "phylum":"arthropoda",    "kingdom":"animalia"},
    "aphid":         {"tier":"T5","sentience_rank":0.12,"tax_group":"insect",    "tax_class":"insecta",       "phylum":"arthropoda",    "kingdom":"animalia"},
    "nematode":      {"tier":"T5","sentience_rank":0.08,"tax_group":"nematode",  "tax_class":"chromadorea",   "phylum":"nematoda",      "kingdom":"animalia"},
    "cockroach":     {"tier":"T5","sentience_rank":0.20,"tax_group":"insect",    "tax_class":"insecta",       "phylum":"arthropoda",    "kingdom":"animalia"},
    "mite":          {"tier":"T5","sentience_rank":0.10,"tax_group":"arachnid",  "tax_class":"arachnida",     "phylum":"arthropoda",    "kingdom":"animalia"},
    "woodlouse":     {"tier":"T5","sentience_rank":0.15,"tax_group":"crustacean","tax_class":"malacostraca",  "phylum":"arthropoda",    "kingdom":"animalia"},
    # ── T6: Non-sentient controls ────────────────────────────────────────────
    "oak tree":      {"tier":"T6","sentience_rank":0.0, "tax_group":"plant",     "tax_class":"magnoliopsida", "phylum":"tracheophyta",  "kingdom":"plantae"},
    "mushroom":      {"tier":"T6","sentience_rank":0.0, "tax_group":"fungus",    "tax_class":"agaricomycetes","phylum":"basidiomycota", "kingdom":"fungi"},
    "bacterium":     {"tier":"T6","sentience_rank":0.0, "tax_group":"bacteria",  "tax_class":"bacteria",      "phylum":"proteobacteria","kingdom":"bacteria"},
    "moss":          {"tier":"T6","sentience_rank":0.0, "tax_group":"plant",     "tax_class":"bryopsida",     "phylum":"bryophyta",     "kingdom":"plantae"},
    "seaweed":       {"tier":"T6","sentience_rank":0.0, "tax_group":"algae",     "tax_class":"phaeophyceae",  "phylum":"ochrophyta",    "kingdom":"chromista"},
    "rock":          {"tier":"T6","sentience_rank":0.0, "tax_group":"mineral",   "tax_class":"inanimate",     "phylum":"inanimate",     "kingdom":"inanimate"},
    "table":         {"tier":"T6","sentience_rank":0.0, "tax_group":"artifact",  "tax_class":"inanimate",     "phylum":"inanimate",     "kingdom":"inanimate"},
    "cloud":         {"tier":"T6","sentience_rank":0.0, "tax_group":"weather",   "tax_class":"inanimate",     "phylum":"inanimate",     "kingdom":"inanimate"},
    "river":         {"tier":"T6","sentience_rank":0.0, "tax_group":"geography", "tax_class":"inanimate",     "phylum":"inanimate",     "kingdom":"inanimate"},
    "statue":        {"tier":"T6","sentience_rank":0.0, "tax_group":"artifact",  "tax_class":"inanimate",     "phylum":"inanimate",     "kingdom":"inanimate"},
}

entity_list = sorted(ENTITY_META.keys())
N_ENTITIES  = len(entity_list)
print(f"Entities: {N_ENTITIES}")


# ══════════════════════════════════════════════════════════════════════════════
# 3. LOAD ENTITY MEAN VECTORS
# ══════════════════════════════════════════════════════════════════════════════

def load_entity_means(hdf5_path: str,
                       entity_list: list,
                       n_layers: int) -> np.ndarray:
    """
    For each entity, average residual stream vectors across all
    sentence templates. Returns (n_entities, n_layers, d_model).
    """
    entity_sums   = {e: None for e in entity_list}
    entity_counts = {e: 0    for e in entity_list}

    with h5py.File(hdf5_path, "r") as f:
        for sid in f.keys():
            entity = f[sid].attrs["entity"]
            if entity not in entity_sums:
                continue
            resid = f[sid]["resid_post"][:].astype(np.float32)  # (L, D)
            if entity_sums[entity] is None:
                entity_sums[entity]   = resid.copy()
            else:
                entity_sums[entity]  += resid
            entity_counts[entity] += 1

    # Stack into matrix
    rows = []
    for entity in entity_list:
        n = entity_counts[entity]
        if n == 0:
            raise ValueError(f"Entity '{entity}' not found in HDF5")
        rows.append(entity_sums[entity] / n)   # (L, D)

    matrix = np.stack(rows)   # (N_entities, L, D)
    print(f"  Entity mean matrix: {matrix.shape}")
    print(f"  Templates averaged per entity: "
          f"{np.mean(list(entity_counts.values())):.1f}")
    return matrix


print("\nLoading entity mean vectors...")
entity_matrix = load_entity_means(HDF5_PATH, entity_list, N_LAYERS)
# (60, 24, 2048)


# ══════════════════════════════════════════════════════════════════════════════
# 4. MODEL SIMILARITY MATRIX
# ══════════════════════════════════════════════════════════════════════════════

def cosine_similarity_matrix(vectors: np.ndarray) -> np.ndarray:
    """
    Compute pairwise cosine similarity for (n, d) matrix.
    Returns (n, n) similarity matrix in [-1, 1].
    """
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1e-9, norms)
    normed = vectors / norms
    return normed @ normed.T


def build_model_similarity_matrices(entity_matrix: np.ndarray,
                                     layers: list) -> dict:
    """
    Build cosine similarity matrix at each requested layer.
    Returns dict: layer → (N_entities, N_entities) array.
    """
    results = {}
    for layer in layers:
        vecs = entity_matrix[:, layer, :]   # (60, D)
        results[layer] = cosine_similarity_matrix(vecs)
    return results


# Compute for all layers + sample layers for visualisation
all_layers    = list(range(N_LAYERS))
sample_layers = [0, 4, 8, 12, 16, 20, N_LAYERS-1]

print("\nBuilding model similarity matrices...")
model_sim = build_model_similarity_matrices(entity_matrix, all_layers)


# ══════════════════════════════════════════════════════════════════════════════
# 5. HYPOTHESIS MATRICES
# ══════════════════════════════════════════════════════════════════════════════

# ── 5a. Sentience-ordered hypothesis ──────────────────────────────────────
# Similarity = 1 - |sentience_rank_i - sentience_rank_j|
# Entities with similar sentience ranks should have similar representations

def build_sentience_matrix(entity_list: list,
                            entity_meta: dict) -> np.ndarray:
    """
    Hypothesis 1: representational similarity should track sentience rank.
    sim(i,j) = 1 - |rank_i - rank_j|
    """
    n   = len(entity_list)
    mat = np.zeros((n, n))
    for i, e_i in enumerate(entity_list):
        for j, e_j in enumerate(entity_list):
            r_i = entity_meta[e_i]["sentience_rank"]
            r_j = entity_meta[e_j]["sentience_rank"]
            mat[i, j] = 1.0 - abs(r_i - r_j)
    return mat


# ── 5b. Taxonomic hypothesis ───────────────────────────────────────────────
# Similarity based on taxonomic proximity using a 6-level hierarchy
# kingdom → phylum → class → order(group) → species

# Define taxonomic distance levels — lower = more similar
TAXONOMIC_DISTANCE = {
    # Same tax_group → distance 1
    # Same tax_class → distance 2
    # Same phylum    → distance 3
    # Same kingdom   → distance 4
    # Different kingdom → distance 5
    # One or both inanimate → distance 6
}

def taxonomic_distance(e_i: dict, e_j: dict) -> float:
    """
    Compute taxonomic distance between two entities.
    Uses a 6-level hierarchy: group < class < phylum < kingdom < cross-kingdom < inanimate.
    Returns integer distance 1–6; converted to similarity as 1 - (dist-1)/5.
    """
    # Both inanimate → maximally distant from everything
    if (e_i["tax_class"] == "inanimate" or
        e_j["tax_class"] == "inanimate"):
        # Two inanimates are similar to each other
        if (e_i["tax_class"] == "inanimate" and
            e_j["tax_class"] == "inanimate"):
            return 1   # inanimates cluster together
        return 6   # inanimate vs living → max distance

    if e_i["tax_group"] == e_j["tax_group"]:
        return 1
    if e_i["tax_class"] == e_j["tax_class"]:
        return 2
    if e_i["phylum"] == e_j["phylum"]:
        return 3
    if e_i["kingdom"] == e_j["kingdom"]:
        return 4
    return 5   # different kingdoms (animal vs plant vs fungi vs bacteria)


def build_taxonomic_matrix(entity_list: list,
                            entity_meta: dict) -> np.ndarray:
    """
    Hypothesis 2: representational similarity should track taxonomic proximity.
    sim(i,j) = 1 - (taxonomic_distance(i,j) - 1) / 5
    """
    n   = len(entity_list)
    mat = np.zeros((n, n))
    for i, e_i in enumerate(entity_list):
        for j, e_j in enumerate(entity_list):
            dist     = taxonomic_distance(entity_meta[e_i], entity_meta[e_j])
            mat[i,j] = 1.0 - (dist - 1) / 5.0
    return mat


# ── 5c. Tier-proximity hypothesis (coarser, for comparison) ───────────────
def build_tier_proximity_matrix(entity_list: list,
                                 entity_meta: dict) -> np.ndarray:
    """
    Hypothesis 3: similarity based on tier adjacency only (ignores within-tier detail).
    |tier_i - tier_j| mapped to similarity.
    """
    tier_rank = {"T1":1,"T2":2,"T3":3,"T4":4,"T5":5,"T6":6}
    n   = len(entity_list)
    mat = np.zeros((n, n))
    for i, e_i in enumerate(entity_list):
        for j, e_j in enumerate(entity_list):
            t_i = tier_rank[entity_meta[e_i]["tier"]]
            t_j = tier_rank[entity_meta[e_j]["tier"]]
            mat[i,j] = 1.0 - abs(t_i - t_j) / 5.0
    return mat


print("\nBuilding hypothesis matrices...")

H_sentience  = build_sentience_matrix(entity_list, ENTITY_META)
H_taxonomic  = build_taxonomic_matrix(entity_list, ENTITY_META)
H_tier       = build_tier_proximity_matrix(entity_list, ENTITY_META)

print(f"  Sentience matrix: {H_sentience.shape}  "
      f"range [{H_sentience.min():.2f}, {H_sentience.max():.2f}]")
print(f"  Taxonomic matrix: {H_taxonomic.shape}  "
      f"range [{H_taxonomic.min():.2f}, {H_taxonomic.max():.2f}]")
print(f"  Tier matrix:      {H_tier.shape}  "
      f"range [{H_tier.min():.2f}, {H_tier.max():.2f}]")

# How correlated are the two hypotheses with each other?
# (if r > 0.9, they're not distinguishable)
tri_idx = np.triu_indices(N_ENTITIES, k=1)
r_hyp, p_hyp = stats.spearmanr(
    H_sentience[tri_idx], H_taxonomic[tri_idx]
)
print(f"\n  Hypothesis collinearity check:")
print(f"  Spearman r (sentience vs taxonomic): {r_hyp:.3f}  p={p_hyp:.4f}")
print(f"  {'WARNING: hypotheses nearly collinear — hard to distinguish' if abs(r_hyp) > 0.85 else 'OK: hypotheses are distinguishable'}")


# ══════════════════════════════════════════════════════════════════════════════
# 6. RSA — SPEARMAN CORRELATIONS ACROSS LAYERS
# ══════════════════════════════════════════════════════════════════════════════

def rsa_layer(model_sim_mat: np.ndarray,
               hypothesis_mat: np.ndarray) -> tuple[float, float]:
    """
    Compute Spearman correlation between upper triangles of
    model similarity matrix and hypothesis matrix.
    Returns (r, p).
    """
    tri = np.triu_indices(model_sim_mat.shape[0], k=1)
    return stats.spearmanr(model_sim_mat[tri], hypothesis_mat[tri])


print("\nRunning RSA across all layers...")

rsa_rows = []
for layer in all_layers:
    M = model_sim[layer]

    r_sent, p_sent = rsa_layer(M, H_sentience)
    r_tax,  p_tax  = rsa_layer(M, H_taxonomic)
    r_tier, p_tier = rsa_layer(M, H_tier)

    rsa_rows.append({
        "layer":        layer,
        "r_sentience":  r_sent,
        "p_sentience":  p_sent,
        "r_taxonomic":  r_tax,
        "p_taxonomic":  p_tax,
        "r_tier":       r_tier,
        "p_tier":       p_tier,
        "r_sent_unique": r_sent - r_tax,   # sentience signal beyond taxonomy
    })

rsa_df = pd.DataFrame(rsa_rows)

# Bonferroni correction (3 hypotheses × 24 layers = 72 tests)
n_tests = 3 * N_LAYERS
rsa_df["p_sent_corrected"] = np.minimum(rsa_df["p_sentience"] * n_tests, 1.0)
rsa_df["p_tax_corrected"]  = np.minimum(rsa_df["p_taxonomic"] * n_tests, 1.0)
rsa_df["p_tier_corrected"] = np.minimum(rsa_df["p_tier"]      * n_tests, 1.0)

print(f"\n  RSA results across all layers:")
print(f"  {'Layer':>5}  {'r_sent':>8}  {'r_tax':>8}  {'r_tier':>8}  "
      f"{'sent-tax':>9}  {'sig_sent':>9}")
for _, row in rsa_df.iterrows():
    sig = "*" if row["p_sent_corrected"] < 0.05 else " "
    print(f"  {int(row['layer']):>5}  "
          f"{row['r_sentience']:>+8.3f}  "
          f"{row['r_taxonomic']:>+8.3f}  "
          f"{row['r_tier']:>+8.3f}  "
          f"{row['r_sent_unique']:>+9.3f}  "
          f"{sig:>9}")

peak_sent_layer = int(rsa_df["r_sentience"].idxmax())
peak_tax_layer  = int(rsa_df["r_taxonomic"].idxmax())
print(f"\n  Peak sentience RSA: r={rsa_df['r_sentience'].max():.3f} "
      f"at layer {peak_sent_layer}")
print(f"  Peak taxonomic RSA: r={rsa_df['r_taxonomic'].max():.3f} "
      f"at layer {peak_tax_layer}")
print(f"  Max sentience-beyond-taxonomy: "
      f"r={rsa_df['r_sent_unique'].max():.3f} "
      f"at layer {int(rsa_df['r_sent_unique'].idxmax())}")


# ══════════════════════════════════════════════════════════════════════════════
# 7. OCTOPUS SPOTLIGHT ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

print("\n── Octopus spotlight analysis ──")
print("  Key question: does the model place octopus closer to vertebrates")
print("  (sentience-consistent) or closer to insects (taxonomy-consistent)?")

SPOTLIGHT_ENTITIES = [
    "child",          # T1 anchor
    "chimpanzee",     # T2 — high sentience mammal
    "dolphin",        # T2 — high sentience mammal
    "crow",           # T3 — cognitively sophisticated bird
    "octopus",        # T4 — THE critical test case
    "squid",          # T4 — cephalopod, less studied than octopus
    "crab",           # T4 — welfare-evidenced crustacean
    "bee",            # T4 — welfare-evidenced insect
    "ant",            # T5 — minimal evidence insect
    "earthworm",      # T5 — minimal evidence
    "oak tree",       # T6 — living but non-sentient
    "rock",           # T6 — inanimate
]

spotlight_idx = {e: entity_list.index(e) for e in SPOTLIGHT_ENTITIES
                 if e in entity_list}

def octopus_analysis(model_sim: dict,
                      entity_list: list,
                      layers: list) -> pd.DataFrame:
    """
    For each layer, compute octopus's similarity to:
    (a) T1 humans (sentience-consistent anchor)
    (b) T2 mammals (high sentience)
    (c) T3 vertebrates (taxonomic neighbours by sentience intuition)
    (d) T5 insects (taxonomic neighbours by phylogeny)
    (e) T6 controls (baseline)

    If octopus is sentience-encoded: (a,b,c) > (d,e)
    If octopus is taxonomy-encoded:  (d) > (b,c)
    """
    oct_idx  = entity_list.index("octopus")
    rows     = []

    tier_indices = {tier: [entity_list.index(e)
                            for e, m in ENTITY_META.items()
                            if m["tier"] == tier]
                    for tier in TIER_ORDER}

    for layer in layers:
        M = model_sim[layer]
        oct_sims = M[oct_idx]

        row = {"layer": layer}
        for tier in TIER_ORDER:
            idxs = [i for i in tier_indices[tier] if i != oct_idx]
            row[f"sim_to_{tier}"] = oct_sims[idxs].mean()

        # Key contrast: does octopus resemble T2 (mammals) more than T5 (insects)?
        row["sentience_pull"] = row["sim_to_T2"] - row["sim_to_T5"]
        # Positive = octopus pulled toward high-sentience entities (sentience encoding)
        # Negative = octopus pulled toward phylogenetic neighbours (taxonomy encoding)

        # Also track similarity to squid (same phylum, control)
        squid_idx = entity_list.index("squid")
        row["sim_to_squid"] = M[oct_idx, squid_idx]

        rows.append(row)

    return pd.DataFrame(rows)


oct_df = octopus_analysis(model_sim, entity_list, all_layers)

print(f"\n  Octopus similarity profile (final layer):")
final_oct = oct_df.iloc[-1]
for tier in TIER_ORDER:
    print(f"    sim to {tier}: {final_oct[f'sim_to_{tier}']:+.4f}  "
          f"({TIER_CONFIG[tier]['label']})")
print(f"\n  Sentience pull index (T2 - T5): {final_oct['sentience_pull']:+.4f}")
print(f"  {'Octopus pulled toward high-sentience (sentience encoding)' if final_oct['sentience_pull'] > 0 else 'Octopus pulled toward taxonomic neighbours (taxonomy encoding)'}")


# ══════════════════════════════════════════════════════════════════════════════
# 8. PARTIAL CORRELATION — SENTIENCE BEYOND TAXONOMY
# ══════════════════════════════════════════════════════════════════════════════

print("\n── Partial correlation: sentience controlling for taxonomy ──")
print("  Does sentience predict model similarity beyond what taxonomy explains?")

def partial_spearman(x: np.ndarray,
                      y: np.ndarray,
                      z: np.ndarray) -> tuple[float, float]:
    """
    Compute partial Spearman correlation of x and y controlling for z.
    Uses the formula: r_xy.z = (r_xy - r_xz*r_yz) / sqrt((1-r_xz²)(1-r_yz²))
    Returns (partial_r, approximate_p_value).
    """
    r_xy, _ = stats.spearmanr(x, y)
    r_xz, _ = stats.spearmanr(x, z)
    r_yz, _ = stats.spearmanr(y, z)

    numerator   = r_xy - r_xz * r_yz
    denominator = np.sqrt((1 - r_xz**2) * (1 - r_yz**2))

    if denominator < 1e-10:
        return 0.0, 1.0

    partial_r = numerator / denominator
    partial_r = np.clip(partial_r, -1, 1)

    # Approximate t-test for partial correlation
    n  = len(x)
    t  = partial_r * np.sqrt((n - 3) / (1 - partial_r**2 + 1e-10))
    p  = 2 * (1 - stats.t.cdf(abs(t), df=n-3))

    return partial_r, p


tri = np.triu_indices(N_ENTITIES, k=1)
model_flat    = None   # filled per layer
sent_flat     = H_sentience[tri]
tax_flat      = H_taxonomic[tri]
tier_flat     = H_tier[tri]

partial_rows = []
for layer in all_layers:
    model_flat = model_sim[layer][tri]

    # Sentience controlling for taxonomy
    pr_sent_tax, p_sent_tax = partial_spearman(
        model_flat, sent_flat, tax_flat
    )
    # Taxonomy controlling for sentience
    pr_tax_sent, p_tax_sent = partial_spearman(
        model_flat, tax_flat, sent_flat
    )

    partial_rows.append({
        "layer":                layer,
        "partial_r_sent_tax":   pr_sent_tax,
        "p_sent_tax":           p_sent_tax,
        "partial_r_tax_sent":   pr_tax_sent,
        "p_tax_sent":           p_tax_sent,
    })

partial_df = pd.DataFrame(partial_rows)

print(f"\n  Layer-by-layer partial correlations (sample):")
print(f"  {'Layer':>5}  {'r(M,sent|tax)':>14}  {'p':>8}  "
      f"{'r(M,tax|sent)':>14}  {'p':>8}")
for _, row in partial_df[partial_df["layer"].isin(sample_layers)].iterrows():
    print(f"  {int(row['layer']):>5}  "
          f"{row['partial_r_sent_tax']:>+14.4f}  "
          f"{row['p_sent_tax']:>8.4f}  "
          f"{row['partial_r_tax_sent']:>+14.4f}  "
          f"{row['p_tax_sent']:>8.4f}")

peak_partial = partial_df["partial_r_sent_tax"].max()
peak_partial_layer = int(partial_df["partial_r_sent_tax"].idxmax())
print(f"\n  Peak partial r(sentience|taxonomy): "
      f"{peak_partial:+.4f} at layer {peak_partial_layer}")
print(f"  Interpretation: "
      f"{'Sentience predicts model similarity beyond taxonomy' if peak_partial > 0.1 else 'No sentience signal beyond taxonomy'}")


# ══════════════════════════════════════════════════════════════════════════════
# 9. MAIN FIGURES
# ══════════════════════════════════════════════════════════════════════════════

print("\nBuilding figures...")

# ── Custom colourmap for similarity matrices
sim_cmap = LinearSegmentedColormap.from_list(
    "sim_cmap",
    ["#08306b", "#4292c6", "#f7fbff", "#fdd0a2", "#d94801"],
    N=256
)
# Blue = dissimilar, white = neutral, orange/red = similar

tier_colors_list = [TIER_CONFIG[ENTITY_META[e]["tier"]]["color"]
                    for e in entity_list]

def annotated_heatmap(ax, mat, title, vmin=-1, vmax=1,
                       show_tier_bar=True, cmap=None):
    """Draw similarity matrix heatmap with tier colour bar on axes."""
    if cmap is None:
        cmap = sim_cmap
    im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax,
                   aspect="auto", interpolation="nearest")
    ax.set_title(title, fontsize=9, fontweight="bold", pad=4)
    ax.set_xticks([])
    ax.set_yticks([])

    # Tier colour bars on edges
    if show_tier_bar:
        bar_w = 0.012
        for i, color in enumerate(tier_colors_list):
            ax.add_patch(plt.Rectangle(
                (-0.5 - N_ENTITIES*bar_w, i - 0.5),
                N_ENTITIES * bar_w, 1,
                color=color, clip_on=False,
                transform=ax.transData
            ))

    return im


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1: RSA main results
# ══════════════════════════════════════════════════════════════════════════════

fig1 = plt.figure(figsize=(18, 20))
gs1  = gridspec.GridSpec(3, 3, figure=fig1,
                          hspace=0.45, wspace=0.35)
fig1.suptitle(
    "Representational Similarity Analysis — Sentience Salience Probe\n"
    f"Pythia-1.4b  |  {N_ENTITIES} entities  |  "
    f"All {N_LAYERS} layers",
    fontsize=13, fontweight="bold", y=0.99
)

# ── Panel A: RSA correlations across layers ────────────────────────────────
ax_a = fig1.add_subplot(gs1[0, :])

ax_a.fill_between(
    rsa_df["layer"],
    rsa_df["r_sentience"] - 0.02,
    rsa_df["r_sentience"] + 0.02,
    alpha=0.15, color="#08306b"
)
ax_a.plot(rsa_df["layer"], rsa_df["r_sentience"],
          color="#08306b", linewidth=2.5, marker="o",
          markersize=5, label="Sentience hypothesis")

ax_a.plot(rsa_df["layer"], rsa_df["r_taxonomic"],
          color="#2ca25f", linewidth=2.5, marker="s",
          markersize=5, label="Taxonomic hypothesis")

ax_a.plot(rsa_df["layer"], rsa_df["r_tier"],
          color="#fd8d3c", linewidth=2, marker="^",
          markersize=5, linestyle="--", label="Tier-proximity hypothesis")

ax_a.plot(rsa_df["layer"], rsa_df["r_sent_unique"],
          color="#d94801", linewidth=2, marker="D",
          markersize=4, linestyle=":",
          label="Sentience beyond taxonomy (Δr)")

ax_a.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.4)

# Mark significance (corrected)
sig_layers_sent = rsa_df[rsa_df["p_sent_corrected"] < 0.05]["layer"]
if len(sig_layers_sent) > 0:
    ax_a.scatter(
        sig_layers_sent,
        rsa_df.loc[rsa_df["p_sent_corrected"] < 0.05, "r_sentience"],
        marker="*", color="#08306b", s=120, zorder=6,
        label="Sentience sig. (Bonferroni p<0.05)"
    )

# Annotate peak
peak_r = rsa_df["r_sentience"].max()
ax_a.annotate(
    f"Peak r={peak_r:.3f}\nlayer {peak_sent_layer}",
    xy     = (peak_sent_layer, peak_r),
    xytext = (peak_sent_layer + 1.5, peak_r + 0.04),
    arrowprops = dict(arrowstyle="->", color="#08306b", lw=1.5),
    fontsize=9, color="#08306b", fontweight="bold"
)

ax_a.set_xlabel("Layer", fontsize=11)
ax_a.set_ylabel("Spearman r", fontsize=11)
ax_a.set_title(
    "Panel A — RSA: Model Similarity vs Hypothesis Matrices Across Layers\n"
    "Does the model's implicit entity ordering match sentience or taxonomy?",
    fontsize=10, fontweight="bold"
)
ax_a.legend(fontsize=9, loc="lower right", ncols=2)
ax_a.set_xticks(all_layers)
ax_a.set_ylim(-0.3, 0.8)


# ── Panel B: Partial correlations across layers ────────────────────────────
ax_b = fig1.add_subplot(gs1[1, :2])

ax_b.fill_between(
    partial_df["layer"],
    0, partial_df["partial_r_sent_tax"],
    alpha=0.2, color="#08306b"
)
ax_b.plot(partial_df["layer"], partial_df["partial_r_sent_tax"],
          color="#08306b", linewidth=2.5, marker="o",
          markersize=4,
          label="r(Model, Sentience | Taxonomy)")

ax_b.fill_between(
    partial_df["layer"],
    0, partial_df["partial_r_tax_sent"],
    alpha=0.2, color="#2ca25f"
)
ax_b.plot(partial_df["layer"], partial_df["partial_r_tax_sent"],
          color="#2ca25f", linewidth=2.5, marker="s",
          markersize=4,
          label="r(Model, Taxonomy | Sentience)")

ax_b.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax_b.axvline(peak_partial_layer, color="#08306b",
             linewidth=1, linestyle=":", alpha=0.6)

ax_b.set_xlabel("Layer", fontsize=10)
ax_b.set_ylabel("Partial Spearman r", fontsize=10)
ax_b.set_title(
    "Panel B — Partial Correlations\n"
    "Sentience effect controlling for taxonomy; taxonomy effect controlling for sentience",
    fontsize=9, fontweight="bold"
)
ax_b.legend(fontsize=9)
ax_b.set_xticks(all_layers[::2])


# ── Panel C: Octopus similarity profile across layers ─────────────────────
ax_c = fig1.add_subplot(gs1[1, 2])

tier_line_styles = {
    "T1": ("-",  "#08306b", "o"),
    "T2": ("-",  "#2171b5", "s"),
    "T3": ("-",  "#6baed6", "^"),
    "T4": ("--", "#fd8d3c", "D"),
    "T5": ("--", "#d7301f", "v"),
    "T6": (":",  "#bdbdbd", "x"),
}
for tier, (ls, color, marker) in tier_line_styles.items():
    ax_c.plot(oct_df["layer"],
              oct_df[f"sim_to_{tier}"],
              linestyle=ls, color=color, marker=marker,
              markersize=3, linewidth=1.6,
              label=TIER_CONFIG[tier]["label"])

ax_c.axhline(0, color="black", linewidth=0.5, linestyle="--", alpha=0.3)
ax_c.set_xlabel("Layer", fontsize=9)
ax_c.set_ylabel("Mean cosine similarity to tier", fontsize=9)
ax_c.set_title(
    "Panel C — Octopus Similarity\nto Each Tier Across Layers",
    fontsize=9, fontweight="bold"
)
ax_c.legend(fontsize=7, loc="lower right")
ax_c.set_xticks(all_layers[::4])


# ── Panel D: Similarity matrices at 3 key layers ──────────────────────────
key_layers = [0, peak_sent_layer, N_LAYERS-1]
key_labels = [f"Layer 0\n(early)", f"Layer {peak_sent_layer}\n(peak RSA)", f"Layer {N_LAYERS-1}\n(final)"]

for col_idx, (layer, label) in enumerate(zip(key_layers, key_labels)):
    ax = fig1.add_subplot(gs1[2, col_idx])
    im = annotated_heatmap(
        ax, model_sim[layer],
        title=label,
        show_tier_bar=False
    )
    plt.colorbar(im, ax=ax, shrink=0.7, label="Cosine sim")

    # Add tier boundary lines
    tier_sizes    = [10] * 6   # 10 entities per tier
    boundaries    = np.cumsum(tier_sizes)[:-1] - 0.5
    for b in boundaries:
        ax.axvline(b, color="white", linewidth=0.8, alpha=0.6)
        ax.axhline(b, color="white", linewidth=0.8, alpha=0.6)

    # Tier labels on diagonal
    tier_centers = [4.5, 14.5, 24.5, 34.5, 44.5, 54.5]
    for i, (tc, tier) in enumerate(zip(tier_centers, TIER_ORDER)):
        ax.text(tc, tc, tier,
                ha="center", va="center",
                fontsize=6.5, fontweight="bold",
                color=TIER_CONFIG[tier]["color"],
                bbox=dict(facecolor="white", alpha=0.6,
                           edgecolor="none", pad=1))

fig1.text(0.5, 0.25, "Panels D–F: 60×60 Cosine Similarity Matrix "
          "(entities sorted by tier; boundaries marked)",
          ha="center", fontsize=9, style="italic")

# Add tier colour legend for matrices
legend_patches = [
    mpatches.Patch(color=TIER_CONFIG[t]["color"],
                   label=TIER_CONFIG[t]["label"])
    for t in TIER_ORDER
]
fig1.legend(handles=legend_patches,
            loc="lower center", ncols=6,
            fontsize=8, framealpha=0.9,
            bbox_to_anchor=(0.5, 0.01))

plt.savefig("rsa_main.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rsa_main.png")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2: Hypothesis matrices + model comparison
# ══════════════════════════════════════════════════════════════════════════════

fig2, axes2 = plt.subplots(2, 3, figsize=(18, 12))
fig2.suptitle(
    "Hypothesis Matrices vs Model Similarity — Final Layer\n"
    f"30×30 shown for readability; full {N_ENTITIES}×{N_ENTITIES} used in RSA",
    fontsize=12, fontweight="bold"
)

titles = [
    "Model (final layer)",
    "Sentience hypothesis\n(sim = 1 - |rank_i - rank_j|)",
    "Taxonomic hypothesis\n(sim = 1 - tax_dist/5)",
    "Tier-proximity hypothesis",
    "Model − Sentience\n(residual)",
    "Model − Taxonomic\n(residual)",
]
matrices = [
    model_sim[N_LAYERS-1],
    H_sentience,
    H_taxonomic,
    H_tier,
    model_sim[N_LAYERS-1] - H_sentience,
    model_sim[N_LAYERS-1] - H_taxonomic,
]
vmins = [-1, 0, 0, 0, -1, -1]
vmaxs = [ 1, 1, 1, 1,  1,  1]

for ax, mat, title, vmin, vmax in zip(
        axes2.flatten(), matrices, titles, vmins, vmaxs):
    im = ax.imshow(mat, cmap=sim_cmap, vmin=vmin, vmax=vmax,
                   aspect="auto", interpolation="nearest")
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, shrink=0.7)

    boundaries = np.cumsum([10]*6)[:-1] - 0.5
    for b in boundaries:
        ax.axvline(b, color="white", linewidth=0.8, alpha=0.5)
        ax.axhline(b, color="white", linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.savefig("rsa_hypothesis_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rsa_hypothesis_matrices.png")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3: Spotlight — octopus and critical entities
# ══════════════════════════════════════════════════════════════════════════════

spotlight_ents = list(spotlight_idx.keys())
spot_idxs      = [entity_list.index(e) for e in spotlight_ents]

fig3, axes3 = plt.subplots(1, 2, figsize=(16, 8))
fig3.suptitle(
    "Spotlight Analysis — Octopus and Critical Entities\n"
    "Does the model encode octopus as sentience-similar to vertebrates, "
    "or phylogeny-similar to insects?",
    fontsize=12, fontweight="bold"
)

# ── Left: 12×12 spotlight similarity matrix (final layer)
spot_sim = model_sim[N_LAYERS-1][np.ix_(spot_idxs, spot_idxs)]
spot_colors = [TIER_CONFIG[ENTITY_META[e]["tier"]]["color"]
               for e in spotlight_ents]

im3 = axes3[0].imshow(spot_sim, cmap=sim_cmap, vmin=-1, vmax=1,
                        aspect="auto", interpolation="nearest")

axes3[0].set_xticks(range(len(spotlight_ents)))
axes3[0].set_yticks(range(len(spotlight_ents)))
axes3[0].set_xticklabels(spotlight_ents, rotation=45, ha="right", fontsize=9)
axes3[0].set_yticklabels(spotlight_ents, fontsize=9)

# Colour tick labels by tier
for tick, entity in zip(axes3[0].get_xticklabels(), spotlight_ents):
    tick.set_color(TIER_CONFIG[ENTITY_META[entity]["tier"]]["color"])
    tick.set_fontweight("bold")
for tick, entity in zip(axes3[0].get_yticklabels(), spotlight_ents):
    tick.set_color(TIER_CONFIG[ENTITY_META[entity]["tier"]]["color"])
    tick.set_fontweight("bold")

# Annotate octopus row
oct_spotlight_idx = spotlight_ents.index("octopus")
axes3[0].axhline(oct_spotlight_idx - 0.5, color="gold",
                  linewidth=2, linestyle="--")
axes3[0].axhline(oct_spotlight_idx + 0.5, color="gold",
                  linewidth=2, linestyle="--")
axes3[0].axvline(oct_spotlight_idx - 0.5, color="gold",
                  linewidth=2, linestyle="--")
axes3[0].axvline(oct_spotlight_idx + 0.5, color="gold",
                  linewidth=2, linestyle="--")

# Add values inside cells
for i in range(len(spotlight_ents)):
    for j in range(len(spotlight_ents)):
        val = spot_sim[i, j]
        axes3[0].text(j, i, f"{val:.2f}",
                       ha="center", va="center",
                       fontsize=7,
                       color="white" if abs(val) > 0.4 else "black")

plt.colorbar(im3, ax=axes3[0], shrink=0.7, label="Cosine similarity")
axes3[0].set_title(
    f"Spotlight Similarity Matrix (final layer {N_LAYERS-1})\n"
    "Gold border = octopus row/column",
    fontsize=10, fontweight="bold"
)

# ── Right: Octopus sentience pull across layers
ax_r = axes3[1]

ax_r.fill_between(oct_df["layer"], 0, oct_df["sentience_pull"],
                   alpha=0.25,
                   color="#08306b" if oct_df["sentience_pull"].mean() > 0
                   else "#d7301f")
ax_r.plot(oct_df["layer"], oct_df["sentience_pull"],
          color="#08306b", linewidth=2.5, marker="o",
          markersize=5, label="Sentience pull (sim_T2 − sim_T5)")
ax_r.axhline(0, color="black", linewidth=1.2, linestyle="--")

# Shade interpretation regions
ax_r.fill_between(oct_df["layer"], 0, 0.1,
                   alpha=0.06, color="#08306b",
                   label="Sentience-consistent region (>0)")
ax_r.fill_between(oct_df["layer"], -0.1, 0,
                   alpha=0.06, color="#d7301f",
                   label="Taxonomy-consistent region (<0)")

ax_r.set_xlabel("Layer", fontsize=11)
ax_r.set_ylabel("Sentience pull index\n(sim to mammals − sim to insects)",
                 fontsize=10)
ax_r.set_title(
    "Octopus Sentience Pull Index Across Layers\n"
    ">0 = octopus encoded like high-sentience mammals (sentience encoding)\n"
    "<0 = octopus encoded like phylogenetic insect neighbours (taxonomy encoding)",
    fontsize=9, fontweight="bold"
)
ax_r.legend(fontsize=9)
ax_r.set_xticks(all_layers)
ax_r.set_ylim(-0.3, 0.3)

# Annotate direction at final layer
final_pull = oct_df["sentience_pull"].iloc[-1]
direction  = ("sentience-\nconsistent" if final_pull > 0
              else "taxonomy-\nconsistent")
ax_r.annotate(
    f"Final layer:\n{final_pull:+.3f}\n({direction})",
    xy     = (N_LAYERS-1, final_pull),
    xytext = (N_LAYERS-4, final_pull + 0.08),
    arrowprops = dict(arrowstyle="->", color="black", lw=1.2),
    fontsize=9, fontweight="bold",
    color="#08306b" if final_pull > 0 else "#d7301f"
)

plt.tight_layout()
plt.savefig("rsa_octopus_spotlight.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rsa_octopus_spotlight.png")


# ══════════════════════════════════════════════════════════════════════════════
# 10. SAVE RESULTS
# ══════════════════════════════════════════════════════════════════════════════

rsa_df.to_csv("rsa_results.csv",          index=False)
partial_df.to_csv("rsa_partial.csv",      index=False)
oct_df.to_csv("rsa_octopus.csv",          index=False)

np.save("model_sim_final.npy",  model_sim[N_LAYERS-1])
np.save("H_sentience.npy",      H_sentience)
np.save("H_taxonomic.npy",      H_taxonomic)

print("\n── Saved files ──")
for fname in ["rsa_results.csv","rsa_partial.csv","rsa_octopus.csv",
              "model_sim_final.npy","H_sentience.npy","H_taxonomic.npy",
              "rsa_main.png","rsa_hypothesis_matrices.png",
              "rsa_octopus_spotlight.png"]:
    size = Path(fname).stat().st_size/1e3 if Path(fname).exists() else 0
    print(f"  {fname:<40} {size:.0f} KB")


# ══════════════════════════════════════════════════════════════════════════════
# 11. RESULTS SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═"*60)
print("  RSA RESULTS SUMMARY")
print("═"*60)
print(f"""
  Hypothesis collinearity
    Sentience vs taxonomic r: {r_hyp:.3f}
    Distinguishable: {'Yes' if abs(r_hyp) < 0.85 else 'WARNING: nearly collinear'}

  Sentience RSA
    Peak r:      {rsa_df['r_sentience'].max():.4f}
    Peak layer:  {peak_sent_layer}
    Significant: {(rsa_df['p_sent_corrected'] < 0.05).sum()} layers (Bonferroni)

  Taxonomic RSA
    Peak r:      {rsa_df['r_taxonomic'].max():.4f}
    Peak layer:  {peak_tax_layer}

  Sentience beyond taxonomy (partial r)
    Peak:        {peak_partial:+.4f} at layer {peak_partial_layer}
    Interpretation: {'Sentience predicts model similarity beyond taxonomy'
                     if peak_partial > 0.1
                     else 'No clear sentience signal beyond taxonomy'}

  Octopus (critical test case)
    Final layer sentience pull: {oct_df['sentience_pull'].iloc[-1]:+.4f}
    Verdict: {'Octopus encoded as sentience-similar to mammals (sentience encoding)'
              if oct_df['sentience_pull'].iloc[-1] > 0
              else 'Octopus encoded as phylogenetically similar to insects (taxonomy encoding)'}

  Peak layer for activation patching (Week 4)
    Sentience RSA peak: layer {peak_sent_layer}
    Partial r peak:     layer {peak_partial_layer}
    Recommendation: target layer {peak_partial_layer}
                    (sentience beyond taxonomy is strongest here)
""")

print("✓  RSA complete")

What the Three Figures Show ^^
Figure 1 (rsa_main.png) is your primary result. Panel A is the headline: if the sentience curve (dark blue) rises above the taxonomic curve (green) and the "sentience beyond taxonomy" delta line (orange dashes) is consistently positive, the model encodes something more than phylogeny. Panel C is the octopus subplot — watch whether its similarity to T2 mammals rises or falls relative to T5 insects across layers.

Figure 2 (rsa_hypothesis_matrices.png) makes the hypotheses legible. The residual panels (model minus sentience, model minus taxonomic) show where the model deviates from each hypothesis — block structure in the residuals indicates systematic under- or over-prediction.

Figure 3 (rsa_octopus_spotlight.png) is the most striking visual for a non-technical audience. The sentience pull index collapses the key question to a single signed number per layer — above zero means the model places octopus closer to dolphins than to ants, regardless of what phylogeny would predict.

For welfare-framed sentences, compute mean attention weight flowing to the entity token at each layer and head 

Compare across tiers: do attention heads weight the entity token differently for humans vs invertebrates? 

Identify the top 3–5 attention heads that show the strongest tier-differential attention then note their layer and head index for the patching experiments in week 4 

In [ ]:
import numpy as np
import pandas as pd
import h5py
import json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
from pathlib import Path
from itertools import combinations

warnings.filterwarnings('ignore')


# ══════════════════════════════════════════════════════════════════════════════
# 1. CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

HDF5_PATH   = "activations_1.4b.h5"
N_LAYERS    = 24
N_HEADS     = 16
RANDOM_SEED = 42

TIER_ORDER  = ["T1","T2","T3","T4","T5","T6"]
TIER_CONFIG = {
    "T1": {"label": "Humans",              "color": "#08306b"},
    "T2": {"label": "Mammals",             "color": "#2171b5"},
    "T3": {"label": "Vertebrates",         "color": "#6baed6"},
    "T4": {"label": "Invertebrates (w+)",  "color": "#fd8d3c"},
    "T5": {"label": "Invertebrates (min)", "color": "#d7301f"},
    "T6": {"label": "Non-sentient",        "color": "#bdbdbd"},
}

WELFARE_TEMPLATES = [
    "pain_suffering", "fear", "moral_consideration", "cross_framing"
]
NEUTRAL_TEMPLATES  = ["neutral_control"]
SCIENCE_TEMPLATES  = ["scientific_welfare"]

# Tier rank for correlation analysis
TIER_RANK = {"T1":1,"T2":2,"T3":3,"T4":4,"T5":5,"T6":6}

# Effect size threshold for "strong" tier-differential heads
EFFECT_SIZE_THRESHOLD = 0.3   # Cohen's d

print("Attention Analysis — Sentience Salience Probe")
print("="*60)


# ══════════════════════════════════════════════════════════════════════════════
# 2. DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_attention_data(hdf5_path: str) -> tuple[dict, pd.DataFrame]:
    """
    Load attn_to arrays (attention flowing TO entity token)
    for all sentences. Returns:
        attn_dict : sentence_id → np.ndarray (n_layers, n_heads, seq_len)
        meta_df   : sentence metadata
    """
    attn_dict = {}
    meta_rows = []

    with h5py.File(hdf5_path, "r") as f:
        for sid in sorted(f.keys()):
            grp = f[sid]
            if "attn_to" not in grp:
                continue

            attn_dict[sid] = grp["attn_to"][:].astype(np.float32)
            # shape: (n_layers, n_heads, seq_len)

            meta_rows.append({
                "sentence_id":   sid,
                "entity":        grp.attrs["entity"],
                "tier":          grp.attrs["tier"],
                "template_id":   grp.attrs["template_id"],
                "template_type": grp.attrs["template_type"],
                "n_tokens":      grp.attrs["n_tokens"],
                "positions":     list(grp.attrs["positions"]),
            })

    meta_df = pd.DataFrame(meta_rows)

    # Framing labels
    def framing(ttype):
        if ttype in WELFARE_TEMPLATES:  return "welfare"
        if ttype in NEUTRAL_TEMPLATES:  return "neutral"
        if ttype in SCIENCE_TEMPLATES:  return "science"
        return "other"

    meta_df["framing"]     = meta_df["template_type"].apply(framing)
    meta_df["tier_rank"]   = meta_df["tier"].map(TIER_RANK)
    meta_df["is_welfare"]  = meta_df["framing"] == "welfare"
    meta_df["is_neutral"]  = meta_df["framing"] == "neutral"

    print(f"  Loaded {len(attn_dict):,} attention arrays")
    print(f"  Welfare sentences : {meta_df['is_welfare'].sum()}")
    print(f"  Neutral sentences : {meta_df['is_neutral'].sum()}")
    print(f"  Tiers: {meta_df['tier'].value_counts().to_dict()}")

    return attn_dict, meta_df


print("\nLoading attention data...")
attn_dict, meta_df = load_attention_data(HDF5_PATH)


# ══════════════════════════════════════════════════════════════════════════════
# 3. COMPUTE MEAN ATTENTION TO ENTITY TOKEN
# ══════════════════════════════════════════════════════════════════════════════

def compute_entity_attention_weight(attn_arr: np.ndarray) -> np.ndarray:
    """
    Given attn_to of shape (n_layers, n_heads, seq_len),
    return the mean attention weight flowing TO the entity token
    summed across entity positions (already averaged in extraction).

    The attn_to array already stores the attention directed at
    entity positions (mean-pooled across subword tokens in extraction).
    We take the mean across all non-entity positions to get
    net attention received.

    Actually: attn_to[l, h, :] = for each query position, how much
    attention it sends to the entity. We want the COLUMN sum — total
    attention flowing into entity = sum over all query positions.

    Returns shape: (n_layers, n_heads)
    """
    # Sum over the sequence dimension = total attention flowing TO entity
    # Each head's attention rows sum to 1 (softmax), so this gives the
    # fraction of total attention budget directed at the entity token
    return attn_arr.sum(axis=-1)   # (n_layers, n_heads)


print("\nComputing entity attention weights...")
attention_weights = {}   # sid → (n_layers, n_heads)

for sid, arr in attn_dict.items():
    attention_weights[sid] = compute_entity_attention_weight(arr)

# Build a flat array for analysis
# Shape: (n_sentences, n_layers, n_heads)
sids_ordered  = [r["sentence_id"] for _, r in meta_df.iterrows()
                 if r["sentence_id"] in attention_weights]
attn_matrix   = np.stack([attention_weights[sid] for sid in sids_ordered])
meta_ordered  = meta_df[meta_df["sentence_id"].isin(sids_ordered)].copy()
meta_ordered  = meta_ordered.set_index("sentence_id").loc[sids_ordered].reset_index()

print(f"  Attention matrix shape: {attn_matrix.shape}")
# (n_sentences, n_layers, n_heads)


# ══════════════════════════════════════════════════════════════════════════════
# 4. MEAN ATTENTION BY TIER — WELFARE SENTENCES
# ══════════════════════════════════════════════════════════════════════════════

print("\nComputing tier-wise mean attention...")

welfare_mask = meta_ordered["is_welfare"].values   # (n_sentences,)
neutral_mask = meta_ordered["is_neutral"].values

# Mean attention per tier, per layer, per head — welfare only
tier_attn_welfare = {}   # tier → (n_layers, n_heads)
tier_attn_neutral = {}

for tier in TIER_ORDER:
    tier_mask_w = (meta_ordered["tier"] == tier).values & welfare_mask
    tier_mask_n = (meta_ordered["tier"] == tier).values & neutral_mask

    if tier_mask_w.sum() > 0:
        tier_attn_welfare[tier] = attn_matrix[tier_mask_w].mean(axis=0)
    if tier_mask_n.sum() > 0:
        tier_attn_neutral[tier] = attn_matrix[tier_mask_n].mean(axis=0)

    print(f"  {tier}: welfare n={tier_mask_w.sum():<5} "
          f"neutral n={tier_mask_n.sum()}")


# ══════════════════════════════════════════════════════════════════════════════
# 5. TIER-DIFFERENTIAL ATTENTION SCORE PER HEAD
# ══════════════════════════════════════════════════════════════════════════════

print("\nComputing tier-differential attention scores...")

def tier_differential_score(attn_matrix:  np.ndarray,
                              meta_df:      pd.DataFrame,
                              framing_mask: np.ndarray,
                              layer:        int,
                              head:         int) -> dict:
    """
    For a single (layer, head), compute statistics quantifying
    how differently this head attends to entity tokens across tiers.

    Metrics:
    1. T1 vs T6 Cohen's d          — extremes of gradient
    2. T1 vs T5 Cohen's d          — humans vs minimal-evidence invertebrates
    3. Spearman r (tier_rank, attn) — does attention correlate with tier order?
    4. F-statistic across 6 tiers  — overall tier effect

    Returns dict of scores.
    """
    mask = framing_mask
    attn_head = attn_matrix[mask, layer, head]   # (n_welfare,)
    tiers     = meta_df[mask]["tier"].values
    tier_ranks= meta_df[mask]["tier_rank"].values

    # Per-tier arrays
    by_tier = {t: attn_head[tiers == t] for t in TIER_ORDER}

    # Cohen's d: T1 vs T6
    t1_vals = by_tier.get("T1", np.array([]))
    t6_vals = by_tier.get("T6", np.array([]))
    t5_vals = by_tier.get("T5", np.array([]))

    def cohens_d(a, b):
        if len(a) < 2 or len(b) < 2:
            return 0.0
        pooled_std = np.sqrt((a.std()**2 + b.std()**2) / 2 + 1e-10)
        return (a.mean() - b.mean()) / pooled_std

    d_t1_t6 = cohens_d(t1_vals, t6_vals)
    d_t1_t5 = cohens_d(t1_vals, t5_vals)

    # Spearman r: tier rank vs attention
    if len(attn_head) > 2:
        spear_r, spear_p = stats.spearmanr(tier_ranks, attn_head)
    else:
        spear_r, spear_p = 0.0, 1.0

    # One-way ANOVA across 6 tiers
    tier_groups = [by_tier[t] for t in TIER_ORDER if len(by_tier[t]) > 1]
    if len(tier_groups) >= 2:
        f_stat, f_p = stats.f_oneway(*tier_groups)
    else:
        f_stat, f_p = 0.0, 1.0

    # Tier means (for directionality)
    tier_means = {t: by_tier[t].mean() if len(by_tier[t]) > 0 else np.nan
                  for t in TIER_ORDER}

    return {
        "layer":        layer,
        "head":         head,
        "head_label":   f"L{layer:02d}H{head:02d}",
        "d_t1_t6":      d_t1_t6,
        "d_t1_t5":      d_t1_t5,
        "spearman_r":   spear_r,
        "spearman_p":   spear_p,
        "f_stat":        f_stat,
        "f_p":           f_p,
        "abs_d_t1_t6":  abs(d_t1_t6),
        "abs_spearman": abs(spear_r),
        "tier_means":   tier_means,
        "t1_mean":      tier_means.get("T1", np.nan),
        "t6_mean":      tier_means.get("T6", np.nan),
        "t1_t6_diff":   (tier_means.get("T1", 0) -
                         tier_means.get("T6", 0)),
    }


# Run across all layers × heads
print("  Scoring all 384 (layer, head) combinations...")
score_rows = []
for layer in range(N_LAYERS):
    for head in range(N_HEADS):
        row = tier_differential_score(
            attn_matrix, meta_ordered,
            welfare_mask, layer, head
        )
        score_rows.append(row)

scores_df = pd.DataFrame(score_rows)

# Composite score: combine Cohen's d and |Spearman r|
scores_df["composite"] = (
    0.5 * scores_df["abs_d_t1_t6"] +
    0.5 * scores_df["abs_spearman"]
)

# Bonferroni correction
n_tests = N_LAYERS * N_HEADS
scores_df["spearman_p_corrected"] = np.minimum(
    scores_df["spearman_p"] * n_tests, 1.0
)
scores_df["f_p_corrected"] = np.minimum(
    scores_df["f_p"] * n_tests, 1.0
)

# Sort by composite score
scores_df = scores_df.sort_values("composite", ascending=False)

print(f"\n  Top 10 tier-differential attention heads:")
print(f"  {'Head':<8} {'Layer':>5} {'Head':>5} {'d(T1-T6)':>10} "
      f"{'|Spear r|':>10} {'F-stat':>8} {'Composite':>10} {'Sig':>5}")
print(f"  {'-'*8} {'-'*5} {'-'*5} {'-'*10} {'-'*10} {'-'*8} {'-'*10} {'-'*5}")

for _, row in scores_df.head(10).iterrows():
    sig = ("**" if row["spearman_p_corrected"] < 0.01 else
           "*"  if row["spearman_p_corrected"] < 0.05 else "")
    print(f"  {row['head_label']:<8} {int(row['layer']):>5} "
          f"{int(row['head']):>5} {row['d_t1_t6']:>+10.3f} "
          f"{row['abs_spearman']:>10.3f} {row['f_stat']:>8.2f} "
          f"{row['composite']:>10.3f} {sig:>5}")


# ══════════════════════════════════════════════════════════════════════════════
# 6. TOP HEADS — FORMAL SELECTION
# ══════════════════════════════════════════════════════════════════════════════

# Selection criteria:
# 1. Top 5 by composite score
# 2. Must have |d_t1_t6| > EFFECT_SIZE_THRESHOLD
# 3. Must be Bonferroni-significant OR have |Spearman r| > 0.2
# 4. Layer diversity preferred (avoid all heads from same layer)

def select_top_heads(scores_df:      pd.DataFrame,
                      n_select:       int = 5,
                      effect_thresh:  float = EFFECT_SIZE_THRESHOLD,
                      max_per_layer:  int = 2) -> pd.DataFrame:
    """
    Select top tier-differential heads with layer diversity constraint.
    """
    candidates = scores_df[
        (scores_df["abs_d_t1_t6"] > effect_thresh) |
        (scores_df["abs_spearman"] > 0.2)
    ].copy()

    selected   = []
    layer_counts = {}

    for _, row in candidates.iterrows():
        layer = int(row["layer"])
        if layer_counts.get(layer, 0) >= max_per_layer:
            continue
        selected.append(row)
        layer_counts[layer] = layer_counts.get(layer, 0) + 1
        if len(selected) >= n_select:
            break

    return pd.DataFrame(selected)


top_heads = select_top_heads(scores_df, n_select=5)

print("\n" + "═"*60)
print("  TOP TIER-DIFFERENTIAL ATTENTION HEADS")
print("  Selected for Week 4 activation patching")
print("═"*60)

patching_targets = []
for rank, (_, row) in enumerate(top_heads.iterrows(), 1):
    print(f"\n  #{rank}  {row['head_label']}")
    print(f"       Layer         : {int(row['layer'])}")
    print(f"       Head          : {int(row['head'])}")
    print(f"       Cohen's d     : {row['d_t1_t6']:+.4f}  (T1 > T6 if positive)")
    print(f"       |Spearman r|  : {row['abs_spearman']:.4f}")
    print(f"       F-stat        : {row['f_stat']:.3f}")
    print(f"       Composite     : {row['composite']:.4f}")
    print(f"       T1 mean attn  : {row['t1_mean']:.4f}")
    print(f"       T6 mean attn  : {row['t6_mean']:.4f}")
    print(f"       T1−T6 diff    : {row['t1_t6_diff']:+.4f}")
    sig = ("Bonferroni-significant" if row["spearman_p_corrected"] < 0.05
           else "not corrected-significant")
    print(f"       Significance  : {sig}")

    patching_targets.append({
        "rank":          rank,
        "head_label":    row["head_label"],
        "layer":         int(row["layer"]),
        "head":          int(row["head"]),
        "d_t1_t6":       row["d_t1_t6"],
        "abs_spearman":  row["abs_spearman"],
        "composite":     row["composite"],
        "t1_mean":       row["t1_mean"],
        "t6_mean":       row["t6_mean"],
    })

patching_df = pd.DataFrame(patching_targets)


# ══════════════════════════════════════════════════════════════════════════════
# 7. ATTENTION PROFILES BY TIER — PER HEAD
# ══════════════════════════════════════════════════════════════════════════════

def get_attention_profile(attn_matrix:  np.ndarray,
                           meta_df:      pd.DataFrame,
                           framing_mask: np.ndarray,
                           layer:        int,
                           head:         int) -> pd.DataFrame:
    """
    For one head, return mean ± se attention to entity per tier.
    """
    rows = []
    for tier in TIER_ORDER:
        mask = (meta_df["tier"] == tier).values & framing_mask
        vals = attn_matrix[mask, layer, head]
        rows.append({
            "tier":   tier,
            "mean":   vals.mean(),
            "se":     vals.std() / np.sqrt(len(vals) + 1e-10),
            "std":    vals.std(),
            "n":      mask.sum(),
        })
    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# 8. LAYER-AVERAGED ATTENTION HEATMAP BY TIER
# ══════════════════════════════════════════════════════════════════════════════

# Mean attention to entity for each (layer, tier) — welfare sentences
layer_tier_attn = np.zeros((N_LAYERS, len(TIER_ORDER)))
for tier_idx, tier in enumerate(TIER_ORDER):
    if tier in tier_attn_welfare:
        layer_tier_attn[:, tier_idx] = tier_attn_welfare[tier].mean(axis=1)

# Mean attention across all heads per layer × tier
layer_tier_attn_allhead = np.zeros((N_LAYERS, len(TIER_ORDER)))
for layer in range(N_LAYERS):
    for tier_idx, tier in enumerate(TIER_ORDER):
        tier_mask_w = ((meta_ordered["tier"] == tier).values & welfare_mask)
        if tier_mask_w.sum() > 0:
            layer_tier_attn_allhead[layer, tier_idx] = (
                attn_matrix[tier_mask_w, layer, :].mean()
            )


# ══════════════════════════════════════════════════════════════════════════════
# 9. COMPOSITE SCORE HEATMAP (ALL HEADS)
# ══════════════════════════════════════════════════════════════════════════════

# Reshape composite scores into (n_layers, n_heads) grid
composite_grid   = np.zeros((N_LAYERS, N_HEADS))
spearman_grid    = np.zeros((N_LAYERS, N_HEADS))
cohens_d_grid    = np.zeros((N_LAYERS, N_HEADS))
t1_t6_diff_grid  = np.zeros((N_LAYERS, N_HEADS))
sig_grid         = np.zeros((N_LAYERS, N_HEADS), dtype=bool)

for _, row in scores_df.iterrows():
    l, h = int(row["layer"]), int(row["head"])
    composite_grid[l, h]  = row["composite"]
    spearman_grid[l, h]   = row["spearman_r"]
    cohens_d_grid[l, h]   = row["d_t1_t6"]
    t1_t6_diff_grid[l, h] = row["t1_t6_diff"]
    sig_grid[l, h]        = row["spearman_p_corrected"] < 0.05


# ══════════════════════════════════════════════════════════════════════════════
# 10. FIGURES
# ══════════════════════════════════════════════════════════════════════════════

print("\nBuilding figures...")

# Custom colourmap
div_cmap  = LinearSegmentedColormap.from_list(
    "div", ["#d94801","#f7f7f7","#08306b"], N=256
)
seq_cmap  = LinearSegmentedColormap.from_list(
    "seq", ["#f7fbff","#08306b"], N=256
)


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1: Main attention analysis
# ══════════════════════════════════════════════════════════════════════════════

fig1 = plt.figure(figsize=(22, 24))
gs1  = gridspec.GridSpec(4, 3, figure=fig1,
                          hspace=0.45, wspace=0.35)
fig1.suptitle(
    "Attention Analysis — Entity Token Attention by Tier\n"
    f"Pythia-1.4b  |  Welfare-framed sentences  |  "
    f"{welfare_mask.sum()} sentences",
    fontsize=14, fontweight="bold", y=0.99
)


# ── Panel A: Layer × tier heatmap (mean across all heads) ─────────────────
ax_a = fig1.add_subplot(gs1[0, :2])

im_a = ax_a.imshow(
    layer_tier_attn_allhead.T,
    cmap="YlOrRd", aspect="auto",
    interpolation="nearest"
)
ax_a.set_xticks(range(N_LAYERS))
ax_a.set_xticklabels([str(l) for l in range(N_LAYERS)], fontsize=8)
ax_a.set_yticks(range(len(TIER_ORDER)))
ax_a.set_yticklabels(
    [f"{t}  {TIER_CONFIG[t]['label']}" for t in TIER_ORDER],
    fontsize=9
)
for i, tier in enumerate(TIER_ORDER):
    ax_a.get_yticklabels()[i].set_color(TIER_CONFIG[tier]["color"])
    ax_a.get_yticklabels()[i].set_fontweight("bold")

plt.colorbar(im_a, ax=ax_a, shrink=0.8,
             label="Mean attention weight to entity")
ax_a.set_xlabel("Layer", fontsize=10)
ax_a.set_title(
    "Panel A — Mean Attention to Entity Token by Tier and Layer\n"
    "(averaged across all heads; welfare sentences only)\n"
    "Brighter = more attention directed at entity token",
    fontsize=10, fontweight="bold"
)


# ── Panel B: T1 − T6 attention difference by layer ────────────────────────
ax_b = fig1.add_subplot(gs1[0, 2])

t1_by_layer = layer_tier_attn_allhead[:, 0]   # T1
t6_by_layer = layer_tier_attn_allhead[:, 5]   # T6
diff_by_layer = t1_by_layer - t6_by_layer

colors_b = ["#08306b" if d > 0 else "#d7301f" for d in diff_by_layer]
ax_b.barh(range(N_LAYERS), diff_by_layer,
          color=colors_b, edgecolor="white", linewidth=0.4)
ax_b.axvline(0, color="black", linewidth=1)
ax_b.set_yticks(range(N_LAYERS))
ax_b.set_yticklabels([f"L{l}" for l in range(N_LAYERS)], fontsize=8)
ax_b.set_xlabel("T1 − T6 attention difference", fontsize=9)
ax_b.set_title(
    "Panel B — T1 vs T6\nAttention Difference by Layer",
    fontsize=9, fontweight="bold"
)
ax_b.invert_yaxis()


# ── Panel C: Composite score heatmap (all 384 heads) ──────────────────────
ax_c = fig1.add_subplot(gs1[1, :2])

im_c = ax_c.imshow(
    composite_grid.T,
    cmap="hot_r", aspect="auto",
    interpolation="nearest",
    vmin=0, vmax=composite_grid.max()
)
# Mark top heads
for _, row in top_heads.iterrows():
    l, h = int(row["layer"]), int(row["head"])
    ax_c.add_patch(plt.Rectangle(
        (l - 0.5, h - 0.5), 1, 1,
        fill=False, edgecolor="cyan",
        linewidth=2.5, zorder=5
    ))
    ax_c.text(l, h, f"#{int(row.name)+1}" if "rank" not in row
              else f"#{int(row['rank'])}",
              ha="center", va="center",
              fontsize=6.5, color="cyan", fontweight="bold")

# Significance dots
for l in range(N_LAYERS):
    for h in range(N_HEADS):
        if sig_grid[l, h]:
            ax_c.plot(l, h, "w.", markersize=2.5, alpha=0.8)

ax_c.set_xticks(range(N_LAYERS))
ax_c.set_xticklabels([str(l) for l in range(N_LAYERS)], fontsize=7)
ax_c.set_yticks(range(N_HEADS))
ax_c.set_yticklabels([f"H{h}" for h in range(N_HEADS)], fontsize=7)
ax_c.set_xlabel("Layer", fontsize=10)
ax_c.set_ylabel("Head", fontsize=10)
plt.colorbar(im_c, ax=ax_c, shrink=0.8,
             label="Composite tier-differential score")
ax_c.set_title(
    "Panel C — Composite Tier-Differential Score: All 384 Attention Heads\n"
    "Cyan boxes = top 5 selected heads  |  "
    "White dots = Bonferroni-significant (p<0.05)",
    fontsize=10, fontweight="bold"
)


# ── Panel D: Spearman r heatmap ────────────────────────────────────────────
ax_d = fig1.add_subplot(gs1[1, 2])

im_d = ax_d.imshow(
    spearman_grid.T,
    cmap=div_cmap, aspect="auto",
    interpolation="nearest",
    vmin=-0.5, vmax=0.5
)
for _, row in top_heads.iterrows():
    l, h = int(row["layer"]), int(row["head"])
    ax_d.add_patch(plt.Rectangle(
        (l - 0.5, h - 0.5), 1, 1,
        fill=False, edgecolor="gold",
        linewidth=2, zorder=5
    ))

ax_d.set_xticks(range(0, N_LAYERS, 4))
ax_d.set_xticklabels([str(l) for l in range(0, N_LAYERS, 4)], fontsize=8)
ax_d.set_yticks(range(N_HEADS))
ax_d.set_yticklabels([f"H{h}" for h in range(N_HEADS)], fontsize=7)
ax_d.set_xlabel("Layer", fontsize=9)
ax_d.set_ylabel("Head", fontsize=9)
plt.colorbar(im_d, ax=ax_d, shrink=0.8, label="Spearman r")
ax_d.set_title(
    "Panel D — Spearman r\n(tier rank vs attention)",
    fontsize=9, fontweight="bold"
)


# ── Panel E: Top 5 heads — attention profiles ─────────────────────────────
n_top = len(top_heads)
for rank_idx, (_, head_row) in enumerate(top_heads.iterrows()):
    col = rank_idx % 3
    row_offset = 2 + rank_idx // 3

    ax = fig1.add_subplot(gs1[row_offset, col])

    layer = int(head_row["layer"])
    head  = int(head_row["head"])

    # Welfare profile
    prof_w = get_attention_profile(
        attn_matrix, meta_ordered, welfare_mask, layer, head
    )
    # Neutral profile
    prof_n = get_attention_profile(
        attn_matrix, meta_ordered, neutral_mask, layer, head
    )

    x = np.arange(len(TIER_ORDER))
    width = 0.35

    bars_w = ax.bar(
        x - width/2,
        prof_w["mean"],
        width,
        yerr=prof_w["se"],
        capsize=3,
        color=[TIER_CONFIG[t]["color"] for t in TIER_ORDER],
        alpha=0.9,
        edgecolor="white",
        linewidth=0.5,
        label="Welfare",
        error_kw={"linewidth": 1, "alpha": 0.7}
    )
    bars_n = ax.bar(
        x + width/2,
        prof_n["mean"],
        width,
        yerr=prof_n["se"],
        capsize=3,
        color=[TIER_CONFIG[t]["color"] for t in TIER_ORDER],
        alpha=0.35,
        edgecolor="white",
        linewidth=0.5,
        label="Neutral",
        error_kw={"linewidth": 1, "alpha": 0.5},
        hatch="///"
    )

    ax.set_xticks(x)
    ax.set_xticklabels(TIER_ORDER, fontsize=8)
    for tick, tier in zip(ax.get_xticklabels(), TIER_ORDER):
        tick.set_color(TIER_CONFIG[tier]["color"])

    rank_val = rank_idx + 1
    ax.set_title(
        f"#{rank_val}  {head_row['head_label']}\n"
        f"d(T1−T6)={head_row['d_t1_t6']:+.2f}  "
        f"|r|={head_row['abs_spearman']:.2f}",
        fontsize=8.5, fontweight="bold"
    )
    ax.set_ylabel("Attn weight to entity", fontsize=8)
    ax.set_xlabel("Tier", fontsize=8)

    if rank_idx == 0:
        ax.legend(fontsize=7, loc="upper right")

    # Add gradient arrow annotation
    t1_val = prof_w.loc[prof_w["tier"]=="T1","mean"].values[0]
    t6_val = prof_w.loc[prof_w["tier"]=="T6","mean"].values[0]
    ax.annotate(
        "",
        xy     = (5 - width/2, t6_val + 0.001),
        xytext = (0 - width/2, t1_val + 0.001),
        arrowprops=dict(
            arrowstyle="->",
            color="black",
            lw=1.2,
            connectionstyle="arc3,rad=0.2"
        )
    )

# Hide unused subplots
for extra_col in range(n_top % 3, 3):
    if n_top % 3 != 0:
        try:
            fig1.add_subplot(gs1[2 + n_top//3, extra_col]).set_visible(False)
        except Exception:
            pass

plt.savefig("attention_analysis_main.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: attention_analysis_main.png")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2: Head-level deep dive for top 5
# ══════════════════════════════════════════════════════════════════════════════

fig2, axes2 = plt.subplots(2, 3, figsize=(18, 12))
fig2.suptitle(
    "Top Tier-Differential Heads — Deep Dive\n"
    "Attention distribution per tier, welfare vs neutral framing",
    fontsize=13, fontweight="bold"
)

for rank_idx, (_, head_row) in enumerate(top_heads.iterrows()):
    ax = axes2.flatten()[rank_idx]
    layer = int(head_row["layer"])
    head  = int(head_row["head"])

    # Violin/box plot of attention distribution per tier
    tier_data_w = []
    tier_data_n = []
    for tier in TIER_ORDER:
        mask_w = (meta_ordered["tier"] == tier).values & welfare_mask
        mask_n = (meta_ordered["tier"] == tier).values & neutral_mask
        tier_data_w.append(attn_matrix[mask_w, layer, head])
        tier_data_n.append(attn_matrix[mask_n, layer, head])

    positions_w = np.arange(len(TIER_ORDER)) * 2
    positions_n = np.arange(len(TIER_ORDER)) * 2 + 0.7

    vp_w = ax.violinplot(
        tier_data_w, positions=positions_w,
        showmedians=True, showextrema=False,
        widths=0.6
    )
    vp_n = ax.violinplot(
        tier_data_n, positions=positions_n,
        showmedians=True, showextrema=False,
        widths=0.6
    )

    for i, (body_w, body_n, tier) in enumerate(zip(
            vp_w["bodies"], vp_n["bodies"], TIER_ORDER)):
        color = TIER_CONFIG[tier]["color"]
        body_w.set_facecolor(color)
        body_w.set_alpha(0.85)
        body_n.set_facecolor(color)
        body_n.set_alpha(0.30)

    for part in ["cmedians"]:
        vp_w[part].set_color("white")
        vp_n[part].set_color("black")
        vp_w[part].set_linewidth(1.5)

    tick_pos = np.arange(len(TIER_ORDER)) * 2 + 0.35
    ax.set_xticks(tick_pos)
    ax.set_xticklabels(TIER_ORDER, fontsize=9)
    for tick, tier in zip(ax.get_xticklabels(), TIER_ORDER):
        tick.set_color(TIER_CONFIG[tier]["color"])
        tick.set_fontweight("bold")

    ax.set_title(
        f"#{rank_idx+1}  {head_row['head_label']}\n"
        f"d={head_row['d_t1_t6']:+.3f}  "
        f"|r|={head_row['abs_spearman']:.3f}  "
        f"F={head_row['f_stat']:.1f}",
        fontsize=9, fontweight="bold"
    )
    ax.set_ylabel("Attention weight to entity", fontsize=8)

    # Legend
    if rank_idx == 0:
        welfare_patch = mpatches.Patch(color="grey", alpha=0.85, label="Welfare")
        neutral_patch = mpatches.Patch(color="grey", alpha=0.30,
                                        label="Neutral", hatch="///")
        ax.legend(handles=[welfare_patch, neutral_patch], fontsize=8)

axes2.flatten()[-1].set_visible(False)

plt.tight_layout()
plt.savefig("attention_top_heads_detail.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: attention_top_heads_detail.png")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3: Layer-level summary + welfare vs neutral comparison
# ══════════════════════════════════════════════════════════════════════════════

fig3, axes3 = plt.subplots(2, 2, figsize=(16, 12))
fig3.suptitle(
    "Layer-Level Attention Summary\n"
    "Welfare vs neutral framing  |  Tier gradient across layers",
    fontsize=13, fontweight="bold"
)

# ── Panel A: Mean attention to entity by tier across layers (line plot) ───
ax_3a = axes3[0, 0]
for tier in TIER_ORDER:
    if tier not in tier_attn_welfare:
        continue
    # Mean across all heads
    vals = tier_attn_welfare[tier].mean(axis=1)   # (n_layers,)
    ax_3a.plot(
        range(N_LAYERS), vals,
        color     = TIER_CONFIG[tier]["color"],
        linewidth = 2,
        marker    = "o",
        markersize= 3.5,
        label     = TIER_CONFIG[tier]["label"],
        alpha     = 0.9
    )

ax_3a.set_xlabel("Layer", fontsize=10)
ax_3a.set_ylabel("Mean attention to entity", fontsize=10)
ax_3a.set_title(
    "Panel A — Entity Attention by Tier Across Layers\n"
    "(welfare sentences, mean across all heads)",
    fontsize=9, fontweight="bold"
)
ax_3a.legend(fontsize=8, loc="upper left")
ax_3a.set_xticks(range(0, N_LAYERS, 2))

# ── Panel B: Welfare vs neutral attention by tier (final layer) ───────────
ax_3b = axes3[0, 1]
final_layer = N_LAYERS - 1

welfare_means_final = [
    tier_attn_welfare.get(t, np.zeros((N_LAYERS, N_HEADS)))[final_layer].mean()
    for t in TIER_ORDER
]
neutral_means_final = [
    tier_attn_neutral.get(t, np.zeros((N_LAYERS, N_HEADS)))[final_layer].mean()
    for t in TIER_ORDER
]

x3b   = np.arange(len(TIER_ORDER))
width = 0.35

ax_3b.bar(x3b - width/2, welfare_means_final, width,
           color=[TIER_CONFIG[t]["color"] for t in TIER_ORDER],
           alpha=0.9, label="Welfare", edgecolor="white")
ax_3b.bar(x3b + width/2, neutral_means_final, width,
           color=[TIER_CONFIG[t]["color"] for t in TIER_ORDER],
           alpha=0.35, label="Neutral", edgecolor="white", hatch="///")

ax_3b.set_xticks(x3b)
ax_3b.set_xticklabels(TIER_ORDER, fontsize=9)
for tick, tier in zip(ax_3b.get_xticklabels(), TIER_ORDER):
    tick.set_color(TIER_CONFIG[tier]["color"])
ax_3b.set_ylabel("Mean attention (final layer)", fontsize=9)
ax_3b.set_title(
    "Panel B — Welfare vs Neutral Attention\nFinal Layer, All Heads",
    fontsize=9, fontweight="bold"
)
ax_3b.legend(fontsize=9)

# ── Panel C: Spearman r (tier rank vs attention) by layer and head ────────
ax_3c = axes3[1, 0]

# Layer-aggregated: mean |Spearman r| across all heads per layer
mean_r_by_layer = np.abs(spearman_grid).mean(axis=1)   # (n_layers,)
max_r_by_layer  = np.abs(spearman_grid).max(axis=1)

ax_3c.fill_between(range(N_LAYERS), mean_r_by_layer,
                    alpha=0.2, color="#08306b")
ax_3c.plot(range(N_LAYERS), mean_r_by_layer,
           color="#08306b", linewidth=2, marker="o",
           markersize=4, label="Mean |r| across heads")
ax_3c.plot(range(N_LAYERS), max_r_by_layer,
           color="#d7301f", linewidth=2, linestyle="--",
           marker="s", markersize=4, label="Max |r| across heads")

# Mark top heads
for _, row in top_heads.iterrows():
    l = int(row["layer"])
    ax_3c.axvline(l, color="#fd8d3c", linewidth=1.5,
                   linestyle=":", alpha=0.8)

ax_3c.set_xlabel("Layer", fontsize=10)
ax_3c.set_ylabel("|Spearman r| (tier rank vs attention)", fontsize=9)
ax_3c.set_title(
    "Panel C — Tier–Attention Correlation by Layer\n"
    "Orange lines = layers containing top-5 heads",
    fontsize=9, fontweight="bold"
)
ax_3c.legend(fontsize=9)
ax_3c.set_xticks(range(N_LAYERS))

# ── Panel D: Top 5 heads summary table ────────────────────────────────────
ax_3d = axes3[1, 1]
ax_3d.axis("off")

table_data = []
col_labels = ["Rank","Head","Layer","Head#",
              "d(T1−T6)","|r|","F","Composite"]
for _, row in patching_df.iterrows():
    table_data.append([
        f"#{int(row['rank'])}",
        row["head_label"],
        str(int(row["layer"])),
        str(int(row["head"])),
        f"{row['d_t1_t6']:+.3f}",
        f"{row['abs_spearman']:.3f}",
        f"{scores_df[(scores_df['layer']==row['layer']) & (scores_df['head']==row['head'])]['f_stat'].values[0]:.1f}",
        f"{row['composite']:.3f}",
    ])

table = ax_3d.table(
    cellText   = table_data,
    colLabels  = col_labels,
    cellLoc    = "center",
    loc        = "center",
    bbox       = [0, 0.2, 1, 0.75]
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.auto_set_column_width(list(range(len(col_labels))))

# Colour header
for j in range(len(col_labels)):
    table[0, j].set_facecolor("#08306b")
    table[0, j].set_text_props(color="white", fontweight="bold")

ax_3d.set_title(
    "Panel D — Top 5 Heads for Activation Patching (Week 4)\n"
    "Selected by composite tier-differential score with layer diversity",
    fontsize=9, fontweight="bold", y=0.97
)

plt.tight_layout()
plt.savefig("attention_layer_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: attention_layer_summary.png")


# ══════════════════════════════════════════════════════════════════════════════
# 11. SAVE RESULTS
# ══════════════════════════════════════════════════════════════════════════════

scores_df.drop(columns=["tier_means"]).to_csv(
    "attention_scores_all_heads.csv", index=False
)
patching_df.to_csv("attention_patching_targets.csv", index=False)

np.save("composite_grid.npy",  composite_grid)
np.save("spearman_grid.npy",   spearman_grid)
np.save("cohens_d_grid.npy",   cohens_d_grid)

print("\n── Saved files ──")
for fname in [
    "attention_scores_all_heads.csv",
    "attention_patching_targets.csv",
    "composite_grid.npy",
    "spearman_grid.npy",
    "cohens_d_grid.npy",
    "attention_analysis_main.png",
    "attention_top_heads_detail.png",
    "attention_layer_summary.png",
]:
    size = Path(fname).stat().st_size/1e3 if Path(fname).exists() else 0
    print(f"  {fname:<45} {size:.0f} KB")


# ══════════════════════════════════════════════════════════════════════════════
# 12. RESULTS SUMMARY + PATCHING TARGETS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═"*60)
print("  ATTENTION ANALYSIS RESULTS SUMMARY")
print("═"*60)

print(f"""
  Overall pattern
    Heads with |d_T1_T6| > {EFFECT_SIZE_THRESHOLD}: {(scores_df['abs_d_t1_t6'] > EFFECT_SIZE_THRESHOLD).sum()}
    Heads with |r| > 0.20:           {(scores_df['abs_spearman'] > 0.20).sum()}
    Bonferroni-significant heads:     {sig_grid.sum()}

  Top 5 activation patching targets for Week 4
""")

for _, row in patching_df.iterrows():
    direction = ("T1 > T6  (humans get MORE attention)"
                 if row["d_t1_t6"] > 0
                 else "T6 > T1  (controls get MORE attention)")
    print(f"    #{int(row['rank'])}  {row['head_label']:<10} "
          f"layer={int(row['layer'])}  head={int(row['head'])}  "
          f"d={row['d_t1_t6']:+.3f}  {direction}")

print(f"""
  Welfare vs neutral framing
    Framing modulates attention: check Panel B for T1−T6 gap by framing
    If welfare gap >> neutral gap: attention tier-differential is
    context-dependent (triggered by welfare language, not entity alone)
    If gaps are similar: tier-differential is in entity representation

  Recommendation for patching
    Primary target:   {patching_df.iloc[0]['head_label']}  (highest composite score)
    Secondary target: {patching_df.iloc[1]['head_label']}  (second highest)
    Patching strategy: replace attention pattern at these heads
    for 'bee' sentences with patterns from 'child' sentences,
    then measure shift in welfare-relevant output probabilities.
""")

print("  Attention analysis complete")